# Exp2D — QLoRA Fine-Tuning + Final 101-Case Evaluation

This notebook contains **only the final Exp2D generator experiment**:

1. Load the already-generated `train.jsonl` and `val.jsonl`
2. Load the final **Exp2D reranked** RAG output
3. Load the full 660-patient MMDental clinical table for:
   - retrieved-case clinical-record lookup at test time
   - test ground truth
4. Validate all inputs
5. Fine-tune MedGemma with **4-bit QLoRA**
6. Generate predictions for the same **101 held-out test patients**
7. Save **ground truth vs LLM predictions**
8. Run the final **9-metric evaluation**

**No frozen MedGemma experiment is included. No BioMedCLIP, Q1–Q4 generation, FAISS retrieval, CSLS, MMR, or XGBoost training is rerun here.**

### Required input files
- `train.jsonl` — corrected 248-example SFT training set
- `val.jsonl` — corrected 28-example validation set
- `rag_output_exp2d_final.json` — final post-XGBoost retrieval for 101 test patients
- `mmdental_cleaned_full.csv` — full 660-patient clinical lookup + ground truth

The first three files alone are **not sufficient** for final inference/evaluation because the Exp2D RAG file stores retrieved `case_id` + `reranker_score`, not the retrieved patients' complete clinical records or the query patients' target ground truth.


In [1]:
# Install once in the Modal image/session.
# If these are already baked into your Modal image, this cell can be skipped.
%pip install -q -U \
    "transformers>=4.53" \
    "bitsandbytes>=0.46.1" \
    peft trl accelerate datasets \
    pandas==2.2.3 scipy==1.14.0 sacrebleu rouge_score nltk pycocoevalcap openpyxl

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.11.1 requires scipy>=1.15, but you have scipy 1.14.0 which is incompatible.
access 1.1.10.post3 requires scipy>=1.14.1, but you have scipy 1.14.0 which is incompatible.


In [2]:
import os
import re
import ast
import json
import random
from pathlib import Path


import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from trl.trainer.sft_trainer import DataCollatorForLanguageModeling

# ============================================================
# EDIT ONLY THESE PATHS FOR MODAL
# ============================================================

TRAIN_JSONL = "/content/train (1).jsonl"
VAL_JSONL = "/content/val (1).jsonl"
RAG_JSON = "/content/rag_output_exp2d_final.json"
FULL_PATIENT_CSV = "/content/mmdental_cleaned_full.csv"

OUTPUT_DIR = "/content/exp2d_qlora_final"

# Keep the same base model used in the previous experiment.
LLM_BASE_MODEL = "unsloth/medgemma-1.5-4b-it"

os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_FIELDS = [
    "Oral Check",
    "Diagnosis",
    "Treatment plan",
    "Handle",
    "Doctor advices",
]

REFERENCE_FIELDS = [
    "Main appeal",
    "Present medical history",
    "Oral Check",
    "Diagnosis",
    "Treatment plan",
    "Handle",
    "Doctor advices",
]

EXPECTED_TRAIN_N = 248
EXPECTED_VAL_N = 28
EXPECTED_TEST_N = 101

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [3]:
# ============================================================
# LOAD INPUTS
# ============================================================

for p in [TRAIN_JSONL, VAL_JSONL, RAG_JSON, FULL_PATIENT_CSV]:
    assert os.path.exists(p), f"Missing required input: {p}"

with open(RAG_JSON, "r", encoding="utf-8") as f:
    rag_output_exp2d = json.load(f)

full_patient_df = pd.read_excel(FULL_PATIENT_CSV)

def norm_id(x):
    if x is None:
        return None
    try:
        return str(int(float(x)))
    except Exception:
        return str(x).strip()

full_patient_df["Filename"] = full_patient_df["Filename"].map(norm_id)

assert len(rag_output_exp2d) == EXPECTED_TEST_N, (
    f"Expected {EXPECTED_TEST_N} Exp2D test queries, got {len(rag_output_exp2d)}"
)
assert full_patient_df["Filename"].nunique() == 660, (
    f"Expected 660 unique MMDental patients, got {full_patient_df['Filename'].nunique()}"
)

print("Exp2D test queries:", len(rag_output_exp2d))
print("Full clinical lookup:", full_patient_df.shape)


Exp2D test queries: 101
Full clinical lookup: (660, 19)


In [4]:
# ============================================================
# BUILD FULL 660-PATIENT CLINICAL LOOKUP
# Used ONLY to resolve retrieved IDs and obtain test GT.
# It does NOT turn all 660 patients into SFT training examples.
# ============================================================

def clean_clinical_value(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    value = str(value).strip()
    if value.lower() in {"nan", "none", "null", "n/a", "na"}:
        return ""
    return value

assert not full_patient_df["Filename"].duplicated().any(), "Duplicate patient IDs in full table."

clinical_lookup = {}
for _, row in full_patient_df.iterrows():
    cid = norm_id(row["Filename"])
    clinical_lookup[cid] = {
        field: clean_clinical_value(row.get(field))
        for field in REFERENCE_FIELDS
    }

assert len(clinical_lookup) == 660

# Every query and every retrieved reference must resolve.
missing_query_gt = []
missing_refs = []

for qid, entry in rag_output_exp2d.items():
    qid = norm_id(qid)
    if qid not in clinical_lookup:
        missing_query_gt.append(qid)

    exemplars = entry.get("retrieved_exemplars", [])
    assert len(exemplars) == 5, f"Query {qid}: expected exactly 5 retrieved cases."

    retrieved_ids = [norm_id(x["case_id"]) for x in exemplars]
    assert qid not in retrieved_ids, f"SELF-RETRIEVAL: query {qid}"
    assert len(retrieved_ids) == len(set(retrieved_ids)), f"Duplicate refs for query {qid}"

    for rid in retrieved_ids:
        if rid not in clinical_lookup:
            missing_refs.append((qid, rid))

assert not missing_query_gt, f"Missing query GT IDs: {missing_query_gt[:10]}"
assert not missing_refs, f"Missing retrieved clinical records: {missing_refs[:10]}"

print("[PASS] All 101 queries and all 505 retrieved references resolve in the 660-patient lookup.")


[PASS] All 101 queries and all 505 retrieved references resolve in the 660-patient lookup.


In [5]:
# ============================================================
# PRE-FLIGHT VALIDATION OF EXISTING TRAIN/VAL JSONL
# ============================================================

EXPECTED_TARGET_FIELDS = set(TARGET_FIELDS)
MISSING_VALUES = {
    "", "none", "nan", "null", "not available",
    "not determined", "n/a", "na",
}

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception as e:
                raise AssertionError(f"Invalid JSON at {path}:{line_no}: {e}")
    return rows

def get_one_message(example, role):
    matches = [m for m in example["messages"] if m.get("role") == role]
    assert len(matches) == 1, (
        f"Case {example.get('case_id')}: expected one {role} message, got {len(matches)}"
    )
    return matches[0]["content"]

def parse_target(example):
    text = get_one_message(example, "assistant").strip()
    target = json.loads(text)
    assert isinstance(target, dict)
    assert set(target.keys()) == EXPECTED_TARGET_FIELDS, (
        f"Case {example.get('case_id')}: target fields = {set(target.keys())}"
    )
    return target

train_rows = load_jsonl(TRAIN_JSONL)
val_rows = load_jsonl(VAL_JSONL)

assert len(train_rows) == EXPECTED_TRAIN_N, len(train_rows)
assert len(val_rows) == EXPECTED_VAL_N, len(val_rows)

train_ids = [norm_id(x["case_id"]) for x in train_rows]
val_ids = [norm_id(x["case_id"]) for x in val_rows]
test_ids = {norm_id(x) for x in rag_output_exp2d.keys()}

assert len(train_ids) == len(set(train_ids))
assert len(val_ids) == len(set(val_ids))
assert not (set(train_ids) & set(val_ids)), "Train/val query-ID overlap."
assert not (set(train_ids) & test_ids), "Train/test query-ID overlap."
assert not (set(val_ids) & test_ids), "Val/test query-ID overlap."

for split_name, examples in [("TRAIN", train_rows), ("VAL", val_rows)]:
    for ex in examples:
        roles = [m.get("role") for m in ex["messages"]]
        assert roles == ["system", "user", "assistant"], (
            f"{split_name} {ex.get('case_id')}: roles={roles}"
        )

        user_prompt = get_one_message(ex, "user")
        assert "CURRENT PATIENT:" in user_prompt
        assert "REFERENCE CASES" in user_prompt
        assert "not facts about the current patient" in user_prompt.lower()

        # Current-patient input must remain Age + Sex + Main Appeal only.
        current_section = user_prompt.split("REFERENCE CASES", 1)[0]
        assert "Present medical history:" not in current_section
        assert "Past medical history:" not in current_section
        assert "Oral Check:" not in current_section
        assert "Diagnosis:" not in current_section
        assert "Treatment plan:" not in current_section

        # Exactly five historical references.
        assert user_prompt.count("[Reference Case ") == 5, (
            f"{split_name} {ex.get('case_id')}: wrong reference count"
        )

        parse_target(ex)

print("[PASS] QLoRA pre-flight")
print("Train:", len(train_rows), "| Val:", len(val_rows), "| Test:", len(test_ids))


[PASS] QLoRA pre-flight
Train: 248 | Val: 28 | Test: 101


## QLoRA model

The frozen-model comparison is intentionally omitted. This section loads MedGemma directly in 4-bit NF4 and adds LoRA adapters.


In [6]:
# ============================================================
# LOAD MEDGEMMA FOR 4-BIT QLoRA
# ============================================================

PAPER_BEST_LORA_CONFIG = dict(
    r=16,
    lora_alpha=64,
    lora_dropout=0.08,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

qlora_tokenizer = AutoTokenizer.from_pretrained(LLM_BASE_MODEL)

if qlora_tokenizer.pad_token is None:
    qlora_tokenizer.pad_token = qlora_tokenizer.eos_token

qlora_model = AutoModelForCausalLM.from_pretrained(
    LLM_BASE_MODEL,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
)

qlora_model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
qlora_model.config.use_cache = False

lora_config = LoraConfig(
    r=PAPER_BEST_LORA_CONFIG["r"],
    lora_alpha=PAPER_BEST_LORA_CONFIG["lora_alpha"],
    lora_dropout=PAPER_BEST_LORA_CONFIG["lora_dropout"],
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=PAPER_BEST_LORA_CONFIG["target_modules"],
)

# SFTTrainer/PEFT can apply this config directly.
qlora_model.print_trainable_parameters() if hasattr(qlora_model, "print_trainable_parameters") else None


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

In [7]:
# ============================================================
# LOAD + CONVERT SFT DATA
# ============================================================

qlora_train_ds = load_dataset("json", data_files=TRAIN_JSONL)["train"]
qlora_val_ds = load_dataset("json", data_files=VAL_JSONL)["train"]

def convert_to_prompt_completion(example):
    messages = example["messages"]
    assert [m["role"] for m in messages] == ["system", "user", "assistant"]

    prompt_text = qlora_tokenizer.apply_chat_template(
        messages[:2],
        tokenize=False,
        add_generation_prompt=True,
    )

    # Explicit assistant completion + EOS is safer than slicing a separately
    # templated full conversation.
    completion_text = messages[2]["content"].strip() + qlora_tokenizer.eos_token

    return {
        "prompt": prompt_text,
        "completion": completion_text,
    }

qlora_train_ds = qlora_train_ds.map(
    convert_to_prompt_completion,
    remove_columns=qlora_train_ds.column_names,
)
qlora_val_ds = qlora_val_ds.map(
    convert_to_prompt_completion,
    remove_columns=qlora_val_ds.column_names,
)

lengths = [
    len(
        qlora_tokenizer(
            ex["prompt"] + ex["completion"],
            add_special_tokens=False,
        )["input_ids"]
    )
    for ex in qlora_train_ds
]

print(
    f"Token lengths: p50={np.percentile(lengths,50):.0f}, "
    f"p90={np.percentile(lengths,90):.0f}, "
    f"p95={np.percentile(lengths,95):.0f}, "
    f"max={max(lengths)}"
)

# Preserve almost all examples while avoiding a single extreme outlier
# controlling memory. Increase manually if Modal GPU memory permits.
max_length = int(np.percentile(lengths, 95)) + 128
max_length = min(max_length, 4096)
print("Training max_length:", max_length)


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/248 [00:00<?, ? examples/s]

Map:   0%|          | 0/28 [00:00<?, ? examples/s]

Token lengths: p50=3086, p90=5299, p95=6520, max=11571
Training max_length: 4096


In [8]:
# ============================================================
# GEMMA-3 TEXT COLLATOR
# token_type_ids are required during training.
# ============================================================

class Gemma3TextDataCollator(DataCollatorForLanguageModeling):
    def __call__(self, examples):
        batch = super().__call__(examples)
        batch["token_type_ids"] = torch.zeros_like(
            batch["input_ids"],
            dtype=torch.long,
        )
        return batch

gemma3_collator = Gemma3TextDataCollator(
    pad_token_id=qlora_tokenizer.pad_token_id
)

print("Gemma3 text collator ready.")


Gemma3 text collator ready.


In [9]:
# ============================================================
# QLoRA TRAINING + LIVE ETA
# ============================================================

import os
import time
import math
import numpy as np
import torch

from transformers import TrainerCallback, EarlyStoppingCallback
from trl import SFTTrainer, SFTConfig


# ============================================================
# LIVE TRAINING TIMER
# ============================================================

class TrainingETACallback(TrainerCallback):

    def __init__(self):
        self.start_time = None
        self.last_print = 0

    @staticmethod
    def format_time(seconds):
        if seconds is None or not np.isfinite(seconds):
            return "calculating..."

        seconds = max(0, int(seconds))

        hours, rem = divmod(seconds, 3600)
        minutes, seconds = divmod(rem, 60)

        if hours:
            return f"{hours}h {minutes}m {seconds}s"
        elif minutes:
            return f"{minutes}m {seconds}s"
        else:
            return f"{seconds}s"

    def on_train_begin(self, args, state, control, **kwargs):

        self.start_time = time.time()

        print("\n" + "=" * 70)
        print("QLoRA TRAINING STARTED")
        print("=" * 70)
        print(f"Maximum optimizer steps : {state.max_steps}")
        print(f"Maximum epochs          : {args.num_train_epochs}")
        print(f"Train batch size        : {args.per_device_train_batch_size}")
        print(f"Gradient accumulation   : {args.gradient_accumulation_steps}")
        print("=" * 70)

    def on_step_end(self, args, state, control, **kwargs):

        if self.start_time is None:
            return

        elapsed = time.time() - self.start_time

        completed = state.global_step
        total = state.max_steps

        if completed == 0:
            return

        seconds_per_step = elapsed / completed
        remaining_steps = max(total - completed, 0)
        eta = seconds_per_step * remaining_steps

        percent = 100 * completed / total

        # Print every optimizer step.
        print(
            f"\r"
            f"Step {completed:>3}/{total} "
            f"| {percent:6.2f}% "
            f"| Epoch {state.epoch:.2f} "
            f"| Elapsed: {self.format_time(elapsed)} "
            f"| ETA: {self.format_time(eta)} "
            f"| {seconds_per_step:.1f}s/step",
            end="",
            flush=True
        )

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):

        elapsed = time.time() - self.start_time

        print("\n")
        print("-" * 70)
        print(f"VALIDATION COMPLETE — epoch {state.epoch:.2f}")

        if metrics and "eval_loss" in metrics:
            print(f"Validation loss: {metrics['eval_loss']:.6f}")

        print(f"Elapsed time   : {self.format_time(elapsed)}")
        print("-" * 70)

    def on_train_end(self, args, state, control, **kwargs):

        elapsed = time.time() - self.start_time

        print("\n")
        print("=" * 70)
        print("QLoRA TRAINING FINISHED")
        print(f"Optimizer steps : {state.global_step}")
        print(f"Final epoch     : {state.epoch:.2f}")
        print(f"Total time      : {self.format_time(elapsed)}")
        print("=" * 70)


# ============================================================
# TRAINING CONFIGURATION
# ============================================================

QLORA_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "medgemma_exp2d_qlora"
)

os.makedirs(
    QLORA_OUTPUT_DIR,
    exist_ok=True
)

effective_batch_size = 8

steps_per_epoch = math.ceil(
    len(qlora_train_ds) / effective_batch_size
)

print("Training examples :", len(qlora_train_ds))
print("Validation examples:", len(qlora_val_ds))
print("Steps per epoch    :", steps_per_epoch)
print("Maximum steps      :", steps_per_epoch * 10)


sft_config = SFTConfig(

    output_dir=QLORA_OUTPUT_DIR,
    max_length=max_length,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    learning_rate=1e-4,

    # Maximum only — early stopping may finish earlier.
    num_train_epochs=10,

    lr_scheduler_type="cosine",

    warmup_steps=max(
        1,
        int(0.1 * steps_per_epoch * 10)
    ),

    # Loss appears frequently.
    logging_steps=5,

    eval_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=1,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    bf16=False,
    fp16=False,

    optim="paged_adamw_8bit",

    report_to="none",

    packing=False,

    seed=42,

    completion_only_loss=True,

    loss_type="nll",
)


early_stop = EarlyStoppingCallback(
    early_stopping_patience=3,
    early_stopping_threshold=0.0,
)

eta_callback = TrainingETACallback()


# ============================================================
# CREATE TRAINER
# ============================================================

qlora_trainer = SFTTrainer(

    model=qlora_model,

    args=sft_config,

    train_dataset=qlora_train_ds,
    eval_dataset=qlora_val_ds,

    data_collator=gemma3_collator,

    processing_class=qlora_tokenizer,

    peft_config=lora_config,

    callbacks=[
        early_stop,
        eta_callback,
    ],
)


# ============================================================
# MANDATORY SANITY CHECK
# ============================================================

preview = next(
    iter(
        qlora_trainer.get_train_dataloader()
    )
)

assert "token_type_ids" in preview

assert torch.all(
    preview["token_type_ids"] == 0
)

labels = preview["labels"]

ignored = int(
    (labels == -100).sum().item()
)

supervised = int(
    (labels != -100).sum().item()
)

print("\n" + "=" * 70)
print("TRAINING BATCH CHECK")
print("=" * 70)

print(
    "Batch shape:",
    tuple(preview["input_ids"].shape)
)

print(
    "Ignored prompt/pad labels:",
    ignored
)

print(
    "Supervised completion labels:",
    supervised
)

assert supervised > 0, (
    "No supervised completion tokens."
)

assert ignored > 0, (
    "Completion-only masking is not active."
)

print("\n[PASS] Training batch sanity check")


# ============================================================
# START TIMER + TRAIN
# ============================================================

print("\nStarting QLoRA training...\n")

wall_start = time.time()

train_result = qlora_trainer.train()

wall_total = time.time() - wall_start


# ============================================================
# SAVE BEST ADAPTER
# ============================================================

qlora_trainer.save_model(
    QLORA_OUTPUT_DIR
)

qlora_tokenizer.save_pretrained(
    QLORA_OUTPUT_DIR
)


# ============================================================
# FINAL SUMMARY
# ============================================================

def format_duration(seconds):

    seconds = int(seconds)

    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)

    return f"{h}h {m}m {s}s"


print("\n" + "=" * 70)
print("FINAL TRAINING SUMMARY")
print("=" * 70)

print(
    "Total wall-clock time:",
    format_duration(wall_total)
)

print(
    "Completed optimizer steps:",
    qlora_trainer.state.global_step
)

print(
    "Completed epochs:",
    qlora_trainer.state.epoch
)

print(
    "Best checkpoint:",
    qlora_trainer.state.best_model_checkpoint
)

print(
    "Best validation loss:",
    qlora_trainer.state.best_metric
)

print(
    "QLoRA adapter:",
    QLORA_OUTPUT_DIR
)

print("=" * 70)

Training examples : 248
Validation examples: 28
Steps per epoch    : 31
Maximum steps      : 310


Adding EOS to train dataset:   0%|          | 0/248 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/248 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/248 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/248 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/28 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/28 [00:00<?, ? examples/s]


TRAINING BATCH CHECK
Batch shape: (1, 4096)
Ignored prompt/pad labels: 3948
Supervised completion labels: 148

[PASS] Training batch sanity check

Starting QLoRA training...


QLoRA TRAINING STARTED
Maximum optimizer steps : 270
Maximum epochs          : 10
Train batch size        : 1
Gradient accumulation   : 8
Step   1/270 |   0.37% | Epoch 0.04 | Elapsed: 11s | ETA: 50m 15s | 11.2s/step

Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,1.589025,1.583881,1.682059,0.672977,618108.000000
2,1.439880,1.417880,1.402667,0.691125,1236216.000000
3,1.278133,1.319393,1.312433,0.707592,1854324.000000
4,1.165538,1.263422,1.151541,0.715778,2472432.000000
5,1.120154,1.245882,1.073233,0.721153,3090540.000000
6,0.871712,1.233242,0.961555,0.726438,3708648.000000
7,0.904467,1.236652,0.945880,0.724219,4326756.000000
8,0.745792,1.269345,0.861005,0.725801,4944864.000000
9,0.679467,1.273538,0.864855,0.727243,5562972.000000


Step  27/270 |  10.00% | Epoch 1.00 | Elapsed: 4m 2s | ETA: 36m 25s | 9.0s/step

----------------------------------------------------------------------
VALIDATION COMPLETE — epoch 1.00
Validation loss: 1.583881
Elapsed time   : 4m 9s
----------------------------------------------------------------------
Step  54/270 |  20.00% | Epoch 2.00 | Elapsed: 8m 12s | ETA: 32m 49s | 9.1s/step

----------------------------------------------------------------------
VALIDATION COMPLETE — epoch 2.00
Validation loss: 1.417880
Elapsed time   : 8m 19s
----------------------------------------------------------------------
Step  81/270 |  30.00% | Epoch 3.00 | Elapsed: 12m 22s | ETA: 28m 52s | 9.2s/step

----------------------------------------------------------------------
VALIDATION COMPLETE — epoch 3.00
Validation loss: 1.319393
Elapsed time   : 12m 29s
----------------------------------------------------------------------
Step 108/270 |  40.00% | Epoch 4.00 | Elapsed: 16m 32s | ETA: 24m 48s | 9.2s/st

## Final 101-case generation

The query patient receives only **Age + Sex + Main Appeal**. The five Exp2D retrieved IDs are resolved through the 660-patient clinical lookup and used as historical reference records.


In [17]:
# ============================================================
# PROMPT + TEST LOOKUP HELPERS
# ============================================================

SYSTEM_PROMPT = (
    "You are a dental clinical documentation assistant. You will be given a patient's "
    "Age, Sex, and Main appeal (chief complaint), along with several similar PAST patient "
    "cases retrieved for reference.\n\n"
    "CRITICAL: The retrieved cases are REFERENCE MATERIAL from OTHER patients. They are "
    "NOT facts about the current patient. Do not assume the current patient has the same "
    "findings, tooth numbers, diagnoses, or treatments as any retrieved case unless the "
    "current patient's own Age/Sex/Main appeal genuinely supports it.\n\n"
    "Rules:\n"
    "1. NEVER invent or copy a tooth number, ICD code, diagnosis, finding, medication, "
    "procedure, or treatment that is not directly supported by the current patient's own "
    "Main appeal or by a clear, justified pattern across the retrieved cases.\n"
    "2. If there is insufficient evidence to determine a field, output exactly "
    '\"Not determined\" for that field. Do not guess.\n'
    "3. Output ONLY one valid JSON object with exactly these 5 keys, in this order: "
    "Oral Check, Diagnosis, Treatment plan, Handle, Doctor advices."
)

def build_query_text(row):
    parts = []

    age = clean_clinical_value(row.get("Age"))
    sex = clean_clinical_value(row.get("Sex"))
    main_appeal = clean_clinical_value(row.get("Main appeal"))

    if age:
        parts.append(f"Age: {age}")
    if sex:
        parts.append(f"Sex: {sex}")
    if main_appeal:
        parts.append(f"Main appeal: {main_appeal}")

    return ". ".join(parts)

def get_exemplar_records(exemplars, max_exemplars=5):
    out = []

    for ex in exemplars[:max_exemplars]:
        cid = norm_id(ex["case_id"])
        assert cid in clinical_lookup, f"Missing clinical lookup for retrieved patient {cid}"

        out.append({
            "case_id": cid,
            "score": float(
                ex.get(
                    "reranker_score",
                    ex.get("similarity", ex.get("csls_score", 0.0))
                )
            ),
            "record": clinical_lookup[cid],
        })

    return out

def build_user_prompt(query_text, exemplar_records):
    blocks = []

    for i, ex in enumerate(exemplar_records):
        label = chr(65 + i)

        lines = [
            f"[Reference Case {label}] (retrieval_score={ex['score']:.3f})"
        ]

        for field in REFERENCE_FIELDS:
            value = clean_clinical_value(ex["record"].get(field))
            if not value:
                value = "Not available"
            lines.append(f"  {field}: {value}")

        blocks.append("\n".join(lines))

    return (
        f"CURRENT PATIENT:\n{query_text}\n\n"
        "REFERENCE CASES (from OTHER patients — not facts about the current patient):\n\n"
        + "\n\n".join(blocks)
        + "\n\nWrite the current patient's own record as a JSON object with keys: "
          "Oral Check, Diagnosis, Treatment plan, Handle, Doctor advices."
    )

test_query_text = {}

for qid in rag_output_exp2d:
    cid = norm_id(qid)
    row = full_patient_df.loc[full_patient_df["Filename"] == cid]
    assert len(row) == 1, f"Expected one GT row for query {cid}"
    test_query_text[cid] = build_query_text(row.iloc[0])

print("Built leak-free query text for", len(test_query_text), "test patients.")


Built leak-free query text for 101 test patients.
Built leak-free query text for 101 test patients.


In [18]:
# ============================================================
# ROBUST JSON EXTRACTION
# No JSONCloseStoppingCriteria: MedGemma reasoning can itself
# contain braces. Generate first, then strip reasoning markers.
# ============================================================

def is_placeholder(value):
    val = str(value).strip()
    if val.lower() in {
        "", "not determined", "not available",
        "none", "null", "nan"
    }:
        return True
    try:
        parsed = ast.literal_eval(val)
        if isinstance(parsed, (dict, list, tuple)) and len(parsed) == 0:
            return True
    except (ValueError, SyntaxError):
        pass
    return False

def extract_final_json(raw_text):
    text = raw_text

    # MedGemma reasoning convention observed in the previous run:
    # <unused94>thought ... <unused95>FINAL
    if "<unused95>" in text:
        text = text.split("<unused95>", 1)[1]

    text = text.replace("<end_of_turn>", "").strip()

    # Prefer first decodable JSON object rather than greedy regex.
    decoder = json.JSONDecoder()

    for match in re.finditer(r"\{", text):
        try:
            obj, _ = decoder.raw_decode(text[match.start():])
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            continue

    return {}

def normalize_prediction(record):
    output = {}
    for field in TARGET_FIELDS:
        value = record.get(field, "Not determined")
        output[field] = "Not determined" if is_placeholder(value) else str(value).strip()
    return output

@torch.inference_mode()
def generate_report(model, tokenizer, query_text, exemplar_records, max_new_tokens=500):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(query_text, exemplar_records)},
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.15,
    )

    raw_text = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=False,
    )

    return normalize_prediction(extract_final_json(raw_text)), raw_text

print("Generation helpers ready.")


Generation helpers ready.
Generation helpers ready.


In [19]:
# ============================================================
# GENERATE FINAL 101 QLoRA PREDICTIONS
# Save incrementally so a late failure does not lose progress.
# ============================================================

qlora_model.config.use_cache = True
qlora_model.eval()

predictions = {}
raw_generations = {}

pred_path = os.path.join(OUTPUT_DIR, "predictions_exp2d_qlora.json")
raw_path = os.path.join(OUTPUT_DIR, "raw_generations_exp2d_qlora.json")

for i, (qid_raw, entry) in enumerate(rag_output_exp2d.items(), 1):
    qid = norm_id(qid_raw)

    query_text = test_query_text[qid]
    exemplar_records = get_exemplar_records(
        entry.get("retrieved_exemplars", [])
    )

    try:
        pred, raw = generate_report(
            qlora_model,
            qlora_tokenizer,
            query_text,
            exemplar_records,
        )
    except Exception as e:
        print(f"[WARN] case {qid} failed: {type(e).__name__}: {e}")
        pred = {f: "Not determined" for f in TARGET_FIELDS}
        raw = f"GENERATION_ERROR: {type(e).__name__}: {e}"

    predictions[qid] = pred
    raw_generations[qid] = raw

    # Checkpoint every case.
    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, indent=2, ensure_ascii=False)

    with open(raw_path, "w", encoding="utf-8") as f:
        json.dump(raw_generations, f, indent=2, ensure_ascii=False)

    if i % 10 == 0 or i == EXPECTED_TEST_N:
        print(f"{i}/{EXPECTED_TEST_N}")

assert len(predictions) == EXPECTED_TEST_N, (
    f"Expected {EXPECTED_TEST_N} predictions, got {len(predictions)}"
)

print("[PASS] Generated all 101 test predictions")
print("Saved:", pred_path)
print("Saved:", raw_path)


10/101
20/101
30/101
40/101
50/101
60/101
70/101
80/101
90/101
100/101
101/101
[PASS] Generated all 101 test predictions
Saved: /content/exp2d_qlora_final/predictions_exp2d_qlora.json
Saved: /content/exp2d_qlora_final/raw_generations_exp2d_qlora.json
10/101
20/101
30/101
40/101
50/101


KeyboardInterrupt: 

In [ ]:
# ============================================================
# GROUND TRUTH VS LLM PREDICTIONS
# ============================================================

comparison_rows = []

for qid_raw in rag_output_exp2d.keys():
    cid = norm_id(qid_raw)

    gt_rows = full_patient_df[
        full_patient_df["Filename"] == cid
    ]
    assert len(gt_rows) == 1

    gt = gt_rows.iloc[0]
    pred = predictions[cid]

    row = {
        "case_id": cid,
        "Age": clean_clinical_value(gt.get("Age")),
        "Sex": clean_clinical_value(gt.get("Sex")),
        "Main appeal": clean_clinical_value(gt.get("Main appeal")),
    }

    for field in TARGET_FIELDS:
        gt_value = clean_clinical_value(gt.get(field))
        if not gt_value:
            gt_value = "Not determined"

        row[f"GT_{field}"] = gt_value
        row[f"PRED_{field}"] = pred.get(field, "Not determined")

    comparison_rows.append(row)

ground_truth_vs_llm = pd.DataFrame(comparison_rows)

assert len(ground_truth_vs_llm) == EXPECTED_TEST_N
assert ground_truth_vs_llm["case_id"].nunique() == EXPECTED_TEST_N

csv_path = os.path.join(
    OUTPUT_DIR,
    "ground_truth_vs_qlora_predictions.csv",
)
xlsx_path = os.path.join(
    OUTPUT_DIR,
    "ground_truth_vs_qlora_predictions.xlsx",
)

ground_truth_vs_llm.to_csv(csv_path, index=False)
ground_truth_vs_llm.to_excel(xlsx_path, index=False)

print("Saved:", csv_path)
print("Saved:", xlsx_path)

ground_truth_vs_llm.head()


Saved: /content/exp2d_qlora_final/ground_truth_vs_qlora_predictions.csv
Saved: /content/exp2d_qlora_final/ground_truth_vs_qlora_predictions.xlsx


,case_id,Age,Sex,Main appeal,GT_Oral Check,PRED_Oral Check,GT_Diagnosis,PRED_Diagnosis,GT_Treatment plan,PRED_Treatment plan,GT_Handle,PRED_Handle,GT_Doctor advices,PRED_Doctor advices
0,88,19,female,"Crowded teeth, consultation for correction","tooth 15 (upper right second premolar), tooth ...",Not determined,"tooth 15 (upper right second premolar), tooth ...",Not determined,"tooth 15 (upper right second premolar), tooth ...",Not determined,"*15, *25 Explain the condition, treatment plan...",Not determined,Follow-up for discomfort. Follow-up for discom...,Not determined
1,378,17,female,The right upper back tooth has been painful du...,tooth 16 (upper right first molar) temporarily...,tooth 16 (upper right first molar),tooth 16 (upper right first molar) tooth defec...,acute apical abscess,tooth 16 (upper right first molar). filling tr...,recommended removal.,"6(1) Explain the condition, treatment plan, co...",Taking medicine routinely refers to taking ant...,Do not bite hard objects and return to the cli...,Routine antibiotic courses include Penicillin ...
2,219,46,female,The upper left back tooth has been missing for...,tooth 13 (upper right canine) tooth 24 (upper ...,*28*CBCT shows low density shadows around the ...,tooth 13 (upper right canine) tooth 24 (upper ...,Missing teeth,tooth 13 (upper right canine) tooth 24 (upper ...,*Extraction recommendation.*,"13 24 44 45 Gingival retraction, decay prepara...",*The patient requested filming and went home t...,Follow-up for discomfort. Routine medical advi...,Follow-up for discomfort.
3,98,40,male,There has been a cavity in the lower left back...,67 a loose white filling was seen in the inter...,tooth 38 (lower left third molar) the carious ...,"tooth 36 (lower left first molar), tooth 37 (l...",chronic apical periodontitis,"tooth 36 (lower left first molar), tooth 37 (l...",root canal retreatment + post space protection...,"36, 37 Clean the caries, remove part of the or...",*Local iodine disinfection *Explain the condit...,Not determined,Follow-up for discomfort.
4,470,28,female,Self-reported tooth decay in the mouth and req...,tooth 47 (lower right second molar). there are...,"*28 Palate medially impacted, buccal mucosa co...",tooth 47 (lower right second molar) deep carie...,Malformed teeth (K03.20x),tooth 47 (lower right second molar) recommend ...,Not determined,"The patient explained the condition, treatment...",The patient did not choose a date for treatment.,Routine medical advice and follow-up consultat...,Routine medical advice and follow-up consultat...


## Final 9-metric evaluation

The evaluation is run only on **Exp2D + QLoRA**.

Metric categories:
1. BLEU-4
2. ROUGE-L
3. METEOR
4. CIDEr
5. FDI Tooth-Set F1
6. ICD-10 Set F1
7. Hallucination / factual-consistency rate
8. Retrieval copy rate
9. Fabricated tooth-reference rate


In [ ]:
# ============================================================
# METRIC SETUP
# ============================================================

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("omw-1.4", quiet=True)

import sacrebleu
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize

try:
    from pycocoevalcap.cider.cider import Cider
    cider_scorer = Cider()
    CIDER_AVAILABLE = True
except Exception as e:
    print("CIDEr unavailable:", e)
    CIDER_AVAILABLE = False

rouge_l_scorer = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True,
)

# Standard adult FDI permanent-tooth notation 11-48.
FDI_PATTERN = re.compile(r"(?<!\d)(?:1[1-8]|2[1-8]|3[1-8]|4[1-8])(?!\d)")
ICD10_PATTERN = re.compile(r"\bK\d{2}(?:\.\d{1,3})?\b", re.IGNORECASE)

def extract_tooth_set(text):
    return set(FDI_PATTERN.findall(str(text)))

def extract_icd_set(text):
    return {x.upper() for x in ICD10_PATTERN.findall(str(text))}

def full_report_text(record):
    return " ".join(
        clean_clinical_value(record.get(f))
        for f in TARGET_FIELDS
    )

def set_f1(pred_set, gt_set):
    if not pred_set and not gt_set:
        return 1.0
    if not pred_set or not gt_set:
        return 0.0

    tp = len(pred_set & gt_set)
    precision = tp / len(pred_set)
    recall = tp / len(gt_set)

    return (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

def get_ngrams(text, n=4):
    words = re.findall(r"\w+", str(text).lower())
    return {
        tuple(words[i:i+n])
        for i in range(len(words) - n + 1)
    } if len(words) >= n else set()

print("Metric helpers ready.")


Metric helpers ready.


[nltk_data] Error loading wordnet: Security Violation
[nltk_data]     [pathsec.urlopen]: refusing a proxied fetch of
[nltk_data]     'https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/index.xml'. A configured proxy performs the
[nltk_data]     egress, so NLTK cannot pin the validated IP and SSRF
[nltk_data]     protection cannot be enforced (CWE-918). If and only
[nltk_data]     if the proxy is trusted to be SSRF-safe, opt in via
[nltk_data]     NLTK_ALLOW_PROXIED_URLOPEN=1 or
[nltk_data]     nltk.pathsec.ALLOW_PROXIED_FETCH=True.
[nltk_data] Error loading punkt: Security Violation [pathsec.urlopen]:
[nltk_data]     refusing a proxied fetch of
[nltk_data]     'https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/index.xml'. A configured proxy performs the
[nltk_data]     egress, so NLTK cannot pin the validated IP and SSRF
[nltk_data]     protection cannot be enforced (CWE-918). If and only
[nltk_data]     if the proxy is trusted to be SSR

In [ ]:
# ============================================================
# 9-METRIC EVALUATION — Exp2D + QLoRA ONLY
# ============================================================

def evaluate_9_metrics(preds, rag_source):
    hyps, refs = [], []
    rouge_scores, meteor_scores = [], []
    cider_gts, cider_res = {}, {}

    tooth_f1_scores = []
    icd_f1_scores = []

    halluc_per_patient = []
    halluc_micro_num = 0
    halluc_micro_den = 0

    copy_rates = []
    fabricated_tooth_rates = []

    per_case = []

    for cid_raw in rag_source.keys():
        cid = norm_id(cid_raw)
        pred = preds[cid]

        gt_rows = full_patient_df[
            full_patient_df["Filename"] == cid
        ]
        assert len(gt_rows) == 1
        gt = gt_rows.iloc[0]

        gt_record = {
            f: clean_clinical_value(gt.get(f))
            for f in TARGET_FIELDS
        }

        gt_text = full_report_text(gt_record)
        pred_text = full_report_text(pred)

        hyps.append(pred_text)
        refs.append(gt_text)

        rouge_l = rouge_l_scorer.score(
            gt_text, pred_text
        )["rougeL"].fmeasure
        rouge_scores.append(rouge_l)

        try:
            meteor = meteor_score(
                [word_tokenize(gt_text)],
                word_tokenize(pred_text),
            )
        except Exception:
            meteor = 0.0
        meteor_scores.append(meteor)

        cider_gts[cid] = [gt_text]
        cider_res[cid] = [pred_text]

        pred_teeth = extract_tooth_set(pred_text)
        gt_teeth = extract_tooth_set(gt_text)

        pred_icd = extract_icd_set(pred_text)
        gt_icd = extract_icd_set(gt_text)

        tooth_f1 = set_f1(pred_teeth, gt_teeth)
        icd_f1 = set_f1(pred_icd, gt_icd)

        tooth_f1_scores.append(tooth_f1)
        icd_f1_scores.append(icd_f1)

        pred_entities = pred_teeth | pred_icd
        gt_entities = gt_teeth | gt_icd
        halluc_entities = pred_entities - gt_entities

        halluc_rate = (
            len(halluc_entities) / len(pred_entities)
            if pred_entities else 0.0
        )
        halluc_per_patient.append(halluc_rate)

        halluc_micro_num += len(halluc_entities)
        halluc_micro_den += len(pred_entities)

        entry = rag_source[cid_raw]
        exemplar_records = get_exemplar_records(
            entry.get("retrieved_exemplars", [])
        )

        exemplar_text = " ".join(
            full_report_text(ex["record"])
            for ex in exemplar_records
        )

        exemplar_teeth = set()
        for ex in exemplar_records:
            exemplar_teeth |= extract_tooth_set(
                full_report_text(ex["record"])
            )

        pred_4grams = get_ngrams(pred_text, 4)
        exemplar_4grams = get_ngrams(exemplar_text, 4)
        gt_4grams = get_ngrams(gt_text, 4)

        copy_ngrams = (
            pred_4grams & exemplar_4grams
        ) - gt_4grams

        copy_rate = (
            len(copy_ngrams) / len(pred_4grams)
            if pred_4grams else 0.0
        )
        copy_rates.append(copy_rate)

        fabricated_teeth = (
            pred_teeth - gt_teeth - exemplar_teeth
        )
        fabricated_rate = (
            len(fabricated_teeth) / len(pred_teeth)
            if pred_teeth else 0.0
        )
        fabricated_tooth_rates.append(fabricated_rate)

        per_case.append({
            "case_id": cid,
            "ROUGE-L": rouge_l,
            "METEOR": meteor,
            "FDI_Tooth_F1": tooth_f1,
            "ICD10_F1": icd_f1,
            "Hallucination_Rate": halluc_rate,
            "Retrieval_Copy_Rate": copy_rate,
            "Fabricated_Tooth_Rate": fabricated_rate,
        })

    assert len(hyps) == EXPECTED_TEST_N

    bleu4 = sacrebleu.corpus_bleu(
        hyps, [refs]
    ).score / 100.0

    if CIDER_AVAILABLE:
        cider_score, _ = cider_scorer.compute_score(
            cider_gts,
            cider_res,
        )
        cider_score = float(cider_score)
    else:
        cider_score = None

    results = {
        "n": len(hyps),
        "BLEU-4": float(bleu4),
        "ROUGE-L": float(np.mean(rouge_scores)),
        "METEOR": float(np.mean(meteor_scores)),
        "CIDEr": cider_score,
        "FDI_Tooth_F1_macro": float(np.mean(tooth_f1_scores)),
        "ICD10_F1_macro": float(np.mean(icd_f1_scores)),
        "Hallucination_per_patient_mean": float(np.mean(halluc_per_patient)),
        "Hallucination_micro_pooled": (
            halluc_micro_num / halluc_micro_den
            if halluc_micro_den else 0.0
        ),
        "Retrieval_Copy_Rate": float(np.mean(copy_rates)),
        "Fabricated_Tooth_Rate": float(np.mean(fabricated_tooth_rates)),
    }

    return results, pd.DataFrame(per_case)

results, per_case_metrics = evaluate_9_metrics(
    predictions,
    rag_output_exp2d,
)

summary_df = pd.DataFrame(
    [results],
    index=["Exp2D + QLoRA"],
)

summary_csv = os.path.join(
    OUTPUT_DIR,
    "qlora_9_metric_evaluation.csv",
)
per_case_csv = os.path.join(
    OUTPUT_DIR,
    "qlora_per_case_metrics.csv",
)

summary_df.to_csv(summary_csv)
per_case_metrics.to_csv(per_case_csv, index=False)

display(summary_df)

print("Saved:", summary_csv)
print("Saved:", per_case_csv)


,n,BLEU-4,ROUGE-L,METEOR,CIDEr,FDI_Tooth_F1_macro,ICD10_F1_macro,Hallucination_per_patient_mean,Hallucination_micro_pooled,Retrieval_Copy_Rate,Fabricated_Tooth_Rate
Exp2D + QLoRA,101,0.000438,0.108161,0.0,0.00516,0.133996,0.950495,0.517571,0.723926,0.090135,0.083247


Saved: /content/exp2d_qlora_final/qlora_9_metric_evaluation.csv
Saved: /content/exp2d_qlora_final/qlora_per_case_metrics.csv


In [20]:
from google.colab import drive
drive.mount("/content/drive")

import os
import shutil
from datetime import datetime

SOURCE = "/content/exp2d_qlora_final"

# Timestamp prevents accidentally overwriting this run
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

DEST = (
    f"/content/drive/MyDrive/"
    f"exp2d_qlora_final_{timestamp}"
)

assert os.path.exists(SOURCE), f"Source folder not found: {SOURCE}"

print("Copying:")
print(" FROM:", SOURCE)
print(" TO:  ", DEST)

shutil.copytree(
    SOURCE,
    DEST,
    dirs_exist_ok=False
)

print("\n[PASS] Complete folder copied to Google Drive")
print("Saved at:", DEST)

Mounted at /content/drive
Copying:
 FROM: /content/exp2d_qlora_final
 TO:   /content/drive/MyDrive/exp2d_qlora_final_20260903_173822

[PASS] Complete folder copied to Google Drive
Saved at: /content/drive/MyDrive/exp2d_qlora_final_20260903_173822


In [25]:
# ============================================================
# FINAL 101-CASE EVALUATION FROM SAVED EXCEL
# NO QLoRA TRAINING / NO GENERATION REQUIRED
# ============================================================

# Install if needed:
!pip install -q sacrebleu rouge-score nltk pycocoevalcap openpyxl

import os
import re
import json
import ast
import numpy as np
import pandas as pd

import sacrebleu
from rouge_score import rouge_scorer

import nltk
from nltk.translate.meteor_score import meteor_score
from nltk import pathsec

from pycocoevalcap.cider.cider import Cider


# ============================================================
# PATHS
# ============================================================

XLSX_PATH = "/content/exp2d_qlora_final/ground_truth_vs_qlora_predictions.csv"
OUTPUT_DIR = "/content/final_qlora_evaluation_metrics"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# NLTK SETUP
# IMPORTANT: DO NOT silently convert METEOR errors to zero
# ============================================================

# Allow NLTK to fetch data through proxy if needed
nltk.pathsec.ALLOW_PROXIED_FETCH = True

# Required mainly for METEOR's WordNet synonym matching.
for pkg in ["wordnet", "omw-1.4"]:
    try:
        nltk.download(pkg, quiet=False)
    except Exception as e:
        print(f"[WARN] NLTK download issue for {pkg}: {e}")

# Explicitly verify WordNet.
try:
    from nltk.corpus import wordnet
    _ = wordnet.synsets("tooth")
    print("[PASS] NLTK WordNet available")
except Exception as e:
    raise RuntimeError(
        "WordNet is unavailable. Do NOT compute METEOR until this is fixed."
    ) from e


# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(XLSX_PATH)

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

# TARGET_FIELDS must be defined before this block for constructing GT/PRED reports
# Moving TARGET_FIELDS definition here or ensuring it's available.
# For now, assuming it's available from a previous cell or define it here.
TARGET_FIELDS = [
    "Oral Check",
    "Diagnosis",
    "Treatment plan",
    "Handle",
    "Doctor advices",
]

required_columns = {"case_id"} # Only assert 'case_id' as other reports are constructed

for field in TARGET_FIELDS:
    required_columns.add(f"GT_{field}")
    required_columns.add(f"PRED_{field}")

missing = required_columns - set(df.columns)

assert not missing, f"Missing columns: {missing}"

assert len(df) == 101, f"Expected 101 patients, found {len(df)}"
assert df["case_id"].nunique() == 101, "Duplicate Case IDs detected"

# Check for non-null values in GT_ and PRED_ fields
for field in TARGET_FIELDS:
    assert df[f"GT_{field}"].notna().all(), f"Missing ground-truth for {field}"
    assert df[f"PRED_{field}"].notna().all(), f"Missing LLM predictions for {field}"

print("\n[PASS] 101 unique test patients")
print("[PASS] GT reports available")
print("[PASS] LLM predictions available")


# ============================================================
# FIND RERANKED EXEMPLAR COLUMN
# ============================================================

possible_exemplar_cols = [
    "Retrieved Exemplars (Reranked)",
    "Retrieved Exemplars",
    "Reranked Retrieved Exemplars",
]

EXEMPLAR_COL = None

for c in possible_exemplar_cols:
    if c in df.columns:
        EXEMPLAR_COL = c
        break

print("\nReranked exemplar column:", EXEMPLAR_COL)

if EXEMPLAR_COL is None:
    print(
        "[WARN] No reranked exemplar column found. "
        "Retrieval Copy Rate will be unavailable."
    )


# ============================================================
# NORMALIZATION
# ============================================================

def normalize(text):

    if pd.isna(text):
        return ""

    text = str(text)

    text = (
        text
        .replace("&#39;", "'")
        .replace("&amp;", "&")
        .replace("&quot;", '"')
    )

    text = re.sub(r"\s+", " ", text).strip().lower()

    return text


def tokenize(text):
    # METEOR accepts already-tokenized input.
    # This avoids dependency on punkt/word_tokenize.
    return normalize(text).split()


def get_ngrams(tokens, n=4):

    if len(tokens) < n:
        return set()

    return {
        tuple(tokens[i:i+n])
        for i in range(len(tokens) - n + 1)
    }


def all_ngrams_list(tokens, n=4):

    if len(tokens) < n:
        return []

    return [
        tuple(tokens[i:i+n])
        for i in range(len(tokens) - n + 1)
    ]


# ============================================================
# CLINICAL ENTITY EXTRACTION
# ============================================================

# Adult permanent FDI teeth 11-18, 21-28, 31-38, 41-48.
#
# Require a clinical tooth marker to reduce false matches from:.
# ages, durations, measurements, etc.

TOOTH_WORD_RE = re.compile(
    r"\btooth\s*[*#:]?\s*"
    r"(1[1-8]|2[1-8]|3[1-8]|4[1-8])\b",
    re.IGNORECASE,
)

TOOTH_STAR_RE = re.compile(
    r"(?<!\d)\*\s*"
    r"(1[1-8]|2[1-8]|3[1-8]|4[1-8])\b"
)


# MMDental-style ICD codes.
#
# Handles:
# K07.305
# k 07.305
# K-07.305
#
# Also permits codes without decimal where present.

ICD10_RE = re.compile(
    r"\b([A-Za-z])[\\s-]"
    r"(\d{2}(?:\.\d{1,4})?)\b",
    re.IGNORECASE,
)


def extract_fdi_teeth(text):

    text = normalize(text)

    return (
        set(TOOTH_WORD_RE.findall(text))
        |
        set(TOOTH_STAR_RE.findall(text))
    )


def extract_icd10(text):

    text = normalize(text)

    return {
        f"{letter.upper()}{digits}"
        for letter, digits in ICD10_RE.findall(text)
    }


# ============================================================
# SET METRICS
# ============================================================

def entity_counts(pred_set, gt_set):

    tp = len(pred_set & gt_set)
    fp = len(pred_set - gt_set)
    fn = len(gt_set - pred_set)

    return tp, fp, fn


def set_f1(pred_set, gt_set):

    # Keep empty-empty as NaN for macro entity F1.
    #
    # This prevents:
    #
    # GT ICD = {}
    # Pred ICD = {}
    #
    # from artificially contributing F1 = 1.

    if not pred_set and not gt_set:
        return np.nan

    if not pred_set or not gt_set:
        return 0.0

    tp = len(pred_set & gt_set)

    precision = tp / len(pred_set)
    recall = tp / len(gt_set)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


# ============================================================
# RETRIEVED EXEMPLAR PARSING
# ============================================================

# TARGET_FIELDS definition duplicated here to ensure it's available in this cell
TARGET_FIELDS = [
    "Oral Check",
    "Diagnosis",
    "Treatment plan",
    "Handle",
    "Doctor advices",
]


def parse_exemplars(cell):

    if pd.isna(cell):
        return []

    if isinstance(cell, list):
        return cell

    text = str(cell).strip()

    if not text:
        return []

    try:
        value = json.loads(text)
        return value if isinstance(value, list) else []
    except Exception:
        pass

    try:
        value = ast.literal_eval(text)
        return value if isinstance(value, list) else []
    except Exception:
        return []


def exemplar_full_text(exemplar):

    if not isinstance(exemplar, dict):
        return ""

    record = exemplar.get("record", exemplar)

    if not isinstance(record, dict):
        return ""

    parts = []

    for field in TARGET_FIELDS:

        value = record.get(field, "")

        if value is None:
            continue

        value = str(value).strip()

        if value and value.lower() not in {
            "nan",
            "none",
            "null",
            "not available",
        }:
            parts.append(value)

    return " ".join(parts)


# ============================================================
# INTEGRITY CHECKS
# ============================================================

print("\n" + "=" * 70)
print("DATA INTEGRITY CHECK")
print("=" * 70)

# Concatenate PRED_ fields to form a single string for duplicate check
def get_full_pred_report(row):
    return " ".join(str(row[f"PRED_{field}"]) for field in TARGET_FIELDS)

normalized_predictions = df.apply(get_full_pred_report, axis=1).apply(normalize)

duplicate_mask = normalized_predictions.duplicated(
    keep=False
)

duplicate_cases = []

if duplicate_mask.any():

    temp = df.loc[
        duplicate_mask,
        ["case_id"] + [f"PRED_{field}" for field in TARGET_FIELDS]
    ].copy()

    temp["norm"] = temp.apply(get_full_pred_report, axis=1).apply(normalize)

    for _, group in temp.groupby("norm"):

        ids = group["case_id"].tolist()

        if len(ids) > 1:
            duplicate_cases.append(ids)

if duplicate_cases:

    print(
        "[WARN] Identical generated reports "
        "found across cases:"
    )

    for ids in duplicate_cases:
        print("   ", ids)

else:
    print(
        "[PASS] No duplicate generated reports "
        "across different patients"
    )


# ============================================================
# METRIC OBJECTS
# ============================================================

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True,
)

cider = Cider()


# ============================================================
# EVALUATION
# ============================================================

rows = []

hyps = []
refs = []

cider_gts = {}
cider_res = {}

# Micro entity counters
fdi_tp = fdi_fp = fdi_fn = 0
icd_tp = icd_fp = icd_fn = 0

halluc_num = 0
halluc_den = 0


for _, row in df.iterrows():

    case_id = str(row["case_id"])

    # Construct gt_raw and pred_raw from GT_ and PRED_ columns
    gt_raw_parts = [str(row[f"GT_{field}"]) for field in TARGET_FIELDS]
    pred_raw_parts = [str(row[f"PRED_{field}"]) for field in TARGET_FIELDS]

    gt_raw = " ".join(gt_raw_parts)
    pred_raw = " ".join(pred_raw_parts)

    gt_norm = normalize(gt_raw)
    pred_norm = normalize(pred_raw)

    gt_tokens = tokenize(gt_raw)
    pred_tokens = tokenize(pred_raw)

    hyps.append(pred_norm)
    refs.append(gt_norm)


    # --------------------------------------------------------
    # ROUGE-L
    # --------------------------------------------------------

    rouge_l = rouge.score(
        gt_norm,
        pred_norm,
    )["rougeL"].fmeasure


    # --------------------------------------------------------
    # METEOR
    # --------------------------------------------------------

    # No silent try/except.
    # If METEOR fails, evaluation stops instead of writing fake zeros.

    meteor = meteor_score(
        [gt_tokens],
        pred_tokens,
    )


    # --------------------------------------------------------
    # CIDEr
    # --------------------------------------------------------

    cider_gts[case_id] = [gt_norm]
    cider_res[case_id] = [pred_norm]


    # --------------------------------------------------------
    # FDI
    # --------------------------------------------------------

    gt_teeth = extract_fdi_teeth(gt_raw)
    pred_teeth = extract_fdi_teeth(pred_raw)

    fdi_f1 = set_f1(
        pred_teeth,
        gt_teeth,
    )

    tp, fp, fn = entity_counts(
        pred_teeth,
        gt_teeth,
    )

    fdi_tp += tp
    fdi_fp += fp
    fdi_fn += fn


    # --------------------------------------------------------
    # ICD
    # --------------------------------------------------------

    gt_icd = extract_icd10(gt_raw)
    pred_icd = extract_icd10(pred_raw)

    icd_f1 = set_f1(
        pred_icd,
        gt_icd,
    )

    tp, fp, fn = entity_counts(
        pred_icd,
        gt_icd,
    )

    icd_tp += tp
    icd_fp += fp
    icd_fn += fn


    # --------------------------------------------------------
    # ENTITY HALLUCINATION
    #
    # IMPORTANT:
    # This is specifically FDI + ICD entity hallucination,
    # NOT a general clinical hallucination metric.
    # --------------------------------------------------------

    gt_entities = gt_teeth | gt_icd
    pred_entities = pred_teeth | pred_icd

    unsupported = (
        pred_entities - gt_entities
    )

    if pred_entities:

        entity_halluc_rate = (
            len(unsupported)
            / len(pred_entities)
        )

        halluc_num += len(unsupported)
        halluc_den += len(pred_entities)

    else:
        entity_halluc_rate = np.nan


    # --------------------------------------------------------
    # RETRIEVAL METRICS
    # --------------------------------------------------------

    copy_rate = np.nan
    fabricated_tooth_rate = np.nan
    n_exemplars = 0

    if EXEMPLAR_COL is not None:

        exemplars = parse_exemplars(
            row[EXEMPLAR_COL]
        )

        n_exemplars = len(exemplars)

        exemplar_texts = [
            exemplar_full_text(ex)
            for ex in exemplars
        ]

        exemplar_texts = [
            x for x in exemplar_texts
            if x
        ]

        exemplar_teeth = set()
        exemplar_ngrams = set()

        for text in exemplar_texts:

            exemplar_teeth |= (
                extract_fdi_teeth(text)
            )

            exemplar_ngrams |= (
                get_ngrams(
                    tokenize(text),
                    4,
                )
            )


        # ----------------------------
        # Retrieval Copy Rate
        # ----------------------------

        pred_ngrams = all_ngrams_list(
            pred_tokens,
            4,
        )

        gt_ngrams = get_ngrams(
            gt_tokens,
            4,
        )

        if pred_ngrams:

            copied = [
                ng
                for ng in pred_ngrams
                if (
                    ng in exemplar_ngrams
                    and ng not in gt_ngrams
                )
            ]

            copy_rate = (
                len(copied)
                / len(pred_ngrams)
            )


        # ----------------------------
        # Fabricated Tooth Rate
        # ----------------------------

        if pred_teeth:

            supported_teeth = (
                gt_teeth
                |
                exemplar_teeth
            )

            fabricated = (
                pred_teeth
                - supported_teeth
            )

            fabricated_tooth_rate = (
                len(fabricated)
                / len(pred_teeth)
            )


    # --------------------------------------------------------
    # STORE CASE
    # --------------------------------------------------------

    rows.append({

        "Case ID": case_id,

        "ROUGE-L": rouge_l,
        "METEOR": meteor,

        "FDI_Tooth_F1": fdi_f1,
        "ICD10_F1": icd_f1,

        "Entity_Hallucination_Rate":
            entity_halluc_rate,

        "Retrieval_Copy_Rate":
            copy_rate,

        "Fabricated_Tooth_Rate":
            fabricated_tooth_rate,

        "N_Retrieved_Exemplars":
            n_exemplars,

        "GT_Teeth":
            sorted(gt_teeth),

        "Pred_Teeth":
            sorted(pred_teeth),

        "GT_ICD10":
            sorted(gt_icd),

        "Pred_ICD10":
            sorted(pred_icd),
    })


per_case = pd.DataFrame(rows)


# ============================================================
# BLEU-4 — CORPUS
# ============================================================

bleu4 = (
    sacrebleu.corpus_bleu(
        hyps,
        [refs],
    ).score
    / 100.0
)


# ============================================================
# BLEU-4 — PER CASE
# ============================================================

sentence_bleu = []

for hyp, ref in zip(hyps, refs):

    score = (
        sacrebleu.sentence_bleu(
            hyp,
            [ref],
        ).score
        / 100.0
    )

    sentence_bleu.append(score)

per_case.insert(
    1,
    "BLEU-4",
    sentence_bleu,
)


# ============================================================
# CIDEr
# ============================================================

cider_score, cider_case = (
    cider.compute_score(
        cider_gts,
        cider_res,
    )
)

case_order = list(
    cider_res.keys()
)

cider_map = dict(
    zip(
        case_order,
        cider_case,
    )
)

per_case["CIDEr"] = (
    per_case["Case ID"]
    .astype(str)
    .map(cider_map)
)


# ============================================================
# MICRO F1
# ============================================================

def micro_f1(tp, fp, fn):

    precision = (
        tp / (tp + fp)
        if tp + fp
        else np.nan
    )

    recall = (
        tp / (tp + fn)
        if tp + fn
        else np.nan
    )

    if (
        pd.isna(precision)
        or pd.isna(recall)
        or precision + recall == 0
    ):
        return 0.0

    return (
        2
        * precision
        * recall
        / (precision + recall)
    )


fdi_micro_f1 = micro_f1(
    fdi_tp,
    fdi_fp,
    fdi_fn,
)

icd_micro_f1 = micro_f1(
    icd_tp,
    icd_fp,
    icd_fn,
)


# ============================================================
# SUMMARY
# ============================================================

def mean_valid(column):

    values = per_case[column].dropna()

    if len(values) == 0:
        return np.nan

    return float(values.mean())


def std_valid(column):

    values = per_case[column].dropna()

    if len(values) <= 1:
        return np.nan

    return float(values.std())


summary_rows = [

    {
        "Metric": "BLEU-4",
        "Mean": bleu4,
        "Std": np.nan,
        "N_Evaluable": 101,
    },

    {
        "Metric": "ROUGE-L",
        "Mean": mean_valid("ROUGE-L"),
        "Std": std_valid("ROUGE-L"),
        "N_Evaluable":
            int(per_case["ROUGE-L"].notna().sum()),
    },

    {
        "Metric": "METEOR",
        "Mean": mean_valid("METEOR"),
        "Std": std_valid("METEOR"),
        "N_Evaluable":
            int(per_case["METEOR"].notna().sum()),
    },

    {
        "Metric": "CIDEr",
        "Mean": float(cider_score),
        "Std": np.nan,
        "N_Evaluable": 101,
    },

    {
        "Metric": "FDI Tooth-Set F1 (macro, entity-evaluable)",
        "Mean": mean_valid("FDI_Tooth_F1"),
        "Std": std_valid("FDI_Tooth_F1"),
        "N_Evaluable":
            int(per_case["FDI_Tooth_F1"].notna().sum()),
    },

    {
        "Metric": "FDI Tooth-Set F1 (micro)",
        "Mean": fdi_micro_f1,
        "Std": np.nan,
        "N_Evaluable": 101,
    },

    {
        "Metric": "ICD-10 Set F1 (macro, entity-evaluable)",
        "Mean": mean_valid("ICD10_F1"),
        "Std": std_valid("ICD10_F1"),
        "N_Evaluable":
            int(per_case["ICD10_F1"].notna().sum()),
    },

    {
        "Metric": "ICD-10 Set F1 (micro)",
        "Mean": icd_micro_f1,
        "Std": np.nan,
        "N_Evaluable": 101,
    },

    {
        "Metric":
            "FDI+ICD Entity Hallucination Rate (patient mean)",

        "Mean":
            mean_valid("Entity_Hallucination_Rate"),

        "Std":
            std_valid("Entity_Hallucination_Rate"),

        "N_Evaluable":
            int(
                per_case[
                    "Entity_Hallucination_Rate"
                ].notna().sum()
            ),
    },

    {
        "Metric":
            "FDI+ICD Entity Hallucination Rate (micro)",

        "Mean":
            (
                halluc_num / halluc_den
                if halluc_den
                else np.nan
            ),

        "Std": np.nan,

        "N_Evaluable":
            int(
                per_case[
                    "Entity_Hallucination_Rate"
                ].notna().sum()
            ),
    },

    {
        "Metric": "Retrieval Copy Rate",
        "Mean":
            mean_valid("Retrieval_Copy_Rate"),
        "Std":
            std_valid("Retrieval_Copy_Rate"),
        "N_Evaluable":
            int(
                per_case[
                    "Retrieval_Copy_Rate"
                ].notna().sum()
            ),
    },

    {
        "Metric":
            "Fabricated Tooth-Reference Rate",

        "Mean":
            mean_valid("Fabricated_Tooth_Rate"),

        "Std":
            std_valid("Fabricated_Tooth_Rate"),

        "N_Evaluable":
            int(
                per_case[
                    "Fabricated_Tooth_Rate"
                ].notna().sum()
            ),
    },
]

summary = pd.DataFrame(
    summary_rows
)


# ============================================================
# SAVE
# ============================================================

per_case_path = os.path.join(
    OUTPUT_DIR,
    "final_metrics_per_case.csv",
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "final_metrics_summary.csv",
)

xlsx_output = os.path.join(
    OUTPUT_DIR,
    "final_101_case_evaluation.xlsx",
)

per_case.to_csv(
    per_case_path,
    index=False,
)

summary.to_csv(
    summary_path,
    index=False,
)

with pd.ExcelWriter(
    xlsx_output,
    engine="openpyxl",
) as writer:

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False,
    )

    per_case.to_excel(
        writer,
        sheet_name="Per Case",
        index=False,
    )

    df.to_excel(
        writer,
        sheet_name="GT vs Prediction",
        index=False,
    )


# ============================================================
# DISPLAY
# ============================================================

pd.set_option(
    "display.float_format",
    lambda x: f"{x:.4f}"
)

print("\n" + "=" * 70)
print("FINAL 101-CASE QLoRA EVALUATION")
print("=" * 70)

display(summary)

print("\nEntity coverage:")
print(
    "Patients containing GT FDI:",
    sum(
        bool(extract_fdi_teeth(x))
        for x in df.apply(lambda row: " ".join(str(row[f"GT_{field}"]) for field in TARGET_FIELDS), axis=1)
    ),
    "/ 101"
)

print(
    "Patients containing predicted FDI:",
    sum(
        bool(extract_fdi_teeth(x))
        for x in df.apply(lambda row: " ".join(str(row[f"PRED_{field}"]) for field in TARGET_FIELDS), axis=1)
    ),
    "/ 101"
)

print(
    "Patients containing GT ICD:",
    sum(
        bool(extract_icd10(x))
        for x in df.apply(lambda row: " ".join(str(row[f"GT_{field}"]) for field in TARGET_FIELDS), axis=1)
    ),
    "/ 101"
)

print(
    "Patients containing predicted ICD:",
    sum(
        bool(extract_icd10(x))
        for x in df.apply(lambda row: " ".join(str(row[f"PRED_{field}"]) for field in TARGET_FIELDS), axis=1)
    ),
    "/ 101"
)

print("\nSaved:")
print(per_case_path)
print(summary_path)
print(xlsx_output)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


[PASS] NLTK WordNet available
Shape: (101, 14)
Columns:
['case_id', 'Age', 'Sex', 'Main appeal', 'GT_Oral Check', 'PRED_Oral Check', 'GT_Diagnosis', 'PRED_Diagnosis', 'GT_Treatment plan', 'PRED_Treatment plan', 'GT_Handle', 'PRED_Handle', 'GT_Doctor advices', 'PRED_Doctor advices']

[PASS] 101 unique test patients
[PASS] GT reports available
[PASS] LLM predictions available

Reranked exemplar column: None
[WARN] No reranked exemplar column found. Retrieval Copy Rate will be unavailable.

DATA INTEGRITY CHECK
[WARN] Identical generated reports found across cases:
    [88, 414, 596, 576, 166, 46, 445, 638, 114, 586, 448, 260, 613, 225, 23, 38]

FINAL 101-CASE QLoRA EVALUATION


,Metric,Mean,Std,N_Evaluable
0,BLEU-4,0.0005,NaN,101
1,ROUGE-L,0.1088,0.0874,101
2,METEOR,0.0688,0.0847,101
3,CIDEr,0.0051,NaN,101
4,"FDI Tooth-Set F1 (macro, entity-evaluable)",0.0878,0.2181,98
5,FDI Tooth-Set F1 (micro),0.1253,NaN,101
6,"ICD-10 Set F1 (macro, entity-evaluable)",NaN,NaN,0
7,ICD-10 Set F1 (micro),0.0000,NaN,101
8,FDI+ICD Entity Hallucination Rate (patient mean),0.7474,0.4335,65
9,FDI+ICD Entity Hallucination Rate (micro),0.7263,NaN,65



Entity coverage:
Patients containing GT FDI: 95 / 101
Patients containing predicted FDI: 65 / 101
Patients containing GT ICD: 0 / 101
Patients containing predicted ICD: 0 / 101

Saved:
/content/final_qlora_evaluation_metrics/final_metrics_per_case.csv
/content/final_qlora_evaluation_metrics/final_metrics_summary.csv
/content/final_qlora_evaluation_metrics/final_101_case_evaluation.xlsx


SECOND EXPERIMENT TO FIX ALL ERRORS

In [29]:
# ============================================================
# EXPERIMENT 2A
# BUILD CORRECTED 101-CASE EVALUATION DATASET
#
# DOES NOT MODIFY ORIGINAL FILE
# DOES NOT RETRAIN QLoRA
# DOES NOT RERUN RETRIEVAL
# ============================================================

import os
import json
import pandas as pd
import numpy as np


# ============================================================
# PATHS
# ============================================================

ORIGINAL_PRED_PATH = (
    "/content/exp2d_qlora_final/"
    "ground_truth_vs_qlora_predictions.csv"
)

RAG_PATH = "/content/rag_output_exp2d_final.json"

FULL_PATIENT_PATH = "/content/mmdental_cleaned_full.csv"

EXP2_DIR = "/content/experiment_2_corrected_evaluation"
os.makedirs(EXP2_DIR, exist_ok=True)

CORRECTED_XLSX = os.path.join(
    EXP2_DIR,
    "ground_truth_vs_qlora_predictions_with_exp2d_exemplars.xlsx"
)

CORRECTED_CSV = os.path.join(
    EXP2_DIR,
    "ground_truth_vs_qlora_predictions_with_exp2d_exemplars.csv"
)


# ============================================================
# HELPERS
# ============================================================

def norm_id(x):
    if pd.isna(x):
        return ""

    s = str(x).strip()

    if s.endswith(".0"):
        s = s[:-2]

    return s


def clean_value(x):
    if pd.isna(x):
        return ""

    x = str(x).strip()

    if x.lower() in {
        "nan",
        "none",
        "null",
    }:
        return ""

    return x


# ============================================================
# LOAD ORIGINAL PREDICTIONS
# ============================================================

pred_df = pd.read_csv(ORIGINAL_PRED_PATH)

pred_df["case_id"] = (
    pred_df["case_id"]
    .map(norm_id)
)

print("Original predictions:", pred_df.shape)


# ============================================================
# LOAD EXP2D RETRIEVAL
# ============================================================

with open(
    RAG_PATH,
    "r",
    encoding="utf-8"
) as f:
    rag = json.load(f)

rag = {
    norm_id(k): v
    for k, v in rag.items()
}

print("Exp2D queries:", len(rag))


# ============================================================
# LOAD FULL MMDENTAL LOOKUP
#
# Your uploaded file may have a .csv filename while actually
# containing Excel data, so detect it safely.
# ============================================================

try:
    full_df = pd.read_csv(FULL_PATIENT_PATH)
    print("Loaded full patient lookup as CSV.")

except Exception:
    full_df = pd.read_excel(FULL_PATIENT_PATH)
    print("Loaded full patient lookup as Excel.")

print("Full lookup:", full_df.shape)
print("Full lookup columns:")
print(full_df.columns.tolist())


# ============================================================
# FIND PATIENT-ID COLUMN
# ============================================================

possible_id_cols = [
    "Filename",
    "case_id",
    "Case ID",
    "patient_id",
    "Patient ID",
]

ID_COL = None

for col in possible_id_cols:
    if col in full_df.columns:
        ID_COL = col
        break

assert ID_COL is not None, (
    "Could not determine patient ID column."
)

print("Lookup ID column:", ID_COL)

full_df["_case_id"] = (
    full_df[ID_COL]
    .map(norm_id)
)


# ============================================================
# REQUIRED TARGET FIELDS
# ============================================================

TARGET_FIELDS = [
    "Oral Check",
    "Diagnosis",
    "Treatment plan",
    "Handle",
    "Doctor advices",
]

missing_fields = [
    x for x in TARGET_FIELDS
    if x not in full_df.columns
]

assert not missing_fields, (
    f"Clinical lookup missing fields: {missing_fields}"
)


# ============================================================
# CHECK 101 TEST CASES
# ============================================================

assert len(pred_df) == 101, (
    f"Expected 101 predictions, got {len(pred_df)}"
)

assert pred_df["case_id"].nunique() == 101, (
    "Prediction file contains duplicate case IDs."
)

assert len(rag) == 101, (
    f"Expected 101 retrieval queries, got {len(rag)}"
)

pred_ids = set(pred_df["case_id"])
rag_ids = set(rag.keys())

assert pred_ids == rag_ids, (
    "Prediction test IDs and Exp2D test IDs differ.\n"
    f"Only predictions: {pred_ids-rag_ids}\n"
    f"Only retrieval: {rag_ids-pred_ids}"
)

print(
    "[PASS] Prediction CSV and Exp2D retrieval "
    "contain exactly the same 101 test patients."
)


# ============================================================
# BUILD CLINICAL LOOKUP
# ============================================================

clinical_lookup = {}

for _, row in full_df.iterrows():

    pid = row["_case_id"]

    if not pid:
        continue

    clinical_lookup[pid] = {
        field: clean_value(row[field])
        for field in TARGET_FIELDS
    }


# ============================================================
# CHECK RETRIEVED PATIENTS
# ============================================================

all_retrieved = set()

for qid, item in rag.items():

    refs = item.get(
        "retrieved_exemplars",
        []
    )

    assert len(refs) == 5, (
        f"Case {qid}: expected 5 retrieved "
        f"exemplars, got {len(refs)}"
    )

    for ref in refs:

        rid = norm_id(
            ref["case_id"]
        )

        all_retrieved.add(rid)


missing_retrieved = (
    all_retrieved
    - set(clinical_lookup.keys())
)

assert not missing_retrieved, (
    "Some retrieved patients are missing "
    "from the clinical lookup:\n"
    f"{sorted(missing_retrieved)}"
)

print(
    f"[PASS] All {len(all_retrieved)} unique "
    "retrieved patients exist in clinical lookup."
)


# ============================================================
# BUILD COMPLETE EXP2D EXEMPLARS
# ============================================================

def build_full_exemplars(qid):

    output = []

    for rank, ref in enumerate(
        rag[qid]["retrieved_exemplars"],
        start=1,
    ):

        rid = norm_id(
            ref["case_id"]
        )

        output.append({
            "rank": rank,
            "case_id": rid,
            "reranker_score": float(
                ref["reranker_score"]
            ),
            "record": clinical_lookup[rid],
        })

    return output


pred_df[
    "Retrieved Exemplars (Reranked)"
] = pred_df["case_id"].apply(
    lambda qid: json.dumps(
        build_full_exemplars(qid),
        ensure_ascii=False,
    )
)


# ============================================================
# FINAL VALIDATION
# ============================================================

for _, row in pred_df.iterrows():

    qid = row["case_id"]

    exemplars = json.loads(
        row[
            "Retrieved Exemplars (Reranked)"
        ]
    )

    assert len(exemplars) == 5

    retrieved_ids = [
        norm_id(x["case_id"])
        for x in exemplars
    ]

    original_ids = [
        norm_id(x["case_id"])
        for x in rag[qid][
            "retrieved_exemplars"
        ]
    ]

    assert retrieved_ids == original_ids, (
        f"Top-5 mismatch for case {qid}"
    )

    # No query should retrieve itself
    assert qid not in retrieved_ids, (
        f"SELF RETRIEVAL detected: {qid}"
    )


print(
    "[PASS] Exactly five original Exp2D "
    "exemplars attached to every test patient."
)

print("[PASS] No self-retrieval detected.")


# ============================================================
# SAVE NEW FILES
# ============================================================

pred_df.to_excel(
    CORRECTED_XLSX,
    index=False,
)

pred_df.to_csv(
    CORRECTED_CSV,
    index=False,
)

print("\nSaved corrected Experiment-2 files:")
print(CORRECTED_XLSX)
print(CORRECTED_CSV)

Original predictions: (101, 14)
Exp2D queries: 101
Loaded full patient lookup as Excel.
Full lookup: (660, 19)
Full lookup columns:
['Filename', 'Main appeal', 'Subsequent', 'Present medical history', 'Past medical history', 'Oral Check', 'Diagnosis', 'Treatment plan', 'Handle', 'Doctor advices', 'Age', 'Age_group', 'Sex', 'Diagnosis_cleaned', 'Diagnosis_categories', 'num_labels', 'primary_diagnosis', 'semantic_text', 'semantic_text_cbct']
Lookup ID column: Filename
[PASS] Prediction CSV and Exp2D retrieval contain exactly the same 101 test patients.
[PASS] All 208 unique retrieved patients exist in clinical lookup.
[PASS] Exactly five original Exp2D exemplars attached to every test patient.
[PASS] No self-retrieval detected.

Saved corrected Experiment-2 files:
/content/experiment_2_corrected_evaluation/ground_truth_vs_qlora_predictions_with_exp2d_exemplars.xlsx
/content/experiment_2_corrected_evaluation/ground_truth_vs_qlora_predictions_with_exp2d_exemplars.csv


In [30]:
# ============================================================
# EXPERIMENT 2B
# VALIDATED CLINICAL ENTITY EXTRACTION
# ============================================================

import re
import pandas as pd
import numpy as np


# ============================================================
# NORMALIZATION
# ============================================================

def normalize(text):

    if pd.isna(text):
        return ""

    text = str(text)

    text = (
        text
        .replace("&#39;", "'")
        .replace("&amp;", "&")
        .replace("&quot;", '"')
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip().lower()


# ============================================================
# FDI
#
# Conservative primary extractor.
# ============================================================

TOOTH_WORD_RE = re.compile(
    r"\btooth\s*[*#:]?\s*"
    r"(1[1-8]|2[1-8]|3[1-8]|4[1-8])\b",
    re.IGNORECASE,
)

TOOTH_STAR_RE = re.compile(
    r"(?<!\d)\*\s*"
    r"(1[1-8]|2[1-8]|3[1-8]|4[1-8])\b",
    re.IGNORECASE,
)


def extract_fdi_teeth(text):

    text = normalize(text)

    return (
        set(TOOTH_WORD_RE.findall(text))
        |
        set(TOOTH_STAR_RE.findall(text))
    )


# ============================================================
# ICD-10
#
# FIXED VERSION
#
# Handles:
# K07.305
# K 07.305
# K-07.305
# K04.0
# ============================================================

ICD10_RE = re.compile(
    r"\b([A-Za-z])[\s-]?"
    r"(\d{2}(?:\.\d{1,4})?)\b",
    re.IGNORECASE,
)


def extract_icd10(text):

    text = normalize(text)

    return {
        f"{letter.upper()}{digits}"
        for letter, digits
        in ICD10_RE.findall(text)
    }


# ============================================================
# UNIT TESTS
# ============================================================

icd_tests = {
    "K07.305": {"K07.305"},
    "K 07.305": {"K07.305"},
    "K-07.305": {"K07.305"},
    "Diagnosis: K04.0": {"K04.0"},
    "Broken tooth (K02.400)": {"K02.400"},
}

for text, expected in icd_tests.items():

    result = extract_icd10(text)

    print(
        f"{text:35s}",
        "->",
        result,
    )

    assert result == expected, (
        f"ICD test failed for {text}"
    )


fdi_tests = {
    "tooth 16": {"16"},
    "Tooth: 38": {"38"},
    "*48": {"48"},
    "tooth #27": {"27"},
}

for text, expected in fdi_tests.items():

    result = extract_fdi_teeth(text)

    print(
        f"{text:35s}",
        "->",
        result,
    )

    assert result == expected, (
        f"FDI test failed for {text}"
    )


print(
    "\n[PASS] ICD-10 extractor validated."
)

print(
    "[PASS] Strict FDI extractor validated."
)

K07.305                             -> {'K07.305'}
K 07.305                            -> {'K07.305'}
K-07.305                            -> {'K07.305'}
Diagnosis: K04.0                    -> {'K04.0'}
Broken tooth (K02.400)              -> {'K02.400'}
tooth 16                            -> {'16'}
Tooth: 38                           -> {'38'}
*48                                 -> {'48'}
tooth #27                           -> {'27'}

[PASS] ICD-10 extractor validated.
[PASS] Strict FDI extractor validated.


In [31]:
# ============================================================
# EXPERIMENT 2C
# CORRECTED 101-CASE QLoRA EVALUATION
# ============================================================

!pip install -q sacrebleu rouge-score nltk pycocoevalcap openpyxl

import os
import re
import json
import ast
import numpy as np
import pandas as pd

import sacrebleu
from rouge_score import rouge_scorer

import nltk
from nltk.translate.meteor_score import meteor_score
from pycocoevalcap.cider.cider import Cider


# ============================================================
# PATHS
# ============================================================

INPUT_PATH = (
    "/content/experiment_2_corrected_evaluation/"
    "ground_truth_vs_qlora_predictions_with_exp2d_exemplars.xlsx"
)

OUTPUT_DIR = (
    "/content/experiment_2_corrected_evaluation/metrics"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True,
)


# ============================================================
# NLTK
# ============================================================

nltk.pathsec.ALLOW_PROXIED_FETCH = True

for pkg in [
    "wordnet",
    "omw-1.4",
]:
    nltk.download(
        pkg,
        quiet=True,
    )

from nltk.corpus import wordnet

_ = wordnet.synsets("tooth")

print(
    "[PASS] NLTK WordNet available"
)


# ============================================================
# LOAD
# ============================================================

df = pd.read_excel(
    INPUT_PATH
)

TARGET_FIELDS = [
    "Oral Check",
    "Diagnosis",
    "Treatment plan",
    "Handle",
    "Doctor advices",
]

assert len(df) == 101
assert df["case_id"].nunique() == 101

for field in TARGET_FIELDS:

    assert (
        df[f"GT_{field}"]
        .notna()
        .all()
    )

    assert (
        df[f"PRED_{field}"]
        .notna()
        .all()
    )


EXEMPLAR_COL = (
    "Retrieved Exemplars (Reranked)"
)

assert EXEMPLAR_COL in df.columns

print(
    "[PASS] 101 unique patients"
)

print(
    "[PASS] GT and predictions available"
)

print(
    "[PASS] Exp2D reranked exemplars available"
)


# ============================================================
# TEXT HELPERS
# ============================================================

def normalize(text):

    if pd.isna(text):
        return ""

    text = str(text)

    text = (
        text
        .replace("&#39;", "'")
        .replace("&amp;", "&")
        .replace("&quot;", '"')
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip().lower()


def tokenize(text):
    return normalize(text).split()


def get_ngrams(tokens, n=4):

    if len(tokens) < n:
        return set()

    return {
        tuple(tokens[i:i+n])
        for i in range(
            len(tokens)-n+1
        )
    }


def all_ngrams_list(tokens, n=4):

    if len(tokens) < n:
        return []

    return [
        tuple(tokens[i:i+n])
        for i in range(
            len(tokens)-n+1
        )
    ]


# ============================================================
# ENTITY EXTRACTION
# ============================================================

TOOTH_WORD_RE = re.compile(
    r"\btooth\s*[*#:]?\s*"
    r"(1[1-8]|2[1-8]|3[1-8]|4[1-8])\b",
    re.IGNORECASE,
)

TOOTH_STAR_RE = re.compile(
    r"(?<!\d)\*\s*"
    r"(1[1-8]|2[1-8]|3[1-8]|4[1-8])\b"
)

ICD10_RE = re.compile(
    r"\b([A-Za-z])[\s-]?"
    r"(\d{2}(?:\.\d{1,4})?)\b",
    re.IGNORECASE,
)


def extract_fdi_teeth(text):

    text = normalize(text)

    return (
        set(
            TOOTH_WORD_RE.findall(text)
        )
        |
        set(
            TOOTH_STAR_RE.findall(text)
        )
    )


def extract_icd10(text):

    text = normalize(text)

    return {
        f"{letter.upper()}{digits}"
        for letter, digits
        in ICD10_RE.findall(text)
    }


# ============================================================
# ENTITY METRICS
# ============================================================

def entity_counts(
    pred_set,
    gt_set,
):

    tp = len(
        pred_set & gt_set
    )

    fp = len(
        pred_set - gt_set
    )

    fn = len(
        gt_set - pred_set
    )

    return tp, fp, fn


def set_f1(
    pred_set,
    gt_set,
):

    # Do NOT reward empty-empty.
    if not pred_set and not gt_set:
        return np.nan

    if not pred_set or not gt_set:
        return 0.0

    tp = len(
        pred_set & gt_set
    )

    precision = (
        tp / len(pred_set)
    )

    recall = (
        tp / len(gt_set)
    )

    if precision + recall == 0:
        return 0.0

    return (
        2
        * precision
        * recall
        / (precision + recall)
    )


# ============================================================
# EXEMPLARS
# ============================================================

def parse_exemplars(cell):

    if pd.isna(cell):
        return []

    if isinstance(cell, list):
        return cell

    text = str(cell).strip()

    try:
        value = json.loads(text)

        if isinstance(value, list):
            return value

    except Exception:
        pass

    try:
        value = ast.literal_eval(text)

        if isinstance(value, list):
            return value

    except Exception:
        pass

    return []


def exemplar_full_text(ex):

    if not isinstance(ex, dict):
        return ""

    record = ex.get(
        "record",
        {}
    )

    if not isinstance(
        record,
        dict,
    ):
        return ""

    parts = []

    for field in TARGET_FIELDS:

        value = record.get(
            field,
            ""
        )

        value = str(
            value
        ).strip()

        if value and value.lower() not in {
            "nan",
            "none",
            "null",
        }:
            parts.append(value)

    return " ".join(parts)


# ============================================================
# DUPLICATE / ABSTENTION AUDIT
# ============================================================

def full_pred(row):

    return " ".join(
        str(row[f"PRED_{field}"])
        for field in TARGET_FIELDS
    )


normalized_preds = (
    df.apply(
        full_pred,
        axis=1,
    )
    .map(normalize)
)

duplicate_mask = (
    normalized_preds
    .duplicated(
        keep=False
    )
)

if duplicate_mask.any():

    print(
        "\n[WARN] Duplicate generated reports:"
    )

    temp = pd.DataFrame({
        "case_id":
            df.loc[
                duplicate_mask,
                "case_id"
            ].astype(str),

        "prediction":
            normalized_preds[
                duplicate_mask
            ],
    })

    for _, group in temp.groupby(
        "prediction"
    ):

        if len(group) > 1:

            print(
                group[
                    "case_id"
                ].tolist()
            )


# All-field abstention

def is_abstention(value):

    value = normalize(value)

    return value in {
        "not determined",
        "not recorded",
        "not available",
    }


all_abstain = []

for _, row in df.iterrows():

    if all(
        is_abstention(
            row[f"PRED_{field}"]
        )
        for field in TARGET_FIELDS
    ):

        all_abstain.append(
            str(row["case_id"])
        )


print(
    "\nAll-field abstentions:",
    len(all_abstain),
)

print(
    all_abstain
)


# ============================================================
# METRIC OBJECTS
# ============================================================

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True,
)

cider = Cider()

rows = []

hyps = []
refs = []

cider_gts = {}
cider_res = {}

fdi_tp = fdi_fp = fdi_fn = 0
icd_tp = icd_fp = icd_fn = 0

halluc_num = 0
halluc_den = 0


# ============================================================
# EVALUATE EACH PATIENT
# ============================================================

for _, row in df.iterrows():

    case_id = str(
        row["case_id"]
    )

    gt_raw = " ".join(
        str(row[f"GT_{field}"])
        for field in TARGET_FIELDS
    )

    pred_raw = " ".join(
        str(row[f"PRED_{field}"])
        for field in TARGET_FIELDS
    )

    gt_norm = normalize(gt_raw)
    pred_norm = normalize(pred_raw)

    gt_tokens = tokenize(gt_raw)
    pred_tokens = tokenize(pred_raw)

    hyps.append(pred_norm)
    refs.append(gt_norm)


    # ------------------------
    # ROUGE-L
    # ------------------------

    rouge_l = rouge.score(
        gt_norm,
        pred_norm,
    )["rougeL"].fmeasure


    # ------------------------
    # METEOR
    # ------------------------

    meteor = meteor_score(
        [gt_tokens],
        pred_tokens,
    )


    # ------------------------
    # CIDEr storage
    # ------------------------

    cider_gts[case_id] = [
        gt_norm
    ]

    cider_res[case_id] = [
        pred_norm
    ]


    # ------------------------
    # FDI
    # ------------------------

    gt_teeth = (
        extract_fdi_teeth(
            gt_raw
        )
    )

    pred_teeth = (
        extract_fdi_teeth(
            pred_raw
        )
    )

    fdi_f1 = set_f1(
        pred_teeth,
        gt_teeth,
    )

    tp, fp, fn = entity_counts(
        pred_teeth,
        gt_teeth,
    )

    fdi_tp += tp
    fdi_fp += fp
    fdi_fn += fn


    # ------------------------
    # ICD
    # ------------------------

    gt_icd = extract_icd10(
        gt_raw
    )

    pred_icd = extract_icd10(
        pred_raw
    )

    icd_f1 = set_f1(
        pred_icd,
        gt_icd,
    )

    tp, fp, fn = entity_counts(
        pred_icd,
        gt_icd,
    )

    icd_tp += tp
    icd_fp += fp
    icd_fn += fn


    # ------------------------
    # Entity hallucination
    # ------------------------

    gt_entities = (
        gt_teeth | gt_icd
    )

    pred_entities = (
        pred_teeth | pred_icd
    )

    unsupported = (
        pred_entities
        - gt_entities
    )

    if pred_entities:

        entity_halluc_rate = (
            len(unsupported)
            /
            len(pred_entities)
        )

        halluc_num += len(
            unsupported
        )

        halluc_den += len(
            pred_entities
        )

    else:
        entity_halluc_rate = np.nan


    # ------------------------
    # Exp2D exemplars
    # ------------------------

    exemplars = parse_exemplars(
        row[EXEMPLAR_COL]
    )

    assert len(exemplars) == 5, (
        f"Case {case_id}: could not "
        "parse exactly 5 exemplars."
    )

    exemplar_texts = [
        exemplar_full_text(ex)
        for ex in exemplars
    ]

    exemplar_teeth = set()
    exemplar_ngrams = set()

    for text in exemplar_texts:

        exemplar_teeth |= (
            extract_fdi_teeth(
                text
            )
        )

        exemplar_ngrams |= (
            get_ngrams(
                tokenize(text),
                4,
            )
        )


    # ------------------------
    # Retrieval Copy Rate
    # ------------------------

    pred_ngrams = (
        all_ngrams_list(
            pred_tokens,
            4,
        )
    )

    gt_ngrams = get_ngrams(
        gt_tokens,
        4,
    )

    if pred_ngrams:

        copied = [
            ng
            for ng in pred_ngrams
            if (
                ng in exemplar_ngrams
                and
                ng not in gt_ngrams
            )
        ]

        copy_rate = (
            len(copied)
            /
            len(pred_ngrams)
        )

    else:
        copy_rate = np.nan


    # ------------------------
    # Fabricated tooth rate
    # ------------------------

    if pred_teeth:

        supported_teeth = (
            gt_teeth
            |
            exemplar_teeth
        )

        fabricated = (
            pred_teeth
            -
            supported_teeth
        )

        fabricated_rate = (
            len(fabricated)
            /
            len(pred_teeth)
        )

    else:
        fabricated_rate = np.nan


    rows.append({

        "Case ID":
            case_id,

        "ROUGE-L":
            rouge_l,

        "METEOR":
            meteor,

        "FDI_Tooth_F1":
            fdi_f1,

        "ICD10_F1":
            icd_f1,

        "Entity_Hallucination_Rate":
            entity_halluc_rate,

        "Retrieval_Copy_Rate":
            copy_rate,

        "Fabricated_Tooth_Rate":
            fabricated_rate,

        "N_Retrieved_Exemplars":
            len(exemplars),

        "GT_Teeth":
            sorted(gt_teeth),

        "Pred_Teeth":
            sorted(pred_teeth),

        "GT_ICD10":
            sorted(gt_icd),

        "Pred_ICD10":
            sorted(pred_icd),
    })


per_case = pd.DataFrame(
    rows
)


# ============================================================
# BLEU-4
# ============================================================

bleu4 = (
    sacrebleu.corpus_bleu(
        hyps,
        [refs],
    ).score
    / 100.0
)

sentence_bleu = []

for hyp, ref in zip(
    hyps,
    refs,
):

    score = (
        sacrebleu
        .sentence_bleu(
            hyp,
            [ref],
        )
        .score
        / 100.0
    )

    sentence_bleu.append(
        score
    )

per_case.insert(
    1,
    "BLEU-4",
    sentence_bleu,
)


# ============================================================
# CIDEr
# ============================================================

cider_score, cider_case = (
    cider.compute_score(
        cider_gts,
        cider_res,
    )
)

case_order = list(
    cider_res.keys()
)

cider_map = dict(
    zip(
        case_order,
        cider_case,
    )
)

per_case["CIDEr"] = (
    per_case["Case ID"]
    .astype(str)
    .map(cider_map)
)


# ============================================================
# MICRO F1
# ============================================================

def micro_f1(
    tp,
    fp,
    fn,
):

    precision = (
        tp / (tp + fp)
        if tp + fp
        else np.nan
    )

    recall = (
        tp / (tp + fn)
        if tp + fn
        else np.nan
    )

    if (
        pd.isna(precision)
        or
        pd.isna(recall)
        or
        precision + recall == 0
    ):
        return 0.0

    return (
        2
        * precision
        * recall
        /
        (precision + recall)
    )


fdi_micro = micro_f1(
    fdi_tp,
    fdi_fp,
    fdi_fn,
)

icd_micro = micro_f1(
    icd_tp,
    icd_fp,
    icd_fn,
)


# ============================================================
# SUMMARY
# ============================================================

def mean_valid(col):

    values = (
        per_case[col]
        .dropna()
    )

    return (
        float(values.mean())
        if len(values)
        else np.nan
    )


def std_valid(col):

    values = (
        per_case[col]
        .dropna()
    )

    return (
        float(values.std())
        if len(values) > 1
        else np.nan
    )


summary = pd.DataFrame([

    {
        "Metric": "BLEU-4",
        "Mean": bleu4,
        "Std": np.nan,
        "N_Evaluable": 101,
    },

    {
        "Metric": "ROUGE-L",
        "Mean":
            mean_valid("ROUGE-L"),
        "Std":
            std_valid("ROUGE-L"),
        "N_Evaluable": 101,
    },

    {
        "Metric": "METEOR",
        "Mean":
            mean_valid("METEOR"),
        "Std":
            std_valid("METEOR"),
        "N_Evaluable": 101,
    },

    {
        "Metric": "CIDEr",
        "Mean":
            float(cider_score),
        "Std": np.nan,
        "N_Evaluable": 101,
    },

    {
        "Metric":
            "FDI Tooth-Set F1 "
            "(macro, entity-evaluable)",
        "Mean":
            mean_valid(
                "FDI_Tooth_F1"
            ),
        "Std":
            std_valid(
                "FDI_Tooth_F1"
            ),
        "N_Evaluable":
            int(
                per_case[
                    "FDI_Tooth_F1"
                ].notna().sum()
            ),
    },

    {
        "Metric":
            "FDI Tooth-Set F1 (micro)",
        "Mean":
            fdi_micro,
        "Std": np.nan,
        "N_Evaluable": 101,
    },

    {
        "Metric":
            "ICD-10 Set F1 "
            "(macro, entity-evaluable)",
        "Mean":
            mean_valid(
                "ICD10_F1"
            ),
        "Std":
            std_valid(
                "ICD10_F1"
            ),
        "N_Evaluable":
            int(
                per_case[
                    "ICD10_F1"
                ].notna().sum()
            ),
    },

    {
        "Metric":
            "ICD-10 Set F1 (micro)",
        "Mean":
            icd_micro,
        "Std": np.nan,
        "N_Evaluable": 101,
    },

    {
        "Metric":
            "FDI+ICD Entity Hallucination "
            "Rate (patient mean)",
        "Mean":
            mean_valid(
                "Entity_Hallucination_Rate"
            ),
        "Std":
            std_valid(
                "Entity_Hallucination_Rate"
            ),
        "N_Evaluable":
            int(
                per_case[
                    "Entity_Hallucination_Rate"
                ].notna().sum()
            ),
    },

    {
        "Metric":
            "FDI+ICD Entity Hallucination "
            "Rate (micro)",
        "Mean":
            (
                halluc_num
                / halluc_den
                if halluc_den
                else np.nan
            ),
        "Std": np.nan,
        "N_Evaluable":
            int(
                per_case[
                    "Entity_Hallucination_Rate"
                ].notna().sum()
            ),
    },

    {
        "Metric":
            "Retrieval Copy Rate",
        "Mean":
            mean_valid(
                "Retrieval_Copy_Rate"
            ),
        "Std":
            std_valid(
                "Retrieval_Copy_Rate"
            ),
        "N_Evaluable":
            int(
                per_case[
                    "Retrieval_Copy_Rate"
                ].notna().sum()
            ),
    },

    {
        "Metric":
            "Fabricated Tooth-Reference Rate",
        "Mean":
            mean_valid(
                "Fabricated_Tooth_Rate"
            ),
        "Std":
            std_valid(
                "Fabricated_Tooth_Rate"
            ),
        "N_Evaluable":
            int(
                per_case[
                    "Fabricated_Tooth_Rate"
                ].notna().sum()
            ),
    },
])


# ============================================================
# ENTITY COVERAGE
# ============================================================

print("\nENTITY COVERAGE")
print("=" * 60)

print(
    "GT FDI:",
    sum(
        len(x) > 0
        for x in per_case[
            "GT_Teeth"
        ]
    ),
    "/ 101",
)

print(
    "Predicted FDI:",
    sum(
        len(x) > 0
        for x in per_case[
            "Pred_Teeth"
        ]
    ),
    "/ 101",
)

print(
    "GT ICD:",
    sum(
        len(x) > 0
        for x in per_case[
            "GT_ICD10"
        ]
    ),
    "/ 101",
)

print(
    "Predicted ICD:",
    sum(
        len(x) > 0
        for x in per_case[
            "Pred_ICD10"
        ]
    ),
    "/ 101",
)


# ============================================================
# SAVE
# ============================================================

PER_CASE_PATH = os.path.join(
    OUTPUT_DIR,
    "experiment2_metrics_per_case.csv"
)

SUMMARY_PATH = os.path.join(
    OUTPUT_DIR,
    "experiment2_metrics_summary.csv"
)

FINAL_XLSX = os.path.join(
    OUTPUT_DIR,
    "experiment2_complete_evaluation.xlsx"
)


per_case.to_csv(
    PER_CASE_PATH,
    index=False,
)

summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

with pd.ExcelWriter(
    FINAL_XLSX,
    engine="openpyxl",
) as writer:

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False,
    )

    per_case.to_excel(
        writer,
        sheet_name="Per Case",
        index=False,
    )

    df.to_excel(
        writer,
        sheet_name="GT vs Prediction",
        index=False,
    )


print("\n" + "=" * 70)
print("EXPERIMENT 2 — CORRECTED EVALUATION")
print("=" * 70)

display(summary)

print("\nSaved:")
print(PER_CASE_PATH)
print(SUMMARY_PATH)
print(FINAL_XLSX)

[PASS] NLTK WordNet available
[PASS] 101 unique patients
[PASS] GT and predictions available
[PASS] Exp2D reranked exemplars available

[WARN] Duplicate generated reports:
['88', '414', '596', '576', '166', '46', '445', '638', '114', '586', '448', '260', '613', '225', '23', '38']

All-field abstentions: 16
['88', '414', '596', '576', '166', '46', '445', '638', '114', '586', '448', '260', '613', '225', '23', '38']

ENTITY COVERAGE
GT FDI: 95 / 101
Predicted FDI: 65 / 101
GT ICD: 23 / 101
Predicted ICD: 5 / 101

EXPERIMENT 2 — CORRECTED EVALUATION


,Metric,Mean,Std,N_Evaluable
0,BLEU-4,0.0005,NaN,101
1,ROUGE-L,0.1088,0.0874,101
2,METEOR,0.0688,0.0847,101
3,CIDEr,0.0051,NaN,101
4,"FDI Tooth-Set F1 (macro, entity-evaluable)",0.0878,0.2181,98
5,FDI Tooth-Set F1 (micro),0.1253,NaN,101
6,"ICD-10 Set F1 (macro, entity-evaluable)",0.0000,0.0000,28
7,ICD-10 Set F1 (micro),0.0000,NaN,101
8,FDI+ICD Entity Hallucination Rate (patient mean),0.7513,0.4313,66
9,FDI+ICD Entity Hallucination Rate (micro),0.7400,NaN,66



Saved:
/content/experiment_2_corrected_evaluation/metrics/experiment2_metrics_per_case.csv
/content/experiment_2_corrected_evaluation/metrics/experiment2_metrics_summary.csv
/content/experiment_2_corrected_evaluation/metrics/experiment2_complete_evaluation.xlsx


Experiment 3

In [33]:
# ============================================================
# EXPERIMENT 3A
# LOAD SAVED QLoRA ADAPTER FOR DIAGNOSTIC INFERENCE
#
# NO TRAINING
# ============================================================

!pip install -q -U transformers peft accelerate bitsandbytes

import os
import json
import re
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel


# ============================================================
# PATHS
# ============================================================

BASE_MODEL = "unsloth/medgemma-1.5-4b-it"

# CHANGE THIS ONLY IF NECESSARY
ADAPTER_PATH = (
    "/content/drive/MyDrive/"
    "exp2d_qlora_final_20260903_173822/"
    "medgemma_exp2d_qlora"
)

RAG_PATH = "/content/rag_output_exp2d_final.json"

FULL_PATIENT_PATH = "/content/mmdental_cleaned_full.csv"

DIAGNOSTIC_DIR = (
    "/content/experiment_3_adapter_diagnostic"
)

os.makedirs(
    DIAGNOSTIC_DIR,
    exist_ok=True,
)


# ============================================================
# VERIFY ADAPTER
# ============================================================

assert os.path.exists(
    ADAPTER_PATH
), f"Adapter not found: {ADAPTER_PATH}"

print("Adapter directory:")
print(ADAPTER_PATH)

print("\nAdapter files:")
print(os.listdir(ADAPTER_PATH))


# ============================================================
# 4-BIT CONFIG
# SAME TYPE USED FOR QLoRA
# ============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


# ============================================================
# TOKENIZER
# Prefer saved tokenizer from adapter directory.
# ============================================================

try:

    tokenizer = AutoTokenizer.from_pretrained(
        ADAPTER_PATH,
        trust_remote_code=True,
    )

    print(
        "[PASS] Tokenizer loaded "
        "from saved adapter."
    )

except Exception:

    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL,
        trust_remote_code=True,
    )

    print(
        "[WARN] Tokenizer loaded "
        "from base model."
    )


# ============================================================
# BASE MODEL
# ============================================================

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)


# ============================================================
# ATTACH SAVED QLoRA
# ============================================================

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
)

model.eval()

print(
    "\n[PASS] Saved QLoRA adapter loaded."
)

print(
    "[PASS] Model is in evaluation mode."
)


Adapter directory:
/content/drive/MyDrive/exp2d_qlora_final_20260903_173822/medgemma_exp2d_qlora

Adapter files:
['adapter_model.safetensors', 'checkpoint-162', 'tokenizer.json', 'README.md', 'training_args.bin', 'tokenizer_config.json', 'adapter_config.json', 'chat_template.jinja']
[PASS] Tokenizer loaded from saved adapter.


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]


[PASS] Saved QLoRA adapter loaded.
[PASS] Model is in evaluation mode.


In [34]:
# ============================================================
# EXPERIMENT 3B
# LOAD ORIGINAL EXP2D TOP-5 + CLINICAL LOOKUP
# ============================================================

def norm_id(x):

    if pd.isna(x):
        return ""

    s = str(x).strip()

    if s.endswith(".0"):
        s = s[:-2]

    return s


def clean_value(x):

    if pd.isna(x):
        return ""

    value = str(x).strip()

    if value.lower() in {
        "nan",
        "none",
        "null",
    }:
        return ""

    return value


# ============================================================
# EXP2D JSON
# ============================================================

with open(
    RAG_PATH,
    "r",
    encoding="utf-8",
) as f:

    rag = json.load(f)


rag = {
    norm_id(k): v
    for k, v in rag.items()
}

assert len(rag) == 101

print(
    "[PASS] Loaded 101 Exp2D queries."
)


# ============================================================
# FULL CLINICAL LOOKUP
# ============================================================

try:

    full_df = pd.read_csv(
        FULL_PATIENT_PATH
    )

except Exception:

    full_df = pd.read_excel(
        FULL_PATIENT_PATH
    )


possible_id_cols = [
    "Filename",
    "case_id",
    "Case ID",
    "patient_id",
    "Patient ID",
]

ID_COL = next(
    (
        c
        for c in possible_id_cols
        if c in full_df.columns
    ),
    None,
)

assert ID_COL is not None

full_df["_case_id"] = (
    full_df[ID_COL]
    .map(norm_id)
)


TARGET_FIELDS = [
    "Oral Check",
    "Diagnosis",
    "Treatment plan",
    "Handle",
    "Doctor advices",
]


clinical_lookup = {}

for _, row in full_df.iterrows():

    pid = row["_case_id"]

    if not pid:
        continue

    clinical_lookup[pid] = {
        field:
            clean_value(
                row.get(field, "")
            )
        for field in TARGET_FIELDS
    }


print(
    "[PASS] Clinical lookup built:",
    len(clinical_lookup),
    "patients"
)


# ============================================================
# VERIFY ALL RETRIEVED IDS
# ============================================================

for qid, item in rag.items():

    refs = item[
        "retrieved_exemplars"
    ]

    assert len(refs) == 5

    for ref in refs:

        rid = norm_id(
            ref["case_id"]
        )

        assert rid in clinical_lookup, (
            f"Missing retrieved patient {rid}"
        )


print(
    "[PASS] All Exp2D retrieved records available."
)

[PASS] Loaded 101 Exp2D queries.
[PASS] Clinical lookup built: 660 patients
[PASS] All Exp2D retrieved records available.


In [36]:
# ============================================================
# EXPERIMENT 3C
# LOAD EXACT TRAINING SYSTEM PROMPT
# ============================================================

TRAIN_JSONL = (
    "/content/train (1).jsonl"
)

SYSTEM_PROMPT = None


if os.path.exists(TRAIN_JSONL):

    with open(
        TRAIN_JSONL,
        "r",
        encoding="utf-8",
    ) as f:

        first = json.loads(
            f.readline()
        )

    for message in first["messages"]:

        if message["role"] == "system":

            SYSTEM_PROMPT = (
                message["content"]
            )

            break


if SYSTEM_PROMPT is None:

    raise RuntimeError(
        "Exact SFT system prompt was not found. "
        "Point TRAIN_JSONL to the corrected training "
        "JSONL rather than creating a new prompt."
    )


print(
    "[PASS] Exact training system prompt loaded."
)

print("\nSystem prompt preview:")
print(SYSTEM_PROMPT[:1000])

[PASS] Exact training system prompt loaded.

System prompt preview:
You are a dental clinical documentation assistant. You will be given a patient's Age, Sex, and Main appeal (chief complaint), along with several similar PAST patient cases retrieved for reference.

CRITICAL: The retrieved cases are REFERENCE MATERIAL from OTHER patients. They are NOT facts about the current patient. Do not assume the current patient has the same findings, tooth numbers, diagnoses, or treatments as any retrieved case unless the current patient's own Age/Sex/Main appeal genuinely supports it.

Rules:
1. NEVER invent or copy a tooth number, ICD code, diagnosis, finding, medication, procedure, or treatment that is not directly supported by the current patient's own Main appeal or by a clear, justified pattern across the retrieved cases.
2. If there is insufficient evidence to determine a field, output exactly "Not determined" for that field. Do not guess, and do not fill it in from the single most similar 

In [37]:
# ============================================================
# EXPERIMENT 3D
# BUILD QUERY + RETRIEVED RECORD PROMPT
# ============================================================

def build_reference_text(
    rank,
    rid,
    score,
    record,
):

    lines = [
        f"Retrieved reference {rank}:",
        f"Patient ID: {rid}",
        f"Retrieval score: {score:.6f}",
    ]

    for field in TARGET_FIELDS:

        value = clean_value(
            record.get(
                field,
                ""
            )
        )

        if not value:
            value = "Not available"

        lines.append(
            f"{field}: {value}"
        )

    return "\n".join(lines)


def build_user_prompt(qid):

    item = rag[qid]

    # The original Exp2D file already contains
    # the query-safe patient text:
    #
    # Age + Sex + Main Appeal
    #
    query_text = clean_value(
        item.get(
            "query_text",
            ""
        )
    )

    refs = []

    for rank, ref in enumerate(
        item[
            "retrieved_exemplars"
        ],
        start=1,
    ):

        rid = norm_id(
            ref["case_id"]
        )

        score = float(
            ref["reranker_score"]
        )

        refs.append(
            build_reference_text(
                rank,
                rid,
                score,
                clinical_lookup[rid],
            )
        )


    user_prompt = (
        "CURRENT PATIENT:\n"
        f"{query_text}\n\n"
        "RETRIEVED HISTORICAL REFERENCES:\n\n"
        +
        "\n\n".join(refs)
    )

    return user_prompt

In [38]:
# ============================================================
# EXPERIMENT 3E
# DIAGNOSTIC JSON PARSER
#
# NEVER converts parser failure into "Not determined"
# ============================================================

EXPECTED_FIELDS = [
    "Oral Check",
    "Diagnosis",
    "Treatment plan",
    "Handle",
    "Doctor advices",
]


def extract_json_object(text):

    if not text:
        return None, "EMPTY_GENERATION"

    cleaned = str(text).strip()


    # Gemma/MedGemma final marker
    if "<unused95>" in cleaned:

        cleaned = (
            cleaned
            .split(
                "<unused95>",
                1,
            )[-1]
            .strip()
        )


    # Remove markdown fences only
    cleaned = (
        cleaned
        .replace("```json", "")
        .replace("```JSON", "")
        .replace("```", "")
        .strip()
    )


    # --------------------------------
    # Attempt 1: direct JSON
    # --------------------------------

    try:

        obj = json.loads(
            cleaned
        )

        if isinstance(
            obj,
            dict,
        ):
            return (
                obj,
                "SUCCESS_DIRECT_JSON",
            )

    except Exception:
        pass


    # --------------------------------
    # Attempt 2:
    # Search balanced JSON objects.
    # --------------------------------

    candidates = []

    depth = 0
    start = None
    in_string = False
    escape = False

    for i, char in enumerate(
        cleaned
    ):

        if escape:
            escape = False
            continue

        if char == "\\":
            escape = True
            continue

        if char == '"':
            in_string = (
                not in_string
            )
            continue

        if in_string:
            continue

        if char == "{":

            if depth == 0:
                start = i

            depth += 1

        elif char == "}":

            if depth > 0:
                depth -= 1

                if (
                    depth == 0
                    and start is not None
                ):

                    candidates.append(
                        cleaned[
                            start:i+1
                        ]
                    )

                    start = None


    # Prefer last valid object.
    for candidate in reversed(
        candidates
    ):

        try:

            obj = json.loads(
                candidate
            )

            if isinstance(
                obj,
                dict,
            ):

                return (
                    obj,
                    "SUCCESS_EXTRACTED_JSON",
                )

        except Exception:
            continue


    return None, "PARSE_FAILURE"


def validate_prediction(obj):

    if obj is None:
        return False, "NO_JSON"

    missing = [
        field
        for field in EXPECTED_FIELDS
        if field not in obj
    ]

    if missing:

        return (
            False,
            "MISSING_FIELDS:"
            +
            ",".join(missing),
        )


    return True, "VALID_SCHEMA"

In [39]:
# ============================================================
# EXPERIMENT 3F
# SINGLE-PATIENT DIAGNOSTIC
# ============================================================

@torch.inference_mode()
def diagnostic_generate(
    qid,
    max_new_tokens=500,
):

    qid = norm_id(qid)

    assert qid in rag

    user_prompt = (
        build_user_prompt(qid)
    )


    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]


    prompt = (
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    )


    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    )


    device = next(
        model.parameters()
    ).device

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }


    input_length = (
        inputs[
            "input_ids"
        ].shape[1]
    )


    outputs = model.generate(
        **inputs,

        max_new_tokens=
            max_new_tokens,

        do_sample=False,

        repetition_penalty=1.15,

        eos_token_id=
            tokenizer.eos_token_id,

        pad_token_id=
            tokenizer.eos_token_id,
    )


    # Decode ONLY newly generated tokens.
    generated_tokens = (
        outputs[0][
            input_length:
        ]
    )


    raw = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=False,
    )


    parsed, parse_status = (
        extract_json_object(
            raw
        )
    )

    valid, schema_status = (
        validate_prediction(
            parsed
        )
    )


    refs = [
        norm_id(
            x["case_id"]
        )
        for x in rag[qid][
            "retrieved_exemplars"
        ]
    ]


    result = {
        "case_id":
            qid,

        "query_text":
            rag[qid].get(
                "query_text",
                ""
            ),

        "retrieved_ids":
            refs,

        "raw_generation":
            raw,

        "parsed_prediction":
            parsed,

        "parse_status":
            parse_status,

        "schema_status":
            schema_status,

        "valid":
            valid,
    }


    return result

In [40]:
result_88 = diagnostic_generate(
    "88"
)

print("=" * 80)
print("CASE:", result_88["case_id"])
print("=" * 80)

print("\nQUERY:")
print(
    result_88[
        "query_text"
    ]
)

print("\nTOP-5:")
print(
    result_88[
        "retrieved_ids"
    ]
)

print("\nRAW MODEL OUTPUT:")
print(
    result_88[
        "raw_generation"
    ]
)

print("\nPARSE STATUS:")
print(
    result_88[
        "parse_status"
    ]
)

print("\nSCHEMA STATUS:")
print(
    result_88[
        "schema_status"
    ]
)

print("\nPARSED:")
print(
    json.dumps(
        result_88[
            "parsed_prediction"
        ],
        indent=2,
        ensure_ascii=False,
    )
    if result_88[
        "parsed_prediction"
    ] is not None
    else "NONE"
)

CASE: 88

QUERY:
Age: 19. Sex: female. Main appeal: Crowded teeth, consultation for correction

TOP-5:
['150', '10', '57', '386', '60']

RAW MODEL OUTPUT:
{"Oral Check": "the whole mouth's gums are red and swollen, and the gum margins are bleeding. several teeth are slightly loose and deep periodontal pockets are seen. cbct showed alveolar bone horizontal resorption.", "Diagnosis": "whole-mouth chronic periodontitis", "Treatment plan": "full mouth systemic periodontal treatment.", "Handle": "** Whole mouth supragingival cleaning, polishing, and applying fluoride.", "Doctor advices": "Follow-up for discomfort."}<end_of_turn>

PARSE STATUS:
SUCCESS_EXTRACTED_JSON

SCHEMA STATUS:
VALID_SCHEMA

PARSED:
{
  "Oral Check": "the whole mouth's gums are red and swollen, and the gum margins are bleeding. several teeth are slightly loose and deep periodontal pockets are seen. cbct showed alveolar bone horizontal resorption.",
  "Diagnosis": "whole-mouth chronic periodontitis",
  "Treatment plan": 

In [41]:
# ============================================================
# EXPERIMENT 3G
# 12-PATIENT DIAGNOSTIC
# ============================================================

DIAGNOSTIC_CASES = [
    # Previously failed / abstained
    "88",
    "414",
    "596",
    "166",
    "638",
    "23",

    # Previously meaningful
    "378",
    "98",
    "278",
    "270",

    # Controls
    "219",
    "314",
]


diagnostic_results = []


for i, qid in enumerate(
    DIAGNOSTIC_CASES,
    start=1,
):

    print(
        "\n\n"
        + "=" * 90
    )

    print(
        f"DIAGNOSTIC "
        f"{i}/{len(DIAGNOSTIC_CASES)} "
        f"— CASE {qid}"
    )

    print(
        "=" * 90
    )


    try:

        result = (
            diagnostic_generate(
                qid
            )
        )

    except Exception as e:

        result = {
            "case_id": qid,
            "query_text":
                rag[qid].get(
                    "query_text",
                    ""
                ),
            "retrieved_ids": [
                norm_id(
                    x["case_id"]
                )
                for x in rag[qid][
                    "retrieved_exemplars"
                ]
            ],
            "raw_generation": None,
            "parsed_prediction": None,
            "parse_status":
                "GENERATION_EXCEPTION",
            "schema_status":
                str(e),
            "valid": False,
        }


    diagnostic_results.append(
        result
    )


    print("\nQUERY:")
    print(
        result[
            "query_text"
        ]
    )

    print("\nTOP-5:")
    print(
        result[
            "retrieved_ids"
        ]
    )

    print("\nRAW:")
    print(
        result[
            "raw_generation"
        ]
    )

    print("\nSTATUS:")
    print(
        result[
            "parse_status"
        ],
        "|",
        result[
            "schema_status"
        ]
    )



DIAGNOSTIC 1/12 — CASE 88

QUERY:
Age: 19. Sex: female. Main appeal: Crowded teeth, consultation for correction

TOP-5:
['150', '10', '57', '386', '60']

RAW:
{"Oral Check": "the whole mouth's gums are red and swollen, and the gum margins are bleeding. several teeth are slightly loose and deep periodontal pockets are seen. cbct showed alveolar bone horizontal resorption.", "Diagnosis": "whole-mouth chronic periodontitis", "Treatment plan": "full mouth systemic periodontal treatment.", "Handle": "** Whole mouth supragingival cleaning, polishing, and applying fluoride.", "Doctor advices": "Follow-up for discomfort."}<end_of_turn>

STATUS:
SUCCESS_EXTRACTED_JSON | VALID_SCHEMA


DIAGNOSTIC 2/12 — CASE 414

QUERY:
Age: 48. Sex: female. Main appeal: The lower front teeth are missing and require examination.

TOP-5:
['28', '212', '26', '386', '352']

RAW:
{"Oral Check": "tooth 36 (lower left first molar). residual roots, loosening degree i", "Diagnosis": "missing teeth", "Treatment plan": 

In [42]:
# ============================================================
# EXPERIMENT 3H
# SAVE DIAGNOSTIC ONLY
# ============================================================

DIAGNOSTIC_JSON = os.path.join(
    DIAGNOSTIC_DIR,
    "saved_adapter_12_case_diagnostic.json",
)

DIAGNOSTIC_CSV = os.path.join(
    DIAGNOSTIC_DIR,
    "saved_adapter_12_case_diagnostic.csv",
)


with open(
    DIAGNOSTIC_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        diagnostic_results,
        f,
        indent=2,
        ensure_ascii=False,
    )


summary_rows = []

for result in diagnostic_results:

    pred = (
        result[
            "parsed_prediction"
        ]
        or {}
    )

    summary_rows.append({

        "case_id":
            result["case_id"],

        "query_text":
            result["query_text"],

        "retrieved_ids":
            json.dumps(
                result[
                    "retrieved_ids"
                ]
            ),

        "parse_status":
            result[
                "parse_status"
            ],

        "schema_status":
            result[
                "schema_status"
            ],

        "valid":
            result[
                "valid"
            ],

        "Oral Check":
            pred.get(
                "Oral Check",
                ""
            ),

        "Diagnosis":
            pred.get(
                "Diagnosis",
                ""
            ),

        "Treatment plan":
            pred.get(
                "Treatment plan",
                ""
            ),

        "Handle":
            pred.get(
                "Handle",
                ""
            ),

        "Doctor advices":
            pred.get(
                "Doctor advices",
                ""
            ),

        # Keep RAW output!
        "raw_generation":
            result[
                "raw_generation"
            ],
    })


diagnostic_df = pd.DataFrame(
    summary_rows
)

diagnostic_df.to_csv(
    DIAGNOSTIC_CSV,
    index=False,
)


print("\nSaved:")
print(DIAGNOSTIC_JSON)
print(DIAGNOSTIC_CSV)

display(
    diagnostic_df[
        [
            "case_id",
            "parse_status",
            "schema_status",
            "valid",
            "Diagnosis",
        ]
    ]
)


Saved:
/content/experiment_3_adapter_diagnostic/saved_adapter_12_case_diagnostic.json
/content/experiment_3_adapter_diagnostic/saved_adapter_12_case_diagnostic.csv


,case_id,parse_status,schema_status,valid,Diagnosis
0,88,SUCCESS_EXTRACTED_JSON,VALID_SCHEMA,True,whole-mouth chronic periodontitis
1,414,SUCCESS_EXTRACTED_JSON,VALID_SCHEMA,True,missing teeth
2,596,SUCCESS_EXTRACTED_JSON,VALID_SCHEMA,True,neck tumor
3,166,PARSE_FAILURE,NO_JSON,False,
4,638,SUCCESS_EXTRACTED_JSON,VALID_SCHEMA,True,chronic impaction
5,23,SUCCESS_EXTRACTED_JSON,VALID_SCHEMA,True,chronic apical periodontitis
6,378,PARSE_FAILURE,NO_JSON,False,
7,98,SUCCESS_EXTRACTED_JSON,VALID_SCHEMA,True,chronic apical periodontitis
8,278,SUCCESS_EXTRACTED_JSON,VALID_SCHEMA,True,tooth 38 (lower left third molar) retained dec...
9,270,SUCCESS_EXTRACTED_JSON,VALID_SCHEMA,True,acute apical abscess


In [43]:
# ============================================================
# EXPERIMENT 3I
# DIAGNOSTIC SUMMARY
# ============================================================

total = len(
    diagnostic_results
)

valid_count = sum(
    bool(x["valid"])
    for x in diagnostic_results
)

parse_failures = sum(
    x["parse_status"]
    == "PARSE_FAILURE"
    for x in diagnostic_results
)

generation_exceptions = sum(
    x["parse_status"]
    == "GENERATION_EXCEPTION"
    for x in diagnostic_results
)


print("=" * 70)
print("SAVED QLoRA ADAPTER DIAGNOSTIC")
print("=" * 70)

print(
    f"Cases tested:              "
    f"{total}"
)

print(
    f"Valid 5-field JSON:        "
    f"{valid_count}/{total}"
)

print(
    f"Parse failures:            "
    f"{parse_failures}/{total}"
)

print(
    f"Generation exceptions:     "
    f"{generation_exceptions}/{total}"
)


print("\nPer-case status:")

for x in diagnostic_results:

    print(
        f"{x['case_id']:>4} | "
        f"{x['parse_status']:<25} | "
        f"{x['schema_status']}"
    )

SAVED QLoRA ADAPTER DIAGNOSTIC
Cases tested:              12
Valid 5-field JSON:        10/12
Parse failures:            2/12
Generation exceptions:     0/12

Per-case status:
  88 | SUCCESS_EXTRACTED_JSON    | VALID_SCHEMA
 414 | SUCCESS_EXTRACTED_JSON    | VALID_SCHEMA
 596 | SUCCESS_EXTRACTED_JSON    | VALID_SCHEMA
 166 | PARSE_FAILURE             | NO_JSON
 638 | SUCCESS_EXTRACTED_JSON    | VALID_SCHEMA
  23 | SUCCESS_EXTRACTED_JSON    | VALID_SCHEMA
 378 | PARSE_FAILURE             | NO_JSON
  98 | SUCCESS_EXTRACTED_JSON    | VALID_SCHEMA
 278 | SUCCESS_EXTRACTED_JSON    | VALID_SCHEMA
 270 | SUCCESS_EXTRACTED_JSON    | VALID_SCHEMA
 219 | SUCCESS_EXTRACTED_JSON    | VALID_SCHEMA
 314 | SUCCESS_EXTRACTED_JSON    | VALID_SCHEMA


## Experiment 4 — Stable Decoding Ablation

In [44]:
# ============================================================
# EXPERIMENT 4A
# STABLE DECODING GENERATION
#
# SAME:
#   - trained QLoRA adapter
#   - prompt
#   - Exp2D retrieval
#   - patient inputs
#
# CHANGED:
#   - max_new_tokens: 500 -> 350
#   - repetition_penalty: 1.15 -> 1.20
#   - no_repeat_ngram_size: 6
#
# NO RETRAINING
# ============================================================

import os
import json
import pandas as pd
import torch

EXP4_DIR = "/content/experiment_4_stable_decoding"
os.makedirs(EXP4_DIR, exist_ok=True)


@torch.inference_mode()
def stable_generate(
    qid,
    max_new_tokens=350,
):

    qid = norm_id(qid)

    assert qid in rag, (
        f"Case {qid} not found in Exp2D retrieval."
    )

    user_prompt = build_user_prompt(qid)

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    )

    device = next(model.parameters()).device

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    input_length = inputs["input_ids"].shape[1]

    outputs = model.generate(
        **inputs,

        max_new_tokens=max_new_tokens,

        # deterministic decoding
        do_sample=False,

        # stronger repetition control
        repetition_penalty=1.20,

        # NEW
        no_repeat_ngram_size=6,

        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

    generated_tokens = outputs[0][input_length:]

    raw = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=False,
    )

    parsed, parse_status = extract_json_object(raw)

    valid, schema_status = validate_prediction(parsed)

    refs = [
        norm_id(x["case_id"])
        for x in rag[qid]["retrieved_exemplars"]
    ]

    return {
        "case_id": qid,
        "query_text": rag[qid].get("query_text", ""),
        "retrieved_ids": refs,
        "raw_generation": raw,
        "parsed_prediction": parsed,
        "parse_status": parse_status,
        "schema_status": schema_status,
        "valid": bool(valid),
    }


print("[PASS] Experiment-4 stable generator ready.")

[PASS] Experiment-4 stable generator ready.


In [45]:
# ============================================================
# EXPERIMENT 4B
# TEST PREVIOUS REPETITION FAILURES FIRST
# ============================================================

REPETITION_FAILURE_CASES = [
    "166",
    "378",
]

exp4_failure_tests = []


for qid in REPETITION_FAILURE_CASES:

    print("\n" + "=" * 90)
    print(f"STABLE DECODING TEST — CASE {qid}")
    print("=" * 90)

    result = stable_generate(qid)

    exp4_failure_tests.append(result)

    print("\nQUERY:")
    print(result["query_text"])

    print("\nTOP-5:")
    print(result["retrieved_ids"])

    print("\nRAW:")
    print(result["raw_generation"])

    print("\nPARSE STATUS:")
    print(result["parse_status"])

    print("\nSCHEMA STATUS:")
    print(result["schema_status"])

    print("\nVALID:")
    print(result["valid"])

    print("\nPARSED:")

    if result["parsed_prediction"] is not None:

        print(
            json.dumps(
                result["parsed_prediction"],
                indent=2,
                ensure_ascii=False,
            )
        )

    else:
        print("NONE")


STABLE DECODING TEST — CASE 166

QUERY:
Age: 23. Sex: female. Main appeal: Blackened front teeth

TOP-5:
['319', '593', '516', '433', '129']

RAW:
{"Oral Check": "*Black stains on the anterior teeth.", "Diagnosis": "tooth discoloration", "Treatment plan": "laser whitening", "Handle": "*Laser equipment cleaning.*Local cotton roll compression suction,*Numbing injection:*None*,Blunt needle puncture/*Cotton ball expansion hole , Laser energy delivery (*Light source power)*Permanent laser bond adhesion*.After applying the adhesive agent, I took out the excess glue using a brush method, applied uniform layers of adhesive agents throughout the mouth, cured with UV lamp, blew away bubbles, reshaped the margins, ground down the friction points between the upper and lower teeth, and polished until comfortable.", "Doctor advices": "Follow-up for discomfort."}<end_of_turn>

PARSE STATUS:
SUCCESS_EXTRACTED_JSON

SCHEMA STATUS:
VALID_SCHEMA

VALID:
True

PARSED:
{
  "Oral Check": "*Black stains on 

In [46]:
# ============================================================
# EXPERIMENT 4C
# REPETITION / DEGENERATION DETECTOR
# ============================================================

from collections import Counter


def repetition_statistics(text, n=6):

    if not text:
        return {
            "num_tokens": 0,
            "num_ngrams": 0,
            "unique_ngram_ratio": 0.0,
            "max_ngram_count": 0,
            "repetition_flag": True,
        }

    tokens = str(text).lower().split()

    if len(tokens) < n:
        return {
            "num_tokens": len(tokens),
            "num_ngrams": 0,
            "unique_ngram_ratio": 1.0,
            "max_ngram_count": 1,
            "repetition_flag": False,
        }

    ngrams = [
        tuple(tokens[i:i+n])
        for i in range(len(tokens)-n+1)
    ]

    counts = Counter(ngrams)

    unique_ratio = (
        len(counts) / len(ngrams)
        if ngrams
        else 1.0
    )

    max_count = max(
        counts.values(),
        default=1,
    )

    # Diagnostic heuristic.
    # Not a clinical metric.
    repetition_flag = (
        unique_ratio < 0.70
        or max_count >= 4
    )

    return {
        "num_tokens": len(tokens),
        "num_ngrams": len(ngrams),
        "unique_ngram_ratio": unique_ratio,
        "max_ngram_count": max_count,
        "repetition_flag": repetition_flag,
    }


for result in exp4_failure_tests:

    stats = repetition_statistics(
        result["raw_generation"]
    )

    print("\nCase", result["case_id"])

    for key, value in stats.items():
        print(f"  {key}: {value}")


Case 166
  num_tokens: 92
  num_ngrams: 87
  unique_ngram_ratio: 1.0
  max_ngram_count: 1
  repetition_flag: False

Case 378
  num_tokens: 263
  num_ngrams: 258
  unique_ngram_ratio: 1.0
  max_ngram_count: 1
  repetition_flag: False


In [47]:
# ============================================================
# EXPERIMENT 4D
# SAME 12-CASE COHORT AS EXPERIMENT 3
# ============================================================

DIAGNOSTIC_CASES = [
    "88",
    "414",
    "596",
    "166",
    "638",
    "23",
    "378",
    "98",
    "278",
    "270",
    "219",
    "314",
]


exp4_results = []


for i, qid in enumerate(
    DIAGNOSTIC_CASES,
    start=1,
):

    print("\n" + "=" * 90)

    print(
        f"EXPERIMENT 4 "
        f"{i}/{len(DIAGNOSTIC_CASES)} "
        f"— CASE {qid}"
    )

    print("=" * 90)

    try:

        result = stable_generate(qid)

    except Exception as e:

        result = {
            "case_id": qid,
            "query_text": rag[qid].get(
                "query_text",
                ""
            ),
            "retrieved_ids": [
                norm_id(x["case_id"])
                for x in rag[qid][
                    "retrieved_exemplars"
                ]
            ],
            "raw_generation": None,
            "parsed_prediction": None,
            "parse_status":
                "GENERATION_EXCEPTION",
            "schema_status": str(e),
            "valid": False,
        }

    stats = repetition_statistics(
        result["raw_generation"]
    )

    result["repetition_stats"] = stats

    exp4_results.append(result)

    print("\nQUERY:")
    print(result["query_text"])

    print("\nRAW:")
    print(result["raw_generation"])

    print("\nSTATUS:")
    print(
        result["parse_status"],
        "|",
        result["schema_status"],
    )

    print(
        "Repetition flag:",
        stats["repetition_flag"]
    )


EXPERIMENT 4 1/12 — CASE 88

QUERY:
Age: 19. Sex: female. Main appeal: Crowded teeth, consultation for correction

RAW:
{"Oral Check": "*21*21*21 There is no abnormal reaction during examination", "Diagnosis": "Malformed permanent teeth.", "Treatment plan": "Full mouth orthodontic adjustment.", "Handle": "*21*11*21 Extract the tooth, clean the hole, use collagen splint fixation method, keep warm and moisten frequently until tomorrow night go back home", "Doctor advices": "Follow precautions."}<end_of_turn>

STATUS:
SUCCESS_EXTRACTED_JSON | VALID_SCHEMA
Repetition flag: False

EXPERIMENT 4 2/12 — CASE 414

QUERY:
Age: 48. Sex: female. Main appeal: The lower front teeth are missing and require examination.

RAW:
{"Oral Check": "**The gum line receded significantly due to long-term poor mastication habits (*). There was severe crowding in the anterior row, moderate crowding in the posterior rows, overbite degree I II III degrees, high vertical overlap angle IV V, bilateral midline defect

In [48]:
# ============================================================
# EXPERIMENT 4E
# EXPERIMENT 3 vs EXPERIMENT 4
# ============================================================

exp3_lookup = {
    str(x["case_id"]): x
    for x in diagnostic_results
}

exp4_lookup = {
    str(x["case_id"]): x
    for x in exp4_results
}


comparison_rows = []


for qid in DIAGNOSTIC_CASES:

    old = exp3_lookup[qid]
    new = exp4_lookup[qid]

    old_rep = repetition_statistics(
        old["raw_generation"]
    )

    new_rep = repetition_statistics(
        new["raw_generation"]
    )

    comparison_rows.append({

        "case_id": qid,

        "Exp3_Valid":
            bool(old["valid"]),

        "Exp4_Valid":
            bool(new["valid"]),

        "Exp3_Parse":
            old["parse_status"],

        "Exp4_Parse":
            new["parse_status"],

        "Exp3_Repetition":
            old_rep["repetition_flag"],

        "Exp4_Repetition":
            new_rep["repetition_flag"],

        "Exp3_Max6GramCount":
            old_rep["max_ngram_count"],

        "Exp4_Max6GramCount":
            new_rep["max_ngram_count"],

        "Exp3_Unique6GramRatio":
            old_rep["unique_ngram_ratio"],

        "Exp4_Unique6GramRatio":
            new_rep["unique_ngram_ratio"],
    })


comparison_df = pd.DataFrame(
    comparison_rows
)

display(comparison_df)


print("\n" + "=" * 70)
print("DECODING ABLATION SUMMARY")
print("=" * 70)


exp3_valid = sum(
    bool(x["valid"])
    for x in diagnostic_results
)

exp4_valid = sum(
    bool(x["valid"])
    for x in exp4_results
)

exp3_rep = sum(
    repetition_statistics(
        x["raw_generation"]
    )["repetition_flag"]
    for x in diagnostic_results
)

exp4_rep = sum(
    x["repetition_stats"][
        "repetition_flag"
    ]
    for x in exp4_results
)


print(
    f"Experiment 3 valid JSON: "
    f"{exp3_valid}/12"
)

print(
    f"Experiment 4 valid JSON: "
    f"{exp4_valid}/12"
)

print(
    f"Experiment 3 repetition flags: "
    f"{exp3_rep}/12"
)

print(
    f"Experiment 4 repetition flags: "
    f"{exp4_rep}/12"
)

,case_id,Exp3_Valid,Exp4_Valid,Exp3_Parse,Exp4_Parse,Exp3_Repetition,Exp4_Repetition,Exp3_Max6GramCount,Exp4_Max6GramCount,Exp3_Unique6GramRatio,Exp4_Unique6GramRatio
0,88,True,True,SUCCESS_EXTRACTED_JSON,SUCCESS_EXTRACTED_JSON,False,False,1,1,1.0000,1.0000
1,414,True,True,SUCCESS_EXTRACTED_JSON,SUCCESS_EXTRACTED_JSON,False,False,1,1,1.0000,1.0000
2,596,True,True,SUCCESS_EXTRACTED_JSON,SUCCESS_EXTRACTED_JSON,False,False,1,1,1.0000,1.0000
3,166,False,True,PARSE_FAILURE,SUCCESS_EXTRACTED_JSON,False,False,2,1,0.8294,1.0000
4,638,True,True,SUCCESS_EXTRACTED_JSON,SUCCESS_EXTRACTED_JSON,False,False,1,1,1.0000,1.0000
5,23,True,True,SUCCESS_EXTRACTED_JSON,SUCCESS_EXTRACTED_JSON,False,False,1,1,1.0000,1.0000
6,378,False,False,PARSE_FAILURE,PARSE_FAILURE,True,False,12,1,0.3252,1.0000
7,98,True,True,SUCCESS_EXTRACTED_JSON,SUCCESS_EXTRACTED_JSON,False,False,1,1,1.0000,1.0000
8,278,True,True,SUCCESS_EXTRACTED_JSON,SUCCESS_EXTRACTED_JSON,False,False,1,1,1.0000,1.0000
9,270,True,True,SUCCESS_EXTRACTED_JSON,SUCCESS_EXTRACTED_JSON,False,False,1,1,1.0000,1.0000



DECODING ABLATION SUMMARY
Experiment 3 valid JSON: 10/12
Experiment 4 valid JSON: 10/12
Experiment 3 repetition flags: 1/12
Experiment 4 repetition flags: 0/12


In [49]:
# ============================================================
# EXPERIMENT 4F
# SAVE 12-CASE DECODING ABLATION
# ============================================================

COMPARISON_PATH = os.path.join(
    EXP4_DIR,
    "experiment3_vs_experiment4_decoding.csv"
)

RESULTS_PATH = os.path.join(
    EXP4_DIR,
    "experiment4_12_case_results.json"
)


comparison_df.to_csv(
    COMPARISON_PATH,
    index=False,
)


with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        exp4_results,
        f,
        indent=2,
        ensure_ascii=False,
    )


print("Saved:")
print(COMPARISON_PATH)
print(RESULTS_PATH)

Saved:
/content/experiment_4_stable_decoding/experiment3_vs_experiment4_decoding.csv
/content/experiment_4_stable_decoding/experiment4_12_case_results.json


In [58]:
# ============================================================
# EXPERIMENT 4G
# FULL-TEST DECISION GATE
# ============================================================

valid_n = sum(
    bool(x["valid"])
    for x in exp4_results
)

rep_n = sum(
    x["repetition_stats"][
        "repetition_flag"
    ]
    for x in exp4_results
)


print("=" * 70)
print("EXPERIMENT 4 DECISION")
print("=" * 70)

print(
    f"Valid schema: {valid_n}/12"
)

print(
    f"Repetition flags: {rep_n}/12"
)


if valid_n == 12 and rep_n == 0:

    print(
        "\n[PASS] Decoding stability criterion met."
    )

    print(
        "Proceed to corrected full 101-case inference."
    )

else:

    print(
        "\n[STOP] Do NOT regenerate all 101 yet."
    )

    print(
        "Inspect remaining invalid/repetitive cases first."
    )

EXPERIMENT 4 DECISION
Valid schema: 10/12
Repetition flags: 0/12

[STOP] Do NOT regenerate all 101 yet.
Inspect remaining invalid/repetitive cases first.


EXPERIMENT 4.1

In [59]:
# ============================================================
# EXPERIMENT 4.1A
# INSPECT ACTUAL TERMINATION TOKENS
# ============================================================

print("tokenizer.eos_token:")
print(repr(tokenizer.eos_token))

print("\ntokenizer.eos_token_id:")
print(tokenizer.eos_token_id)

print("\n<end_of_turn> ID:")
end_turn_id = tokenizer.convert_tokens_to_ids(
    "<end_of_turn>"
)
print(end_turn_id)

print("\n<eos> ID:")
print(
    tokenizer.convert_tokens_to_ids(
        "<eos>"
    )
)

print("\nDecode end_of_turn:")
print(
    repr(
        tokenizer.decode(
            [end_turn_id]
        )
    )
)

tokenizer.eos_token:
'<end_of_turn>'

tokenizer.eos_token_id:
106

<end_of_turn> ID:
106

<eos> ID:
1

Decode end_of_turn:
'<end_of_turn>'


In [60]:
# ============================================================
# EXPERIMENT 4.1B
# CORRECT MEDGEMMA TERMINATION
# ============================================================

END_TURN_ID = tokenizer.convert_tokens_to_ids(
    "<end_of_turn>"
)

EOS_ID = tokenizer.eos_token_id


STOP_IDS = list(
    dict.fromkeys(
        [
            EOS_ID,
            END_TURN_ID,
        ]
    )
)

print("Stopping token IDs:", STOP_IDS)


@torch.inference_mode()
def stable_generate_v2(
    qid,
    max_new_tokens=350,
):

    qid = norm_id(qid)

    assert qid in rag

    user_prompt = build_user_prompt(qid)

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    )

    device = next(
        model.parameters()
    ).device

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    input_length = (
        inputs["input_ids"].shape[1]
    )

    outputs = model.generate(
        **inputs,

        max_new_tokens=max_new_tokens,

        do_sample=False,

        repetition_penalty=1.20,

        no_repeat_ngram_size=6,

        # IMPORTANT CHANGE
        eos_token_id=STOP_IDS,

        pad_token_id=tokenizer.eos_token_id,
    )

    generated_tokens = (
        outputs[0][input_length:]
    )

    raw = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=False,
    )

    parsed, parse_status = (
        extract_json_object(raw)
    )

    valid, schema_status = (
        validate_prediction(parsed)
    )

    return {
        "case_id": qid,

        "query_text":
            rag[qid].get(
                "query_text",
                ""
            ),

        "retrieved_ids": [
            norm_id(x["case_id"])
            for x in rag[qid][
                "retrieved_exemplars"
            ]
        ],

        "raw_generation": raw,

        "parsed_prediction": parsed,

        "parse_status": parse_status,

        "schema_status": schema_status,

        "valid": bool(valid),

        "generated_tokens":
            int(len(generated_tokens)),
    }

Stopping token IDs: [106]


In [61]:
# ============================================================
# EXPERIMENT 4.1C
# STRICT GENERATION-STRUCTURE CHECK
# ============================================================

def generation_quality(result):

    raw = (
        result.get(
            "raw_generation"
        )
        or ""
    )

    status = []

    if not result["valid"]:
        status.append(
            "INVALID_JSON"
        )

    # Number of obvious JSON starts
    json_starts = raw.count(
        '{"Oral Check"'
    )

    if json_starts > 1:
        status.append(
            "MULTIPLE_REPORTS"
        )

    # Continued generation after explicit turn end
    if "<end_of_turn>" in raw:

        trailing = (
            raw.split(
                "<end_of_turn>",
                1,
            )[1]
            .strip()
        )

        # Ignore pure terminal tokens
        trailing = (
            trailing
            .replace("<eos>", "")
            .strip()
        )

        if trailing:
            status.append(
                "TRAILING_GENERATION"
            )

    # Did generation hit max token budget?
    if (
        result.get(
            "generated_tokens",
            0
        )
        >= 350
    ):
        status.append(
            "MAX_TOKEN_TRUNCATION"
        )

    if not status:
        status = [
            "CLEAN"
        ]

    return status

In [62]:
TEST_CASES = [
    "414",   # multiple-report continuation
    "378",   # long procedural continuation
    "219",   # long finding enumeration
]


v2_results = []


for qid in TEST_CASES:

    print("\n" + "=" * 90)
    print(
        f"TERMINATION TEST — CASE {qid}"
    )
    print("=" * 90)

    result = stable_generate_v2(
        qid
    )

    quality = generation_quality(
        result
    )

    result[
        "generation_quality"
    ] = quality

    v2_results.append(
        result
    )

    print("\nQUERY:")
    print(
        result["query_text"]
    )

    print("\nRAW:")
    print(
        result["raw_generation"]
    )

    print("\nPARSE:")
    print(
        result["parse_status"]
    )

    print("\nSCHEMA:")
    print(
        result["schema_status"]
    )

    print("\nTOKENS:")
    print(
        result["generated_tokens"]
    )

    print("\nQUALITY:")
    print(
        quality
    )

    print("\nPARSED:")

    if result[
        "parsed_prediction"
    ]:

        print(
            json.dumps(
                result[
                    "parsed_prediction"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )

    else:
        print("NONE")


TERMINATION TEST — CASE 414

QUERY:
Age: 48. Sex: female. Main appeal: The lower front teeth are missing and require examination.

RAW:
{"Oral Check": "**The gum line receded significantly due to long-term poor mastication habits (*). There was severe crowding in the anterior row, moderate crowding in the posterior rows, overbite degree I II III degrees, high vertical overlap angle IV V, bilateral midline defects", "Diagnosis": "Malformed permanent teeth : malformation type i nervous system defect.", "Treatment plan": "Full set of braces correction + surgical repositioning of maxillary lateral incisors +/- distal wedge resection under the mandibular molars == Full set of braces correction + maxillofacial reconstruction implants.", "Handle": "Explain the condition, keep the patient calm, ask him questions, show pictures, discuss various methods, compare pros and cons, choose an appropriate method according to the patient’s preferences and budget, and get detailed instructions from each

4.2A — First-valid-JSON stopping criterion

In [63]:
# ============================================================
# EXPERIMENT 4.2A
# STOP AS SOON AS FIRST VALID 5-FIELD JSON IS COMPLETE
# ============================================================

import json
import torch

from transformers import StoppingCriteria, StoppingCriteriaList


EXPECTED_FIELDS = [
    "Oral Check",
    "Diagnosis",
    "Treatment plan",
    "Handle",
    "Doctor advices",
]


def find_first_complete_json(text):
    """
    Returns:
        (dict, json_text)
    for the FIRST complete top-level JSON object
    containing all five expected fields.

    Otherwise:
        (None, None)
    """

    if not text:
        return None, None

    starts = [
        i
        for i, ch in enumerate(text)
        if ch == "{"
    ]

    for start in starts:

        depth = 0
        in_string = False
        escape = False

        for i in range(start, len(text)):

            ch = text[i]

            if escape:
                escape = False
                continue

            if ch == "\\" and in_string:
                escape = True
                continue

            if ch == '"':
                in_string = not in_string
                continue

            if in_string:
                continue

            if ch == "{":
                depth += 1

            elif ch == "}":

                depth -= 1

                if depth == 0:

                    candidate = text[
                        start:i + 1
                    ]

                    try:
                        obj = json.loads(
                            candidate
                        )
                    except Exception:
                        break

                    if (
                        isinstance(obj, dict)
                        and all(
                            field in obj
                            for field in EXPECTED_FIELDS
                        )
                    ):
                        return obj, candidate

                    break

    return None, None

In [64]:
class FirstValidJSONStoppingCriteria(
    StoppingCriteria
):

    def __init__(
        self,
        tokenizer,
        prompt_length,
    ):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length

    def __call__(
        self,
        input_ids,
        scores,
        **kwargs,
    ):

        generated_ids = input_ids[
            0,
            self.prompt_length:
        ]

        text = self.tokenizer.decode(
            generated_ids,
            skip_special_tokens=False,
        )

        obj, _ = find_first_complete_json(
            text
        )

        return obj is not None

In [65]:
# ============================================================
# EXPERIMENT 4.2B
# STABLE + FIRST VALID JSON GENERATION
# ============================================================

@torch.inference_mode()
def stable_generate_v3(
    qid,
    max_new_tokens=350,
):

    qid = norm_id(qid)

    assert qid in rag

    user_prompt = build_user_prompt(
        qid
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    )

    device = next(
        model.parameters()
    ).device

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    input_length = (
        inputs["input_ids"].shape[1]
    )

    stopping_criteria = StoppingCriteriaList([
        FirstValidJSONStoppingCriteria(
            tokenizer=tokenizer,
            prompt_length=input_length,
        )
    ])

    outputs = model.generate(
        **inputs,

        max_new_tokens=max_new_tokens,

        do_sample=False,

        repetition_penalty=1.20,

        no_repeat_ngram_size=6,

        stopping_criteria=
            stopping_criteria,

        eos_token_id=[
            tokenizer.eos_token_id,
            tokenizer.convert_tokens_to_ids(
                "<end_of_turn>"
            ),
        ],

        pad_token_id=
            tokenizer.eos_token_id,
    )

    generated_tokens = (
        outputs[0][input_length:]
    )

    raw = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=False,
    )

    # IMPORTANT:
    # Always take FIRST complete valid report.
    parsed, json_text = (
        find_first_complete_json(
            raw
        )
    )

    if parsed is not None:

        parse_status = (
            "SUCCESS_FIRST_VALID_JSON"
        )

        valid = True

        schema_status = (
            "VALID_SCHEMA"
        )

    else:

        parse_status = (
            "PARSE_FAILURE"
        )

        valid = False

        schema_status = (
            "NO_COMPLETE_5_FIELD_JSON"
        )

    return {

        "case_id": qid,

        "query_text":
            rag[qid].get(
                "query_text",
                ""
            ),

        "retrieved_ids": [
            norm_id(x["case_id"])
            for x in rag[qid][
                "retrieved_exemplars"
            ]
        ],

        "raw_generation":
            raw,

        "first_valid_json_text":
            json_text,

        "parsed_prediction":
            parsed,

        "parse_status":
            parse_status,

        "schema_status":
            schema_status,

        "valid":
            valid,

        "generated_tokens":
            int(
                len(generated_tokens)
            ),

        "hit_max_tokens":
            len(generated_tokens)
            >= max_new_tokens,
    }

In [66]:
TEST_CASES = [
    "414",
    "378",
    "219",
]

v3_results = []


for qid in TEST_CASES:

    print("\n" + "=" * 90)
    print(
        f"FIRST-JSON STOP TEST — CASE {qid}"
    )
    print("=" * 90)

    result = stable_generate_v3(
        qid
    )

    v3_results.append(
        result
    )

    print("\nQUERY:")
    print(
        result["query_text"]
    )

    print("\nRAW:")
    print(
        result["raw_generation"]
    )

    print("\nSTATUS:")
    print(
        result["parse_status"]
    )

    print("\nSCHEMA:")
    print(
        result["schema_status"]
    )

    print("\nGENERATED TOKENS:")
    print(
        result["generated_tokens"]
    )

    print("\nHIT MAX TOKENS:")
    print(
        result["hit_max_tokens"]
    )

    print("\nPARSED:")

    if result[
        "parsed_prediction"
    ] is not None:

        print(
            json.dumps(
                result[
                    "parsed_prediction"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )

    else:

        print("NONE")


FIRST-JSON STOP TEST — CASE 414

QUERY:
Age: 48. Sex: female. Main appeal: The lower front teeth are missing and require examination.

RAW:
{"Oral Check": "**The gum line receded significantly due to long-term poor mastication habits (*). There was severe crowding in the anterior row, moderate crowding in the posterior rows, overbite degree I II III degrees, high vertical overlap angle IV V, bilateral midline defects", "Diagnosis": "Malformed permanent teeth : malformation type i nervous system defect.", "Treatment plan": "Full set of braces correction + surgical repositioning of maxillary lateral incisors +/- distal wedge resection under the mandibular molars == Full set of braces correction + maxillofacial reconstruction implants.", "Handle": "Explain the condition, keep the patient calm, ask him questions, show pictures, discuss various methods, compare pros and cons, choose an appropriate method according to the patient’s preferences and budget, and get detailed instructions from 

Experiment 4.3 — Concise structured-output constraint

In [67]:
# ============================================================
# EXPERIMENT 4.3
# CONCISE STRUCTURED OUTPUT ABLATION
# ============================================================

OUTPUT_CONSTRAINT = """
OUTPUT CONSTRAINT:
Return exactly one JSON object containing exactly these five keys:
"Oral Check", "Diagnosis", "Treatment plan", "Handle", "Doctor advices".

Keep every field concise.
Use at most 60 words for "Oral Check".
Use at most 20 words for "Diagnosis".
Use at most 30 words for "Treatment plan".
Use at most 80 words for "Handle".
Use at most 30 words for "Doctor advices".

Do not create lists of additional findings.
Do not repeat procedures.
Do not generate another JSON object.
End immediately after the final closing brace.
""".strip()

In [68]:
@torch.inference_mode()
def stable_generate_v4(
    qid,
    max_new_tokens=350,
):

    qid = norm_id(qid)

    assert qid in rag

    original_user_prompt = (
        build_user_prompt(qid)
    )

    # ONLY Experiment-4.3 intervention
    user_prompt = (
        original_user_prompt
        + "\n\n"
        + OUTPUT_CONSTRAINT
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    )

    device = next(
        model.parameters()
    ).device

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    input_length = (
        inputs["input_ids"].shape[1]
    )

    stopping_criteria = (
        StoppingCriteriaList([
            FirstValidJSONStoppingCriteria(
                tokenizer=tokenizer,
                prompt_length=input_length,
            )
        ])
    )

    outputs = model.generate(
        **inputs,

        max_new_tokens=max_new_tokens,

        do_sample=False,

        repetition_penalty=1.20,

        no_repeat_ngram_size=6,

        stopping_criteria=
            stopping_criteria,

        eos_token_id=[
            tokenizer.eos_token_id,
            tokenizer.convert_tokens_to_ids(
                "<end_of_turn>"
            ),
        ],

        pad_token_id=
            tokenizer.eos_token_id,
    )

    generated_tokens = (
        outputs[0][input_length:]
    )

    raw = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=False,
    )

    parsed, json_text = (
        find_first_complete_json(raw)
    )

    if parsed is not None:

        parse_status = (
            "SUCCESS_FIRST_VALID_JSON"
        )

        schema_status = (
            "VALID_SCHEMA"
        )

        valid = True

    else:

        parse_status = "PARSE_FAILURE"

        schema_status = (
            "NO_COMPLETE_5_FIELD_JSON"
        )

        valid = False

    return {

        "case_id": qid,

        "query_text":
            rag[qid].get(
                "query_text",
                ""
            ),

        "retrieved_ids": [
            norm_id(x["case_id"])
            for x in
            rag[qid][
                "retrieved_exemplars"
            ]
        ],

        "raw_generation":
            raw,

        "first_valid_json_text":
            json_text,

        "parsed_prediction":
            parsed,

        "parse_status":
            parse_status,

        "schema_status":
            schema_status,

        "valid":
            valid,

        "generated_tokens":
            int(
                len(generated_tokens)
            ),

        "hit_max_tokens":
            len(generated_tokens)
            >= max_new_tokens,
    }

In [69]:
TEST_CASES = [
    "378",
    "219",
]

v4_results = []


for qid in TEST_CASES:

    print("\n" + "=" * 90)
    print(
        f"CONCISE OUTPUT TEST — CASE {qid}"
    )
    print("=" * 90)

    result = stable_generate_v4(
        qid
    )

    v4_results.append(
        result
    )

    print("\nQUERY:")
    print(
        result["query_text"]
    )

    print("\nRAW:")
    print(
        result["raw_generation"]
    )

    print("\nSTATUS:")
    print(
        result["parse_status"]
    )

    print("\nSCHEMA:")
    print(
        result["schema_status"]
    )

    print("\nGENERATED TOKENS:")
    print(
        result["generated_tokens"]
    )

    print("\nHIT MAX TOKENS:")
    print(
        result["hit_max_tokens"]
    )

    print("\nPARSED:")

    if result[
        "parsed_prediction"
    ] is not None:

        print(
            json.dumps(
                result[
                    "parsed_prediction"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )

    else:

        print("NONE")


CONCISE OUTPUT TEST — CASE 378

QUERY:
Age: 17. Sex: female. Main appeal: The right upper back tooth has been painful due to hot and cold stimulation for a week

RAW:
{"Oral Check": "*16*15 Black spots detected.", "Diagnosis": "Malformed teeth Malformation of teeth Chronic apical periodontitis", "Treatment plan": "Inlay restoration Crown restoration Root Canal Treatments", "Handle": "Explain the condition, keep the whole body warm, take medicine (*16*, *15)", "Doctor advices": "Follow-up for discomfort."}

STATUS:
SUCCESS_FIRST_VALID_JSON

SCHEMA:
VALID_SCHEMA

GENERATED TOKENS:
80

HIT MAX TOKENS:
False

PARSED:
{
  "Oral Check": "*16*15 Black spots detected.",
  "Diagnosis": "Malformed teeth Malformation of teeth Chronic apical periodontitis",
  "Treatment plan": "Inlay restoration Crown restoration Root Canal Treatments",
  "Handle": "Explain the condition, keep the whole body warm, take medicine (*16*, *15)",
  "Doctor advices": "Follow-up for discomfort."
}

CONCISE OUTPUT TEST —

EXPERIMENT 4.3B to test the 12 cases


In [70]:
# ============================================================
# EXPERIMENT 4.3B
# FINAL 12-CASE VALIDATION OF CANDIDATE INFERENCE CONFIGURATION
# ============================================================

FINAL_DIAGNOSTIC_CASES = [
    "88",
    "414",
    "596",
    "166",
    "638",
    "23",
    "378",
    "98",
    "278",
    "270",
    "219",
    "314",
]


final_12_results = []


for i, qid in enumerate(
    FINAL_DIAGNOSTIC_CASES,
    start=1,
):

    print("\n" + "=" * 90)

    print(
        f"FINAL INFERENCE VALIDATION "
        f"{i}/12 — CASE {qid}"
    )

    print("=" * 90)

    try:

        result = stable_generate_v4(qid)

    except Exception as e:

        result = {
            "case_id": qid,
            "query_text":
                rag[qid].get(
                    "query_text",
                    ""
                ),
            "retrieved_ids": [
                norm_id(x["case_id"])
                for x in rag[qid][
                    "retrieved_exemplars"
                ]
            ],
            "raw_generation": None,
            "first_valid_json_text": None,
            "parsed_prediction": None,
            "parse_status":
                "GENERATION_EXCEPTION",
            "schema_status":
                str(e),
            "valid": False,
            "generated_tokens": 0,
            "hit_max_tokens": False,
        }


    # --------------------------------------
    # Additional generation-quality checks
    # --------------------------------------

    raw = (
        result.get(
            "raw_generation"
        )
        or ""
    )

    parsed = (
        result.get(
            "parsed_prediction"
        )
        or {}
    )


    # Exactly the expected five fields?
    exact_fields = (
        set(parsed.keys())
        ==
        set(EXPECTED_FIELDS)
    )


    # Any empty fields?
    empty_fields = [
        field
        for field in EXPECTED_FIELDS
        if not str(
            parsed.get(
                field,
                ""
            )
        ).strip()
    ]


    result[
        "exact_five_fields"
    ] = exact_fields

    result[
        "empty_fields"
    ] = empty_fields


    # Clean technical success definition
    result[
        "technical_success"
    ] = bool(
        result["valid"]
        and exact_fields
        and not result[
            "hit_max_tokens"
        ]
    )


    final_12_results.append(
        result
    )


    print("\nRAW:")
    print(raw)

    print("\nPARSE:")
    print(
        result[
            "parse_status"
        ]
    )

    print(
        "SCHEMA:",
        result[
            "schema_status"
        ]
    )

    print(
        "TOKENS:",
        result[
            "generated_tokens"
        ]
    )

    print(
        "HIT MAX:",
        result[
            "hit_max_tokens"
        ]
    )

    print(
        "EXACT 5 FIELDS:",
        exact_fields
    )

    print(
        "EMPTY FIELDS:",
        empty_fields
    )

    print(
        "TECHNICAL SUCCESS:",
        result[
            "technical_success"
        ]
    )


FINAL INFERENCE VALIDATION 1/12 — CASE 88

RAW:
{"Oral Check": "**1:**: Unaligned bilateral midline palatal cusp relationship.", "Diagnosis": "Malformed permanent teeth : Malformation type II (K04.3)", "Treatment plan": "**1:** Removal of malformed teeth.", "Handle": "**1:* Keep the patient calm during preparation and keep him awake throughout the process until he goes home. Take pictures frequently during processing.", "Doctor advices": "Follow-up for discomfort."}

PARSE:
SUCCESS_FIRST_VALID_JSON
SCHEMA: VALID_SCHEMA
TOKENS: 95
HIT MAX: False
EXACT 5 FIELDS: True
EMPTY FIELDS: []
TECHNICAL SUCCESS: True

FINAL INFERENCE VALIDATION 2/12 — CASE 414

RAW:
{"Oral Check": "**The gum line recedes significantly due to long-term ill-fitting dentures, exposing large areas of neck roots.", "Diagnosis": "Poor denture support type II", "Treatment plan": "Full mouth removable prosthetics + partial fixed restorations", "Handle": "Explain the condition, keep the patient calm, ask him to sit down c

In [71]:
# ============================================================
# EXPERIMENT 4.3C
# FINAL 12-CASE TECHNICAL SUMMARY
# ============================================================

summary_rows = []


for result in final_12_results:

    summary_rows.append({

        "case_id":
            result["case_id"],

        "valid_json":
            result["valid"],

        "exact_five_fields":
            result[
                "exact_five_fields"
            ],

        "empty_fields":
            ", ".join(
                result[
                    "empty_fields"
                ]
            ),

        "generated_tokens":
            result[
                "generated_tokens"
            ],

        "hit_max_tokens":
            result[
                "hit_max_tokens"
            ],

        "technical_success":
            result[
                "technical_success"
            ],

        "parse_status":
            result[
                "parse_status"
            ],
    })


final_12_df = pd.DataFrame(
    summary_rows
)

display(final_12_df)


n = len(final_12_df)

success_n = int(
    final_12_df[
        "technical_success"
    ].sum()
)

valid_n = int(
    final_12_df[
        "valid_json"
    ].sum()
)

max_hit_n = int(
    final_12_df[
        "hit_max_tokens"
    ].sum()
)


print("\n" + "=" * 72)
print("FINAL 12-CASE INFERENCE VALIDATION")
print("=" * 72)

print(
    f"Valid JSON:            "
    f"{valid_n}/{n}"
)

print(
    f"Technical success:     "
    f"{success_n}/{n}"
)

print(
    f"Max-token truncations: "
    f"{max_hit_n}/{n}"
)

print(
    f"Success rate:          "
    f"{100 * success_n/n:.1f}%"
)

,case_id,valid_json,exact_five_fields,empty_fields,generated_tokens,hit_max_tokens,technical_success,parse_status
0,88,True,True,,95,False,True,SUCCESS_FIRST_VALID_JSON
1,414,True,True,,220,False,True,SUCCESS_FIRST_VALID_JSON
2,596,True,True,,52,False,True,SUCCESS_FIRST_VALID_JSON
3,166,False,False,"Oral Check, Diagnosis, Treatment plan, Handle,...",350,True,False,PARSE_FAILURE
4,638,True,True,,190,False,True,SUCCESS_FIRST_VALID_JSON
5,23,True,True,,116,False,True,SUCCESS_FIRST_VALID_JSON
6,378,True,True,,80,False,True,SUCCESS_FIRST_VALID_JSON
7,98,True,True,,150,False,True,SUCCESS_FIRST_VALID_JSON
8,278,True,True,,90,False,True,SUCCESS_FIRST_VALID_JSON
9,270,True,True,,141,False,True,SUCCESS_FIRST_VALID_JSON



FINAL 12-CASE INFERENCE VALIDATION
Valid JSON:            11/12
Technical success:     11/12
Max-token truncations: 1/12
Success rate:          91.7%


In [75]:
qid = "166"

original_user_prompt = build_user_prompt(qid)

user_prompt = (
    original_user_prompt
    + "\n\n"
    + OUTPUT_CONSTRAINT
)

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT,
    },
    {
        "role": "user",
        "content": user_prompt,
    },
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print("===== LAST 1500 CHARACTERS OF PROMPT =====")
print(prompt[-1500:])

print("\n===== repr() OF LAST 500 =====")
print(repr(prompt[-500:]))

===== LAST 1500 CHARACTERS OF PROMPT =====
arly. Eat and drink after 2 hours.

Retrieved reference 5:
Patient ID: 129
Retrieval score: 0.120915
Oral Check: tooth 27 (upper left second molar) a large area of filling material is seen in the crown, which is painful to percussion. cbct shows that the filling material in the root canal is acceptable, and the mesial and buccal periapical low-density shadows tooth 28 (upper left third molar) there is an underlying impaction in the mesial and low position, and mesial gingival tenderness. cbct shows that the filling material in the mesial and buccal areas is acceptable. low level impaction
Diagnosis: tooth 27 (upper left second molar) chronic apical periodontitis tooth 28 (upper left third molar) impacted teeth
Treatment plan: tooth 27 (upper left second molar). observation is recommended. tooth 28 (upper left third molar). extraction is recommended.
Handle: Not processed
Doctor advices: Follow-up for discomfort.

OUTPUT CONSTRAINT:
Return exac

In [76]:
import json

with open(
    TRAIN_JSONL,
    "r",
    encoding="utf-8"
) as f:
    example = json.loads(
        next(f)
    )

print(example)

{'case_id': '224', 'messages': [{'role': 'system', 'content': 'You are a dental clinical documentation assistant. You will be given a patient\'s Age, Sex, and Main appeal (chief complaint), along with several similar PAST patient cases retrieved for reference.\n\nCRITICAL: The retrieved cases are REFERENCE MATERIAL from OTHER patients. They are NOT facts about the current patient. Do not assume the current patient has the same findings, tooth numbers, diagnoses, or treatments as any retrieved case unless the current patient\'s own Age/Sex/Main appeal genuinely supports it.\n\nRules:\n1. NEVER invent or copy a tooth number, ICD code, diagnosis, finding, medication, procedure, or treatment that is not directly supported by the current patient\'s own Main appeal or by a clear, justified pattern across the retrieved cases.\n2. If there is insufficient evidence to determine a field, output exactly "Not determined" for that field. Do not guess, and do not fill it in from the single most simi

FINAL RUN


In [77]:
# ============================================================
# FINAL RUN-2 — ROBUST QLORA INFERENCE ON ALL 101 TEST PATIENTS
# ============================================================

import os
import json
import pandas as pd
from tqdm.auto import tqdm

RUN2_DIR = "/content/drive/MyDrive/Exp2D_QLoRA_Run2"
os.makedirs(RUN2_DIR, exist_ok=True)

RUN2_RESULTS_JSON = os.path.join(
    RUN2_DIR,
    "run2_robust_generation_all101.json"
)

RUN2_PREDICTIONS_JSON = os.path.join(
    RUN2_DIR,
    "run2_predictions_successful.json"
)

RUN2_FAILURES_JSON = os.path.join(
    RUN2_DIR,
    "run2_generation_failures.json"
)


# ------------------------------------------------------------
# Get the EXACT 101 test IDs from Exp2D RAG
# ------------------------------------------------------------

TEST_IDS = [norm_id(x) for x in rag.keys()]

assert len(TEST_IDS) == 101
assert len(set(TEST_IDS)) == 101

print("[PASS] Test patients:", len(TEST_IDS))


# ------------------------------------------------------------
# Resume-safe loading
# ------------------------------------------------------------

if os.path.exists(RUN2_RESULTS_JSON):

    with open(
        RUN2_RESULTS_JSON,
        "r",
        encoding="utf-8",
    ) as f:
        run2_results = json.load(f)

    run2_results = {
        norm_id(k): v
        for k, v in run2_results.items()
    }

    print(
        f"[RESUME] Loaded {len(run2_results)}/101 completed cases."
    )

else:

    run2_results = {}

    print("[NEW] Starting Run-2 from scratch.")


# ------------------------------------------------------------
# Generate
# ------------------------------------------------------------

for qid in tqdm(TEST_IDS):

    if qid in run2_results:
        continue

    try:

        result = stable_generate_v4(
            qid,
            max_new_tokens=350,
        )

        pred = (
            result.get("parsed_prediction")
            or {}
        )

        exact_five_fields = (
            set(pred.keys())
            == set(EXPECTED_FIELDS)
        )

        empty_fields = [
            field
            for field in EXPECTED_FIELDS
            if not str(
                pred.get(field, "")
            ).strip()
        ]

        technical_success = bool(
            result.get("valid", False)
            and exact_five_fields
            and not result.get(
                "hit_max_tokens",
                False,
            )
        )

        result["exact_five_fields"] = (
            exact_five_fields
        )

        result["empty_fields"] = (
            empty_fields
        )

        result["technical_success"] = (
            technical_success
        )

    except Exception as e:

        result = {

            "case_id": qid,

            "query_text":
                rag[qid].get(
                    "query_text",
                    ""
                ),

            "retrieved_ids": [
                norm_id(x["case_id"])
                for x in
                rag[qid][
                    "retrieved_exemplars"
                ]
            ],

            "raw_generation": None,

            "first_valid_json_text":
                None,

            "parsed_prediction":
                None,

            "parse_status":
                "GENERATION_EXCEPTION",

            "schema_status":
                "NO_VALID_SCHEMA",

            "valid": False,

            "generated_tokens": 0,

            "hit_max_tokens": False,

            "exact_five_fields":
                False,

            "empty_fields":
                EXPECTED_FIELDS.copy(),

            "technical_success":
                False,

            "exception":
                repr(e),
        }


    run2_results[qid] = result


    # ------------------------------------------
    # SAVE AFTER EVERY PATIENT
    # ------------------------------------------

    with open(
        RUN2_RESULTS_JSON,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            run2_results,
            f,
            indent=2,
            ensure_ascii=False,
        )


print(
    f"[PASS] Completed {len(run2_results)}/101 patients."
)

[PASS] Test patients: 101
[NEW] Starting Run-2 from scratch.


  0%|          | 0/101 [00:00<?, ?it/s]

[PASS] Completed 101/101 patients.


In [78]:
# ============================================================
# SPLIT SUCCESSFUL PREDICTIONS AND TECHNICAL FAILURES
# ============================================================

successful_predictions = {}

generation_failures = {}


for qid, result in run2_results.items():

    if result.get(
        "technical_success",
        False,
    ):

        successful_predictions[qid] = (
            result[
                "parsed_prediction"
            ]
        )

    else:

        generation_failures[qid] = {
            "parse_status":
                result.get(
                    "parse_status"
                ),

            "schema_status":
                result.get(
                    "schema_status"
                ),

            "generated_tokens":
                result.get(
                    "generated_tokens"
                ),

            "hit_max_tokens":
                result.get(
                    "hit_max_tokens"
                ),

            "raw_generation":
                result.get(
                    "raw_generation"
                ),

            "exception":
                result.get(
                    "exception"
                ),
        }


with open(
    RUN2_PREDICTIONS_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        successful_predictions,
        f,
        indent=2,
        ensure_ascii=False,
    )


with open(
    RUN2_FAILURES_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        generation_failures,
        f,
        indent=2,
        ensure_ascii=False,
    )


print(
    "Successful:",
    len(successful_predictions),
)

print(
    "Technical failures:",
    len(generation_failures),
)

assert (
    len(successful_predictions)
    + len(generation_failures)
    == 101
)

Successful: 77
Technical failures: 24


In [79]:
# ============================================================
# RUN-2 TECHNICAL GENERATION EVALUATION
# ============================================================

technical_rows = []


for qid in TEST_IDS:

    r = run2_results[qid]

    pred = (
        r.get(
            "parsed_prediction"
        )
        or {}
    )

    all_not_determined = (
        r.get(
            "technical_success",
            False,
        )
        and all(
            str(
                pred.get(
                    field,
                    ""
                )
            ).strip().lower()
            == "not determined"

            for field
            in EXPECTED_FIELDS
        )
    )

    technical_rows.append({

        "case_id": qid,

        "valid_json":
            bool(
                r.get(
                    "valid",
                    False,
                )
            ),

        "exact_five_fields":
            bool(
                r.get(
                    "exact_five_fields",
                    False,
                )
            ),

        "technical_success":
            bool(
                r.get(
                    "technical_success",
                    False,
                )
            ),

        "generated_tokens":
            r.get(
                "generated_tokens",
                0,
            ),

        "hit_max_tokens":
            bool(
                r.get(
                    "hit_max_tokens",
                    False,
                )
            ),

        "parse_status":
            r.get(
                "parse_status"
            ),

        "all_fields_not_determined":
            all_not_determined,
    })


technical_df = pd.DataFrame(
    technical_rows
)


N = len(technical_df)

n_success = int(
    technical_df[
        "technical_success"
    ].sum()
)

n_valid = int(
    technical_df[
        "valid_json"
    ].sum()
)

n_truncated = int(
    technical_df[
        "hit_max_tokens"
    ].sum()
)

n_all_nd = int(
    technical_df[
        "all_fields_not_determined"
    ].sum()
)


print("=" * 70)
print("RUN-2 TECHNICAL GENERATION RESULTS")
print("=" * 70)

print(f"Total patients:           {N}")

print(
    f"Valid JSON:              "
    f"{n_valid}/{N} "
    f"({100*n_valid/N:.1f}%)"
)

print(
    f"Technical success:       "
    f"{n_success}/{N} "
    f"({100*n_success/N:.1f}%)"
)

print(
    f"Technical failures:      "
    f"{N-n_success}/{N} "
    f"({100*(N-n_success)/N:.1f}%)"
)

print(
    f"Max-token truncations:   "
    f"{n_truncated}/{N}"
)

print(
    f"True all-field "
    f"'Not determined':       "
    f"{n_all_nd}/{N}"
)


TECHNICAL_CSV = os.path.join(
    RUN2_DIR,
    "run2_technical_evaluation.csv"
)

technical_df.to_csv(
    TECHNICAL_CSV,
    index=False,
)

RUN-2 TECHNICAL GENERATION RESULTS
Total patients:           101
Valid JSON:              77/101 (76.2%)
Technical success:       77/101 (76.2%)
Technical failures:      24/101 (23.8%)
Max-token truncations:   22/101
True all-field 'Not determined':       0/101


In [82]:
import os

GT_RUN1_CSV = CORRECTED_CSV

base_df = pd.read_csv(
    GT_RUN1_CSV
)

base_df["case_id"] = (
    base_df["case_id"]
    .map(norm_id)
)

assert len(base_df) == 101
assert base_df["case_id"].nunique() == 101

In [83]:
# ============================================================
# BUILD GT vs RUN-2 TABLE
# ============================================================

rows = []


for _, old_row in base_df.iterrows():

    qid = norm_id(
        old_row["case_id"]
    )

    result = run2_results[qid]

    success = result.get(
        "technical_success",
        False,
    )

    pred = (
        result.get(
            "parsed_prediction"
        )
        if success
        else None
    )


    row = {

        "case_id": qid,

        "Age":
            old_row.get(
                "Age",
                ""
            ),

        "Sex":
            old_row.get(
                "Sex",
                ""
            ),

        "Main appeal":
            old_row.get(
                "Main appeal",
                ""
            ),

        "generation_success":
            success,

        "parse_status":
            result.get(
                "parse_status"
            ),

        "generated_tokens":
            result.get(
                "generated_tokens"
            ),

        "hit_max_tokens":
            result.get(
                "hit_max_tokens"
            ),
    }


    for field in EXPECTED_FIELDS:

        row[
            f"GT_{field}"
        ] = old_row[
            f"GT_{field}"
        ]

        row[
            f"PRED_{field}"
        ] = (
            pred.get(
                field,
                ""
            )
            if pred
            else None
        )


    rows.append(row)


run2_eval_df = pd.DataFrame(
    rows
)


assert len(run2_eval_df) == 101

assert (
    run2_eval_df[
        "case_id"
    ].nunique()
    == 101
)


RUN2_GT_PRED_CSV = os.path.join(
    RUN2_DIR,
    "run2_ground_truth_vs_predictions.csv"
)

run2_eval_df.to_csv(
    RUN2_GT_PRED_CSV,
    index=False,
)


print("[PASS] Created Run-2 GT vs prediction table.")
print(run2_eval_df.shape)

[PASS] Created Run-2 GT vs prediction table.
(101, 18)


In [84]:
clinical_df = (
    run2_eval_df[
        run2_eval_df[
            "generation_success"
        ] == True
    ]
    .copy()
)

print(
    "Clinical evaluation:",
    len(clinical_df),
    "successful generations"
)

print(
    "Technical failures excluded:",
    101 - len(clinical_df)
)

Clinical evaluation: 77 successful generations
Technical failures excluded: 24


In [85]:
def clean_text(x):

    if pd.isna(x):
        return ""

    return str(x).strip()


def combine_report(
    row,
    prefix,
):

    return " ".join([
        clean_text(
            row[
                f"{prefix}_{field}"
            ]
        )
        for field
        in EXPECTED_FIELDS
    ]).strip()


clinical_df[
    "GT_FULL"
] = clinical_df.apply(
    lambda r:
        combine_report(
            r,
            "GT",
        ),
    axis=1,
)


clinical_df[
    "PRED_FULL"
] = clinical_df.apply(
    lambda r:
        combine_report(
            r,
            "PRED",
        ),
    axis=1,
)

In [86]:
# ============================================================
# STEP 7 — FINAL RUN-2 CLINICAL EVALUATION
# ============================================================

import os
import re
import json
import math
import numpy as np
import pandas as pd

from collections import Counter

# ------------------------------------------------------------
# NLTK
# ------------------------------------------------------------

import nltk

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt", quiet=True)

from nltk.translate.bleu_score import (
    corpus_bleu,
    sentence_bleu,
    SmoothingFunction,
)

from nltk.translate.meteor_score import meteor_score


# ------------------------------------------------------------
# ROUGE
# ------------------------------------------------------------

try:
    from rouge_score import rouge_scorer
except ImportError:
    !pip -q install rouge-score
    from rouge_score import rouge_scorer


rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True,
)

smooth = SmoothingFunction().method1


# ============================================================
# BASIC HELPERS
# ============================================================

def safe_text(x):

    if x is None:
        return ""

    if isinstance(x, float) and np.isnan(x):
        return ""

    return str(x).strip()


def tokenize_text(text):

    return re.findall(
        r"\b\w+\b",
        safe_text(text).lower()
    )


# ============================================================
# FDI EXTRACTION
#
# STRICT:
#   tooth 48
#   tooth 48 (...)
#   *48
#
# Avoid treating arbitrary two-digit numbers as teeth.
# ============================================================

FDI_TOOTH_RE = re.compile(
    r"\btooth\s*([1-4][1-8])\b",
    flags=re.IGNORECASE,
)

FDI_STAR_RE = re.compile(
    r"(?<!\d)\*([1-4][1-8])(?=\D|$)",
    flags=re.IGNORECASE,
)


def extract_fdi(text):

    text = safe_text(text)

    teeth = set()

    for x in FDI_TOOTH_RE.findall(text):
        teeth.add(x)

    for x in FDI_STAR_RE.findall(text):
        teeth.add(x)

    return teeth


# ============================================================
# ICD-10 EXTRACTION
#
# IMPORTANT:
# We will apply this ONLY to Diagnosis fields.
# ============================================================

ICD10_RE = re.compile(
    r"\b([A-Za-z])[\s-]?(\d{2}(?:\.\d{1,4})?)\b",
    re.IGNORECASE,
)


def extract_icd10(text):

    text = safe_text(text)

    codes = set()

    for letter, number in ICD10_RE.findall(text):

        code = (
            letter.upper()
            + number
        )

        codes.add(code)

    return codes


# ============================================================
# SET F1
#
# Empty-empty = NaN, NOT 1.
# ============================================================

def set_f1(
    truth,
    pred,
):

    truth = set(truth)
    pred = set(pred)

    if (
        len(truth) == 0
        and len(pred) == 0
    ):
        return np.nan

    tp = len(
        truth & pred
    )

    fp = len(
        pred - truth
    )

    fn = len(
        truth - pred
    )

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    if (
        precision + recall
        == 0
    ):
        return 0.0

    return (
        2
        * precision
        * recall
        / (precision + recall)
    )


def micro_set_f1(
    truth_sets,
    pred_sets,
):

    tp = 0
    fp = 0
    fn = 0

    for truth, pred in zip(
        truth_sets,
        pred_sets,
    ):

        truth = set(truth)
        pred = set(pred)

        tp += len(
            truth & pred
        )

        fp += len(
            pred - truth
        )

        fn += len(
            truth - pred
        )

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    if (
        precision + recall
        == 0
    ):
        return 0.0

    return (
        2
        * precision
        * recall
        / (precision + recall)
    )


# ============================================================
# SIMPLE CIDEr IMPLEMENTATION
# Same concept as corrected evaluation:
# TF-IDF weighted n-gram cosine similarity, n=1..4.
# ============================================================

def get_ngrams(
    tokens,
    n,
):

    return [
        tuple(
            tokens[i:i+n]
        )
        for i in range(
            len(tokens) - n + 1
        )
    ]


def cosine_counter(
    a,
    b,
):

    if not a or not b:
        return 0.0

    common = (
        set(a.keys())
        & set(b.keys())
    )

    numerator = sum(
        a[x] * b[x]
        for x in common
    )

    norm_a = math.sqrt(
        sum(
            v * v
            for v in a.values()
        )
    )

    norm_b = math.sqrt(
        sum(
            v * v
            for v in b.values()
        )
    )

    if (
        norm_a == 0
        or norm_b == 0
    ):
        return 0.0

    return (
        numerator
        / (norm_a * norm_b)
    )


def cider_single(
    reference,
    candidate,
):

    ref_tokens = tokenize_text(
        reference
    )

    cand_tokens = tokenize_text(
        candidate
    )

    scores = []

    for n in range(1, 5):

        ref_ng = Counter(
            get_ngrams(
                ref_tokens,
                n,
            )
        )

        cand_ng = Counter(
            get_ngrams(
                cand_tokens,
                n,
            )
        )

        scores.append(
            cosine_counter(
                ref_ng,
                cand_ng,
            )
        )

    return float(
        np.mean(scores)
    )


# ============================================================
# TEXT METRICS
# ============================================================

references_bleu = []
hypotheses_bleu = []

bleu_case = []
rouge_case = []
meteor_case = []
cider_case = []


# ============================================================
# ENTITY STORAGE
# ============================================================

gt_fdi_sets = []
pred_fdi_sets = []

gt_icd_sets = []
pred_icd_sets = []

fdi_case_f1 = []
icd_case_f1 = []


# ============================================================
# PER-CASE LOOP
# ============================================================

clinical_metric_rows = []


for _, row in clinical_df.iterrows():

    qid = norm_id(
        row["case_id"]
    )

    gt_full = safe_text(
        row["GT_FULL"]
    )

    pred_full = safe_text(
        row["PRED_FULL"]
    )

    gt_tokens = tokenize_text(
        gt_full
    )

    pred_tokens = tokenize_text(
        pred_full
    )


    # --------------------------------------------------------
    # BLEU
    # --------------------------------------------------------

    references_bleu.append(
        [gt_tokens]
    )

    hypotheses_bleu.append(
        pred_tokens
    )

    if pred_tokens:

        b = sentence_bleu(
            [gt_tokens],
            pred_tokens,
            weights=(
                0.25,
                0.25,
                0.25,
                0.25,
            ),
            smoothing_function=smooth,
        )

    else:
        b = 0.0

    bleu_case.append(b)


    # --------------------------------------------------------
    # ROUGE-L
    # --------------------------------------------------------

    r = rouge.score(
        gt_full,
        pred_full,
    )["rougeL"].fmeasure

    rouge_case.append(r)


    # --------------------------------------------------------
    # METEOR
    # --------------------------------------------------------

    if (
        gt_tokens
        and pred_tokens
    ):

        m = meteor_score(
            [gt_tokens],
            pred_tokens,
        )

    else:
        m = 0.0

    meteor_case.append(m)


    # --------------------------------------------------------
    # CIDEr
    # --------------------------------------------------------

    c = cider_single(
        gt_full,
        pred_full,
    )

    cider_case.append(c)


    # --------------------------------------------------------
    # FDI
    #
    # Use complete report because tooth references can
    # legitimately appear in Oral Check / Treatment / Handle.
    # --------------------------------------------------------

    gt_fdi = extract_fdi(
        gt_full
    )

    pred_fdi = extract_fdi(
        pred_full
    )

    gt_fdi_sets.append(
        gt_fdi
    )

    pred_fdi_sets.append(
        pred_fdi
    )

    fdi_f1 = set_f1(
        gt_fdi,
        pred_fdi,
    )

    fdi_case_f1.append(
        fdi_f1
    )


    # --------------------------------------------------------
    # ICD — DIAGNOSIS ONLY
    # --------------------------------------------------------

    gt_icd = extract_icd10(
        row[
            "GT_Diagnosis"
        ]
    )

    pred_icd = extract_icd10(
        row[
            "PRED_Diagnosis"
        ]
    )

    gt_icd_sets.append(
        gt_icd
    )

    pred_icd_sets.append(
        pred_icd
    )

    icd_f1 = set_f1(
        gt_icd,
        pred_icd,
    )

    icd_case_f1.append(
        icd_f1
    )


    clinical_metric_rows.append({

        "case_id":
            qid,

        "BLEU4":
            b,

        "ROUGE_L":
            r,

        "METEOR":
            m,

        "CIDEr":
            c,

        "GT_FDI":
            sorted(gt_fdi),

        "PRED_FDI":
            sorted(pred_fdi),

        "FDI_F1":
            fdi_f1,

        "GT_Diagnosis_ICD10":
            sorted(gt_icd),

        "PRED_Diagnosis_ICD10":
            sorted(pred_icd),

        "Diagnosis_ICD10_F1":
            icd_f1,
    })


clinical_metrics_df = pd.DataFrame(
    clinical_metric_rows
)


# ============================================================
# CORPUS BLEU
# ============================================================

corpus_bleu4 = corpus_bleu(
    references_bleu,
    hypotheses_bleu,
    weights=(
        0.25,
        0.25,
        0.25,
        0.25,
    ),
    smoothing_function=smooth,
)


# ============================================================
# AGGREGATE ENTITY METRICS
# ============================================================

fdi_evaluable = [
    x
    for x in fdi_case_f1
    if not pd.isna(x)
]

icd_evaluable = [
    x
    for x in icd_case_f1
    if not pd.isna(x)
]


fdi_macro = (
    float(
        np.mean(
            fdi_evaluable
        )
    )
    if fdi_evaluable
    else np.nan
)

fdi_micro = micro_set_f1(
    gt_fdi_sets,
    pred_fdi_sets,
)


icd_macro = (
    float(
        np.mean(
            icd_evaluable
        )
    )
    if icd_evaluable
    else np.nan
)

icd_micro = micro_set_f1(
    gt_icd_sets,
    pred_icd_sets,
)


print("=" * 72)
print("RUN-2 CORE CLINICAL METRICS")
print("=" * 72)

print(
    f"Clinical N:              "
    f"{len(clinical_df)}"
)

print(
    f"BLEU-4:                 "
    f"{corpus_bleu4:.6f}"
)

print(
    f"ROUGE-L:                "
    f"{np.mean(rouge_case):.6f}"
)

print(
    f"METEOR:                 "
    f"{np.mean(meteor_case):.6f}"
)

print(
    f"CIDEr:                  "
    f"{np.mean(cider_case):.6f}"
)

print(
    f"FDI F1 Macro:           "
    f"{fdi_macro:.6f} "
    f"(N={len(fdi_evaluable)})"
)

print(
    f"FDI F1 Micro:           "
    f"{fdi_micro:.6f}"
)

print(
    f"Diagnosis ICD F1 Macro: "
    f"{icd_macro:.6f} "
    f"(N={len(icd_evaluable)})"
)

print(
    f"Diagnosis ICD F1 Micro: "
    f"{icd_micro:.6f}"
)

RUN-2 CORE CLINICAL METRICS
Clinical N:              77
BLEU-4:                 0.001564
ROUGE-L:                0.097237
METEOR:                 0.077898
CIDEr:                  0.089467
FDI F1 Macro:           0.034963 (N=75)
FDI F1 Micro:           0.044776
Diagnosis ICD F1 Macro: 0.000000 (N=25)
Diagnosis ICD F1 Micro: 0.000000


In [87]:
# ============================================================
# STEP 8 — RECONSTRUCT EXACT EXP2D TOP-5 RETRIEVAL EVIDENCE
#             + GROUNDING METRICS
# ============================================================

# ------------------------------------------------------------
# SET THESE TO THE SAME FILES USED FOR YOUR FINAL PIPELINE
# ------------------------------------------------------------

EXP2D_JSON_PATH = (
    "/content/rag_output_exp2d_final.json"
)

FULL_PATIENT_PATH = (
    "/content/mmdental_cleaned_full.csv"
)


# ============================================================
# LOAD EXP2D JSON
# ============================================================

with open(
    EXP2D_JSON_PATH,
    "r",
    encoding="utf-8",
) as f:

    exp2d_raw = json.load(f)


# Handle either direct dictionary or wrapped dictionary.
if isinstance(exp2d_raw, dict):

    if (
        len(exp2d_raw) == 1
        and isinstance(
            next(
                iter(
                    exp2d_raw.values()
                )
            ),
            dict,
        )
        and len(
            next(
                iter(
                    exp2d_raw.values()
                )
            )
        ) == 101
    ):

        exp2d_data = next(
            iter(
                exp2d_raw.values()
            )
        )

    else:

        exp2d_data = exp2d_raw

else:

    raise ValueError(
        "Unexpected Exp2D JSON structure."
    )


exp2d_data = {
    norm_id(k): v
    for k, v
    in exp2d_data.items()
}


assert len(exp2d_data) == 101

print(
    "[PASS] Exp2D queries:",
    len(exp2d_data),
)


# ============================================================
# LOAD FULL 660-PATIENT LOOKUP
#
# IMPORTANT:
# Your file previously had .csv extension but was actually
# Excel-format. This handles either case.
# ============================================================

try:

    full_lookup_df = pd.read_excel(
        FULL_PATIENT_PATH
    )

except Exception:

    full_lookup_df = pd.read_csv(
        FULL_PATIENT_PATH
    )


print(
    "[PASS] Full lookup shape:",
    full_lookup_df.shape,
)


assert "Filename" in full_lookup_df.columns


full_lookup_df[
    "_case_id"
] = (
    full_lookup_df[
        "Filename"
    ]
    .map(norm_id)
)


assert (
    full_lookup_df[
        "_case_id"
    ].nunique()
    >= 600
)


lookup_by_id = (
    full_lookup_df
    .drop_duplicates(
        "_case_id"
    )
    .set_index(
        "_case_id"
    )
)


# ============================================================
# FIND RETRIEVAL LIST
#
# Accommodates the naming variants we've encountered.
# ============================================================

def get_exp2d_refs(
    entry,
):

    possible_keys = [

        "retrieved_exemplars",

        "retrieved_cases",

        "top5",

        "top_5",

        "results",

        "retrieved",
    ]

    refs = None

    for key in possible_keys:

        if (
            key in entry
            and isinstance(
                entry[key],
                list,
            )
        ):

            refs = entry[key]
            break

    if refs is None:

        raise KeyError(
            "Could not locate retrieved "
            "reference list. Available keys: "
            + str(
                list(
                    entry.keys()
                )
            )
        )

    return refs


def get_ref_id(
    ref,
):

    if isinstance(
        ref,
        dict,
    ):

        for key in [
            "case_id",
            "patient_id",
            "id",
            "Filename",
        ]:

            if key in ref:
                return norm_id(
                    ref[key]
                )

    return norm_id(ref)


def get_ref_score(
    ref,
):

    if not isinstance(
        ref,
        dict,
    ):
        return np.nan

    for key in [
        "reranker_score",
        "score",
        "retrieval_score",
        "similarity",
    ]:

        if key in ref:

            try:
                return float(
                    ref[key]
                )

            except Exception:
                return np.nan

    return np.nan


# ============================================================
# BUILD ENRICHED REFERENCES
# ============================================================

REFERENCE_FIELDS = [
    "Main appeal",
    "Present medical history",
    "Oral Check",
    "Diagnosis",
    "Treatment plan",
    "Handle",
    "Doctor advices",
]


enriched_retrieval = {}


for qid in TEST_IDS:

    assert qid in exp2d_data, (
        f"Missing query {qid}"
    )

    refs = get_exp2d_refs(
        exp2d_data[qid]
    )

    assert len(refs) == 5, (
        qid,
        len(refs),
    )

    enriched = []

    for rank, ref in enumerate(
        refs,
        start=1,
    ):

        rid = get_ref_id(ref)

        assert rid != qid, (
            f"Self retrieval: {qid}"
        )

        assert rid in lookup_by_id.index, (
            f"Reference {rid} "
            f"missing from lookup."
        )

        source = (
            lookup_by_id.loc[
                rid
            ]
        )

        record = {}

        for field in REFERENCE_FIELDS:

            record[field] = safe_text(
                source.get(
                    field,
                    ""
                )
            )

        enriched.append({

            "rank":
                rank,

            "case_id":
                rid,

            "reranker_score":
                get_ref_score(
                    ref
                ),

            "record":
                record,
        })

    enriched_retrieval[
        qid
    ] = enriched


assert len(
    enriched_retrieval
) == 101


print(
    "[PASS] Enriched retrieval "
    "for all 101 queries."
)


# ============================================================
# HELPER: COMBINE RETRIEVED EVIDENCE
# ============================================================

def combine_retrieved_text(
    qid,
):

    parts = []

    for ref in (
        enriched_retrieval[
            qid
        ]
    ):

        record = ref[
            "record"
        ]

        for field in REFERENCE_FIELDS:

            value = safe_text(
                record.get(
                    field,
                    ""
                )
            )

            if value:
                parts.append(
                    value
                )

    return " ".join(
        parts
    )


# ============================================================
# RETRIEVAL COPY RATE
#
# Fraction of prediction 4-grams appearing in retrieved
# evidence but NOT appearing in GT.
# ============================================================

def ngram_set(
    text,
    n=4,
):

    tokens = tokenize_text(
        text
    )

    return set(
        get_ngrams(
            tokens,
            n,
        )
    )


def retrieval_copy_rate(
    pred,
    gt,
    retrieved,
    n=4,
):

    pred_ng = ngram_set(
        pred,
        n,
    )

    if not pred_ng:
        return 0.0

    gt_ng = ngram_set(
        gt,
        n,
    )

    ret_ng = ngram_set(
        retrieved,
        n,
    )

    copied = (
        pred_ng
        & ret_ng
    ) - gt_ng

    return (
        len(copied)
        / len(pred_ng)
    )


# ============================================================
# FABRICATED TOOTH RATE
#
# A predicted tooth is considered supported if it appears
# in either:
#   current patient's GT
# OR
#   retrieved Exp2D evidence.
#
# This measures unsupported fabrication, NOT whether the
# predicted tooth is correct for the current patient.
# ============================================================

def fabricated_tooth_rate(
    pred,
    gt,
    retrieved,
):

    pred_teeth = extract_fdi(
        pred
    )

    if not pred_teeth:
        return np.nan

    support = (
        extract_fdi(gt)
        | extract_fdi(
            retrieved
        )
    )

    fabricated = (
        pred_teeth
        - support
    )

    return (
        len(fabricated)
        / len(pred_teeth)
    )


# ============================================================
# FDI + DIAGNOSIS-ICD ENTITY HALLUCINATION
#
# Unsupported predicted entity relative to CURRENT PATIENT GT.
#
# NOTE:
# ICD is Diagnosis-only.
# ============================================================

def entity_hallucination_rate(
    gt_fdi,
    pred_fdi,
    gt_icd,
    pred_icd,
):

    predicted_entities = (
        {
            "FDI:" + x
            for x in pred_fdi
        }
        |
        {
            "ICD:" + x
            for x in pred_icd
        }
    )

    if not predicted_entities:
        return np.nan

    true_entities = (
        {
            "FDI:" + x
            for x in gt_fdi
        }
        |
        {
            "ICD:" + x
            for x in gt_icd
        }
    )

    unsupported = (
        predicted_entities
        - true_entities
    )

    return (
        len(unsupported)
        / len(
            predicted_entities
        )
    )


# ============================================================
# CALCULATE GROUNDING METRICS
# ============================================================

grounding_rows = []

hallucination_numerator = 0
hallucination_denominator = 0

fabricated_numerator = 0
fabricated_denominator = 0


for _, row in clinical_df.iterrows():

    qid = norm_id(
        row["case_id"]
    )

    gt = safe_text(
        row["GT_FULL"]
    )

    pred = safe_text(
        row["PRED_FULL"]
    )

    retrieved = (
        combine_retrieved_text(
            qid
        )
    )


    # --------------------------------------------------------
    # Retrieval copy
    # --------------------------------------------------------

    copy_rate = (
        retrieval_copy_rate(
            pred,
            gt,
            retrieved,
            n=4,
        )
    )


    # --------------------------------------------------------
    # Fabricated teeth
    # --------------------------------------------------------

    pred_fdi = extract_fdi(
        pred
    )

    gt_fdi = extract_fdi(
        gt
    )

    retrieved_fdi = extract_fdi(
        retrieved
    )

    fab_rate = (
        fabricated_tooth_rate(
            pred,
            gt,
            retrieved,
        )
    )

    if pred_fdi:

        fabricated_numerator += len(
            pred_fdi
            - (
                gt_fdi
                | retrieved_fdi
            )
        )

        fabricated_denominator += len(
            pred_fdi
        )


    # --------------------------------------------------------
    # Diagnosis-only ICD
    # --------------------------------------------------------

    gt_icd = extract_icd10(
        row[
            "GT_Diagnosis"
        ]
    )

    pred_icd = extract_icd10(
        row[
            "PRED_Diagnosis"
        ]
    )


    # --------------------------------------------------------
    # Entity hallucination
    # --------------------------------------------------------

    hall_rate = (
        entity_hallucination_rate(
            gt_fdi,
            pred_fdi,
            gt_icd,
            pred_icd,
        )
    )


    pred_entities = (
        {
            "FDI:" + x
            for x in pred_fdi
        }
        |
        {
            "ICD:" + x
            for x in pred_icd
        }
    )

    gt_entities = (
        {
            "FDI:" + x
            for x in gt_fdi
        }
        |
        {
            "ICD:" + x
            for x in gt_icd
        }
    )


    if pred_entities:

        hallucination_numerator += len(
            pred_entities
            - gt_entities
        )

        hallucination_denominator += len(
            pred_entities
        )


    grounding_rows.append({

        "case_id":
            qid,

        "Retrieval_Copy_Rate":
            copy_rate,

        "Fabricated_Tooth_Rate":
            fab_rate,

        "Entity_Hallucination_Rate":
            hall_rate,

        "Retrieved_FDI":
            sorted(
                retrieved_fdi
            ),
    })


grounding_df = pd.DataFrame(
    grounding_rows
)


# ============================================================
# AGGREGATES
# ============================================================

hall_values = (
    grounding_df[
        "Entity_Hallucination_Rate"
    ]
    .dropna()
)

fabricated_values = (
    grounding_df[
        "Fabricated_Tooth_Rate"
    ]
    .dropna()
)


hall_patient_mean = (
    float(
        hall_values.mean()
    )
    if len(hall_values)
    else np.nan
)


hall_micro = (
    hallucination_numerator
    / hallucination_denominator

    if hallucination_denominator
    else np.nan
)


copy_mean = float(
    grounding_df[
        "Retrieval_Copy_Rate"
    ].mean()
)


fabricated_mean = (
    float(
        fabricated_values.mean()
    )
    if len(
        fabricated_values
    )
    else np.nan
)


fabricated_micro = (
    fabricated_numerator
    / fabricated_denominator

    if fabricated_denominator
    else np.nan
)


print("\n" + "=" * 72)
print("RUN-2 GROUNDING / HALLUCINATION METRICS")
print("=" * 72)

print(
    "Entity hallucination patient mean:",
    round(
        hall_patient_mean,
        6,
    ),
    f"(N={len(hall_values)})",
)

print(
    "Entity hallucination micro:",
    round(
        hall_micro,
        6,
    ),
)

print(
    "Retrieval Copy Rate:",
    round(
        copy_mean,
        6,
    ),
)

print(
    "Fabricated Tooth Rate:",
    round(
        fabricated_mean,
        6,
    ),
    f"(N={len(fabricated_values)})",
)

print(
    "Fabricated Tooth Rate micro:",
    round(
        fabricated_micro,
        6,
    ),
)

[PASS] Exp2D queries: 101
[PASS] Full lookup shape: (660, 19)
[PASS] Enriched retrieval for all 101 queries.

RUN-2 GROUNDING / HALLUCINATION METRICS
Entity hallucination patient mean: 0.918605 (N=43)
Entity hallucination micro: 0.894737
Retrieval Copy Rate: 0.01725
Fabricated Tooth Rate: 0.141026 (N=39)
Fabricated Tooth Rate micro: 0.136364


In [88]:
# ============================================================
# STEP 9 — FINAL RUN-2 SUMMARY + RUN-1 COMPARISON
# ============================================================

# ------------------------------------------------------------
# BUILD FINAL RUN-2 METRIC DICTIONARY
# ------------------------------------------------------------

run2_metrics = {

    "BLEU-4":
        float(
            corpus_bleu4
        ),

    "ROUGE-L":
        float(
            np.mean(
                rouge_case
            )
        ),

    "METEOR":
        float(
            np.mean(
                meteor_case
            )
        ),

    "CIDEr":
        float(
            np.mean(
                cider_case
            )
        ),

    "FDI Tooth-Set F1 Macro":
        float(
            fdi_macro
        ),

    "FDI Tooth-Set F1 Micro":
        float(
            fdi_micro
        ),

    "Diagnosis ICD-10 F1 Macro":
        float(
            icd_macro
        ),

    "Diagnosis ICD-10 F1 Micro":
        float(
            icd_micro
        ),

    "Entity Hallucination Patient Mean":
        float(
            hall_patient_mean
        ),

    "Entity Hallucination Micro":
        float(
            hall_micro
        ),

    "Retrieval Copy Rate":
        float(
            copy_mean
        ),

    "Fabricated Tooth-Reference Rate":
        float(
            fabricated_mean
        ),
}


# ------------------------------------------------------------
# SUMMARY TABLE
# ------------------------------------------------------------

run2_summary_rows = []


def add_summary(
    metric,
    value,
    n,
    note="",
):

    run2_summary_rows.append({

        "Metric":
            metric,

        "Value":
            value,

        "N":
            n,

        "Note":
            note,
    })


add_summary(
    "Technical Success Rate",
    n_success / 101,
    101,
    "Valid 5-field JSON and no max-token truncation",
)

add_summary(
    "BLEU-4",
    run2_metrics[
        "BLEU-4"
    ],
    len(clinical_df),
)

add_summary(
    "ROUGE-L",
    run2_metrics[
        "ROUGE-L"
    ],
    len(clinical_df),
)

add_summary(
    "METEOR",
    run2_metrics[
        "METEOR"
    ],
    len(clinical_df),
)

add_summary(
    "CIDEr",
    run2_metrics[
        "CIDEr"
    ],
    len(clinical_df),
)

add_summary(
    "FDI Tooth-Set F1 Macro",
    run2_metrics[
        "FDI Tooth-Set F1 Macro"
    ],
    len(fdi_evaluable),
    "Empty-empty excluded",
)

add_summary(
    "FDI Tooth-Set F1 Micro",
    run2_metrics[
        "FDI Tooth-Set F1 Micro"
    ],
    len(clinical_df),
)

add_summary(
    "Diagnosis ICD-10 F1 Macro",
    run2_metrics[
        "Diagnosis ICD-10 F1 Macro"
    ],
    len(icd_evaluable),
    "Diagnosis field only; empty-empty excluded",
)

add_summary(
    "Diagnosis ICD-10 F1 Micro",
    run2_metrics[
        "Diagnosis ICD-10 F1 Micro"
    ],
    len(clinical_df),
    "Diagnosis field only",
)

add_summary(
    "FDI+ICD Entity Hallucination Rate",
    run2_metrics[
        "Entity Hallucination Patient Mean"
    ],
    len(hall_values),
    "Patient mean; ICD extracted from Diagnosis only",
)

add_summary(
    "FDI+ICD Entity Hallucination Rate Micro",
    run2_metrics[
        "Entity Hallucination Micro"
    ],
    len(hall_values),
)

add_summary(
    "Retrieval Copy Rate",
    run2_metrics[
        "Retrieval Copy Rate"
    ],
    len(clinical_df),
    "Prediction 4-grams in retrieved evidence but absent from GT",
)

add_summary(
    "Fabricated Tooth-Reference Rate",
    run2_metrics[
        "Fabricated Tooth-Reference Rate"
    ],
    len(
        fabricated_values
    ),
    "Unsupported by GT or retrieved Exp2D evidence",
)


run2_summary_df = pd.DataFrame(
    run2_summary_rows
)

display(
    run2_summary_df
)


# ------------------------------------------------------------
# SAVE SUMMARY
# ------------------------------------------------------------

RUN2_METRICS_CSV = os.path.join(
    RUN2_DIR,
    "run2_final_metrics.csv",
)

run2_summary_df.to_csv(
    RUN2_METRICS_CSV,
    index=False,
)


# ============================================================
# RUN-1 VALUES FROM CORRECTED EXPERIMENT 2
# ============================================================

run1_metrics = {

    "BLEU-4":
        0.0005,

    "ROUGE-L":
        0.1088,

    "METEOR":
        0.0688,

    "CIDEr":
        0.0051,

    "FDI Tooth-Set F1 Macro":
        0.0878,

    "FDI Tooth-Set F1 Micro":
        0.1253,

    # Previous corrected run was 0 either way,
    # but label explicitly as previous evaluation.
    "Diagnosis ICD-10 F1 Macro":
        0.0000,

    "Diagnosis ICD-10 F1 Micro":
        0.0000,

    "Entity Hallucination Patient Mean":
        0.7513,

    "Entity Hallucination Micro":
        0.7400,

    "Retrieval Copy Rate":
        0.0779,

    "Fabricated Tooth-Reference Rate":
        0.1103,
}


# ============================================================
# COMPARISON
# ============================================================

comparison_rows = []


# For error metrics:
# lower is better.
LOWER_IS_BETTER = {

    "Entity Hallucination Patient Mean",

    "Entity Hallucination Micro",

    "Retrieval Copy Rate",

    "Fabricated Tooth-Reference Rate",
}


for metric in run2_metrics:

    run1 = run1_metrics.get(
        metric,
        np.nan,
    )

    run2 = run2_metrics[
        metric
    ]

    delta = (
        run2 - run1
        if not pd.isna(
            run1
        )
        else np.nan
    )


    if pd.isna(delta):

        interpretation = (
            "Not directly compared"
        )

    elif abs(delta) < 1e-12:

        interpretation = (
            "No change"
        )

    elif metric in LOWER_IS_BETTER:

        interpretation = (
            "Improved"
            if delta < 0
            else "Worsened"
        )

    else:

        interpretation = (
            "Improved"
            if delta > 0
            else "Worsened"
        )


    comparison_rows.append({

        "Metric":
            metric,

        "Run-1":
            run1,

        "Run-2":
            run2,

        "Absolute Change":
            delta,

        "Direction":
            interpretation,
    })


comparison_df = pd.DataFrame(
    comparison_rows
)


print("\n" + "=" * 72)
print("RUN-1 vs RUN-2")
print("=" * 72)

display(
    comparison_df
)


COMPARISON_CSV = os.path.join(
    RUN2_DIR,
    "run1_vs_run2_metric_comparison.csv",
)

comparison_df.to_csv(
    COMPARISON_CSV,
    index=False,
)

,Metric,Value,N,Note
0,Technical Success Rate,0.7624,101,Valid 5-field JSON and no max-token truncation
1,BLEU-4,0.0016,77,
2,ROUGE-L,0.0972,77,
3,METEOR,0.0779,77,
4,CIDEr,0.0895,77,
5,FDI Tooth-Set F1 Macro,0.0350,75,Empty-empty excluded
6,FDI Tooth-Set F1 Micro,0.0448,77,
7,Diagnosis ICD-10 F1 Macro,0.0000,25,Diagnosis field only; empty-empty excluded
8,Diagnosis ICD-10 F1 Micro,0.0000,77,Diagnosis field only
9,FDI+ICD Entity Hallucination Rate,0.9186,43,Patient mean; ICD extracted from Diagnosis only



RUN-1 vs RUN-2


,Metric,Run-1,Run-2,Absolute Change,Direction
0,BLEU-4,0.0005,0.0016,0.0011,Improved
1,ROUGE-L,0.1088,0.0972,-0.0116,Worsened
2,METEOR,0.0688,0.0779,0.0091,Improved
3,CIDEr,0.0051,0.0895,0.0844,Improved
4,FDI Tooth-Set F1 Macro,0.0878,0.0350,-0.0528,Worsened
5,FDI Tooth-Set F1 Micro,0.1253,0.0448,-0.0805,Worsened
6,Diagnosis ICD-10 F1 Macro,0.0000,0.0000,0.0000,No change
7,Diagnosis ICD-10 F1 Micro,0.0000,0.0000,0.0000,No change
8,Entity Hallucination Patient Mean,0.7513,0.9186,0.1673,Worsened
9,Entity Hallucination Micro,0.7400,0.8947,0.1547,Worsened


In [90]:
# ============================================================
# STEP 9B — CREATE COMMON-PATIENT RUN1 vs RUN2 DATASET
# ============================================================

successful_ids = set(
    clinical_df[
        "case_id"
    ].map(
        norm_id
    )
)


run1_common_df = (
    base_df[
        base_df[
            "case_id"
        ].map(
            norm_id
        ).isin(
            successful_ids
        )
    ]
    .copy()
)


run2_common_df = (
    run2_eval_df[
        run2_eval_df[
            "generation_success"
        ] == True
    ]
    .copy()
)


run1_common_df[
    "case_id"
] = (
    run1_common_df[
        "case_id"
    ].map(
        norm_id
    )
)


run2_common_df[
    "case_id"
] = (
    run2_common_df[
        "case_id"
    ].map(
        norm_id
    )
)


assert (
    set(
        run1_common_df[
            "case_id"
        ]
    )
    ==
    set(
        run2_common_df[
            "case_id"
        ]
    )
)


print(
    "[PASS] Common comparison cohort:",
    len(
        run1_common_df
    ),
)


# ------------------------------------------------------------
# Put Run-1 and Run-2 predictions side by side
# ------------------------------------------------------------

paired_rows = []


run1_indexed = (
    run1_common_df
    .set_index(
        "case_id"
    )
)


run2_indexed = (
    run2_common_df
    .set_index(
        "case_id"
    )
)


for qid in sorted(
    successful_ids
):

    old = (
        run1_indexed.loc[
            qid
        ]
    )

    new = (
        run2_indexed.loc[
            qid
        ]
    )

    out = {
        "case_id":
            qid,
    }


    for field in EXPECTED_FIELDS:

        out[
            f"GT_{field}"
        ] = new[
            f"GT_{field}"
        ]

        out[
            f"RUN1_{field}"
        ] = old[
            f"PRED_{field}"
        ]

        out[
            f"RUN2_{field}"
        ] = new[
            f"PRED_{field}"
        ]


    paired_rows.append(
        out
    )


paired_df = pd.DataFrame(
    paired_rows
)


PAIRED_CSV = os.path.join(
    RUN2_DIR,
    "run1_run2_common_patient_predictions.csv",
)

paired_df.to_csv(
    PAIRED_CSV,
    index=False,
)


print(
    "[PASS] Saved paired comparison:",
    PAIRED_CSV,
)

[PASS] Common comparison cohort: 77
[PASS] Saved paired comparison: /content/drive/MyDrive/Exp2D_QLoRA_Run2/run1_run2_common_patient_predictions.csv


In [91]:
# ============================================================
# STEP 10 — FINAL COMPLETE RUN-2 EXCEL WORKBOOK
# ============================================================

FINAL_XLSX = os.path.join(
    RUN2_DIR,
    "RUN2_FINAL_COMPLETE_EVALUATION.xlsx",
)


# ------------------------------------------------------------
# Make enriched retrieval Excel-safe
# ------------------------------------------------------------

retrieval_export_rows = []


for qid in TEST_IDS:

    refs = (
        enriched_retrieval[
            qid
        ]
    )

    row = {
        "case_id":
            qid,
    }

    for ref in refs:

        rank = ref[
            "rank"
        ]

        row[
            f"R{rank}_case_id"
        ] = ref[
            "case_id"
        ]

        row[
            f"R{rank}_score"
        ] = ref[
            "reranker_score"
        ]

        record = ref[
            "record"
        ]

        for field in [
            "Main appeal",
            "Oral Check",
            "Diagnosis",
            "Treatment plan",
            "Handle",
            "Doctor advices",
        ]:

            safe_col = (
                field
                .replace(
                    " ",
                    "_"
                )
            )

            row[
                f"R{rank}_{safe_col}"
            ] = safe_text(
                record.get(
                    field,
                    ""
                )
            )

    retrieval_export_rows.append(
        row
    )


retrieval_export_df = (
    pd.DataFrame(
        retrieval_export_rows
    )
)


# ------------------------------------------------------------
# Merge clinical + grounding per-case metrics
# ------------------------------------------------------------

per_case_metrics_df = (
    clinical_metrics_df
    .merge(
        grounding_df,
        on="case_id",
        how="left",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# Raw generation export
# ------------------------------------------------------------

raw_generation_rows = []


for qid in TEST_IDS:

    r = run2_results[
        qid
    ]

    raw_generation_rows.append({

        "case_id":
            qid,

        "technical_success":
            r.get(
                "technical_success",
                False,
            ),

        "parse_status":
            r.get(
                "parse_status",
                "",
            ),

        "schema_status":
            r.get(
                "schema_status",
                "",
            ),

        "generated_tokens":
            r.get(
                "generated_tokens",
                0,
            ),

        "hit_max_tokens":
            r.get(
                "hit_max_tokens",
                False,
            ),

        "raw_generation":
            safe_text(
                r.get(
                    "raw_generation",
                    ""
                )
            ),
    })


raw_generation_df = pd.DataFrame(
    raw_generation_rows
)


# ============================================================
# WRITE WORKBOOK
# ============================================================

with pd.ExcelWriter(
    FINAL_XLSX,
    engine="openpyxl",
) as writer:

    # ------------------------------------------
    # Final metric summary
    # ------------------------------------------

    run2_summary_df.to_excel(
        writer,
        sheet_name="Run2_Summary",
        index=False,
    )


    # ------------------------------------------
    # Technical generation results — all 101
    # ------------------------------------------

    technical_df.to_excel(
        writer,
        sheet_name="Technical_101",
        index=False,
    )


    # ------------------------------------------
    # GT vs Run2 — all 101
    # Failed generations remain empty/None.
    # ------------------------------------------

    run2_eval_df.to_excel(
        writer,
        sheet_name="GT_vs_Run2_101",
        index=False,
    )


    # ------------------------------------------
    # Successful clinical cohort only
    # ------------------------------------------

    clinical_df.to_excel(
        writer,
        sheet_name="Clinical_Success",
        index=False,
    )


    # ------------------------------------------
    # Per-case clinical metrics
    # ------------------------------------------

    per_case_metrics_df.to_excel(
        writer,
        sheet_name="Per_Case_Metrics",
        index=False,
    )


    # ------------------------------------------
    # Retrieval evidence
    # ------------------------------------------

    retrieval_export_df.to_excel(
        writer,
        sheet_name="Exp2D_Top5",
        index=False,
    )


    # ------------------------------------------
    # Run1 vs Run2 aggregate
    # ------------------------------------------

    comparison_df.to_excel(
        writer,
        sheet_name="Run1_vs_Run2",
        index=False,
    )


    # ------------------------------------------
    # Common patient prediction comparison
    # ------------------------------------------

    paired_df.to_excel(
        writer,
        sheet_name="Paired_Predictions",
        index=False,
    )


    # ------------------------------------------
    # Raw generation audit
    # ------------------------------------------

    raw_generation_df.to_excel(
        writer,
        sheet_name="Raw_Generations",
        index=False,
    )


# ============================================================
# FINAL VALIDATION
# ============================================================

assert len(
    technical_df
) == 101

assert len(
    run2_eval_df
) == 101

assert len(
    retrieval_export_df
) == 101

assert (
    technical_df[
        "case_id"
    ].nunique()
    == 101
)

assert (
    run2_eval_df[
        "case_id"
    ].nunique()
    == 101
)


print("\n" + "=" * 72)
print("FINAL RUN-2 EXPORT COMPLETE")
print("=" * 72)

print(
    "Total test patients:       ",
    len(
        run2_eval_df
    ),
)

print(
    "Successful generations:    ",
    n_success,
)

print(
    "Technical failures:        ",
    101 - n_success,
)

print(
    "Clinical evaluation N:     ",
    len(
        clinical_df
    ),
)

print(
    "Exp2D evidence patients:   ",
    len(
        retrieval_export_df
    ),
)

print(
    "\nFinal workbook:"
)

print(
    FINAL_XLSX
)


FINAL RUN-2 EXPORT COMPLETE
Total test patients:        101
Successful generations:     77
Technical failures:         24
Clinical evaluation N:      77
Exp2D evidence patients:    101

Final workbook:
/content/drive/MyDrive/Exp2D_QLoRA_Run2/RUN2_FINAL_COMPLETE_EVALUATION.xlsx


In [92]:
import shutil
import os

output_dirs_to_zip = [
    '/content/exp2d_qlora_final',
    '/content/experiment_2_corrected_evaluation',
    '/content/experiment_3_adapter_diagnostic',
    '/content/experiment_4_stable_decoding',
]

# Also include the Run-2 directory, even though it's already in Drive
output_dirs_to_zip.append(RUN2_DIR)

print("Creating zip archives for output directories...")
zipped_files = []

for output_dir in output_dirs_to_zip:
    if os.path.exists(output_dir):
        # Base name for the zip file will be the directory name
        archive_name = os.path.basename(output_dir)
        # Create the zip file in /content/
        zip_path = shutil.make_archive(f'/content/{archive_name}', 'zip', output_dir)
        zipped_files.append(zip_path)
        print(f"- Created: {zip_path}")
    else:
        print(f"- Directory not found, skipping: {output_dir}")

print("\nAll output directories zipped. You can download them from the file browser on the left, or by running the following shell commands if you prefer to use the command line:")
for zip_file in zipped_files:
    print(f"!cp {zip_file} .") # This will copy the zip to the root of the notebook, making it easier to download

Creating zip archives for output directories...
- Created: /content/exp2d_qlora_final.zip
- Created: /content/experiment_2_corrected_evaluation.zip
- Created: /content/experiment_3_adapter_diagnostic.zip
- Created: /content/experiment_4_stable_decoding.zip
- Created: /content/Exp2D_QLoRA_Run2.zip

All output directories zipped. You can download them from the file browser on the left, or by running the following shell commands if you prefer to use the command line:
!cp /content/exp2d_qlora_final.zip .
!cp /content/experiment_2_corrected_evaluation.zip .
!cp /content/experiment_3_adapter_diagnostic.zip .
!cp /content/experiment_4_stable_decoding.zip .
!cp /content/Exp2D_QLoRA_Run2.zip .


In [97]:
# ============================================================
# FINAL HUMAN-READABLE CSV
# Ground Truth + Run-2 LLM Prediction + 5 Retrieval Exemplars
# One row = one test patient
# ============================================================

import os
import json
import pandas as pd
import numpy as np


# ============================================================
# 1. PATHS
# ============================================================

RUN2_JSON = "/content/drive/MyDrive/Exp2D_QLoRA_Run2/run2_robust_generation_all101.json"

# Your original 101-patient GT file
GT_FILE = "/content/ground_truth_vs_qlora_with_exp2d_exemplars.xlsx"

# Full MMDental 660-patient clinical lookup
FULL_PATIENT_FILE = "/content/mmdental_cleaned_full.csv"

OUTPUT_CSV = "/content/drive/MyDrive/Exp2D_QLoRA_Run2/final_GT_LLM_RetrievalExemplars_101.csv"


# ============================================================
# 2. HELPERS
# ============================================================

TARGET_FIELDS = [
    "Oral Check",
    "Diagnosis",
    "Treatment plan",
    "Handle",
    "Doctor advices",
]


def clean_id(x):
    """
    Normalize IDs such as:
      88
      '88'
      88.0
    -> '88'
    """
    if pd.isna(x):
        return None

    x = str(x).strip()

    if x.endswith(".0"):
        x = x[:-2]

    return x


def clean_value(x):
    """Convert missing values to empty strings."""
    if x is None:
        return ""

    try:
        if pd.isna(x):
            return ""
    except:
        pass

    return str(x).strip()


# ============================================================
# 3. LOAD RUN-2 GENERATIONS
# ============================================================

with open(RUN2_JSON, "r", encoding="utf-8") as f:
    run2 = json.load(f)

print("Run-2 cases:", len(run2))


# ============================================================
# 4. LOAD GROUND TRUTH
# ============================================================

gt_df = pd.read_excel(GT_FILE)

gt_df["case_id"] = gt_df["case_id"].apply(clean_id)

assert gt_df["case_id"].nunique() == 101, \
    f"Expected 101 GT patients, found {gt_df['case_id'].nunique()}"

gt_lookup = gt_df.set_index("case_id").to_dict("index")

print("Ground-truth patients:", len(gt_lookup))


# ============================================================
# 5. LOAD FULL 660-PATIENT LOOKUP
#
# NOTE:
# Your MMDental file may have .csv extension while actually
# being Excel format, so try Excel first.
# ============================================================

try:
    full_df = pd.read_excel(FULL_PATIENT_FILE)
    print("Loaded full patient file using pd.read_excel()")
except Exception:
    full_df = pd.read_csv(FULL_PATIENT_FILE)
    print("Loaded full patient file using pd.read_csv()")


print("Full patient records:", len(full_df))
print("Columns:", full_df.columns.tolist())


# MMDental patient identifier
assert "Filename" in full_df.columns, \
    "Could not find 'Filename' patient ID column."

full_df["patient_id"] = full_df["Filename"].apply(clean_id)

full_lookup = full_df.set_index("patient_id").to_dict("index")


# ============================================================
# 6. BUILD FINAL 101-ROW TABLE
# ============================================================

rows = []


for case_id_raw, result in run2.items():

    case_id = clean_id(case_id_raw)

    if case_id not in gt_lookup:
        print(f"[WARNING] Case {case_id} missing from GT file")
        continue

    gt = gt_lookup[case_id]

    # --------------------------------------------------------
    # Current patient information
    # --------------------------------------------------------

    row = {
        "Case_ID": case_id,

        "Patient_Age": clean_value(gt.get("Age")),
        "Patient_Sex": clean_value(gt.get("Sex")),
        "Patient_Main_Appeal": clean_value(gt.get("Main appeal")),
    }


    # ========================================================
    # GROUND TRUTH
    # ========================================================

    row.update({

        "GROUND_TRUTH_Oral_Check":
            clean_value(gt.get("GT_Oral Check")),

        "GROUND_TRUTH_Diagnosis":
            clean_value(gt.get("GT_Diagnosis")),

        "GROUND_TRUTH_Treatment_Plan":
            clean_value(gt.get("GT_Treatment plan")),

        "GROUND_TRUTH_Handle":
            clean_value(gt.get("GT_Handle")),

        "GROUND_TRUTH_Doctor_Advices":
            clean_value(gt.get("GT_Doctor advices")),
    })


    # ========================================================
    # RUN-2 LLM PREDICTION
    # ========================================================

    technical_success = bool(
        result.get("technical_success", False)
    )

    prediction = result.get("parsed_prediction")

    if technical_success and isinstance(prediction, dict):

        row.update({

            "LLM_PREDICTION_Oral_Check":
                clean_value(prediction.get("Oral Check")),

            "LLM_PREDICTION_Diagnosis":
                clean_value(prediction.get("Diagnosis")),

            "LLM_PREDICTION_Treatment_Plan":
                clean_value(prediction.get("Treatment plan")),

            "LLM_PREDICTION_Handle":
                clean_value(prediction.get("Handle")),

            "LLM_PREDICTION_Doctor_Advices":
                clean_value(prediction.get("Doctor advices")),
        })

    else:

        # IMPORTANT:
        # Do NOT convert generation failures into
        # "Not determined".
        #
        # Mark them explicitly as technical failures.

        row.update({

            "LLM_PREDICTION_Oral_Check":
                "[TECHNICAL GENERATION FAILURE]",

            "LLM_PREDICTION_Diagnosis":
                "[TECHNICAL GENERATION FAILURE]",

            "LLM_PREDICTION_Treatment_Plan":
                "[TECHNICAL GENERATION FAILURE]",

            "LLM_PREDICTION_Handle":
                "[TECHNICAL GENERATION FAILURE]",

            "LLM_PREDICTION_Doctor_Advices":
                "[TECHNICAL GENERATION FAILURE]",
        })


    # ========================================================
    # GENERATION STATUS
    # ========================================================

    row.update({

        "Technical_Success":
            technical_success,

        "Parse_Status":
            clean_value(result.get("parse_status")),

        "Schema_Status":
            clean_value(result.get("schema_status")),

        "Generated_Tokens":
            result.get("generated_tokens", ""),

        "Hit_Max_Tokens":
            result.get("hit_max_tokens", ""),

    })


    # ========================================================
    # RETRIEVED EXEMPLARS
    # ========================================================

    retrieved_ids = result.get("retrieved_ids", [])

    if len(retrieved_ids) != 5:
        print(
            f"[WARNING] Case {case_id}: "
            f"expected 5 retrieved exemplars, "
            f"found {len(retrieved_ids)}"
        )


    for rank in range(1, 6):

        if rank <= len(retrieved_ids):

            ref_id = clean_id(retrieved_ids[rank - 1])

        else:

            ref_id = None


        prefix = f"RETRIEVAL_{rank}"


        # ----------------------------------------------------
        # Missing reference
        # ----------------------------------------------------

        if ref_id is None or ref_id not in full_lookup:

            row[f"{prefix}_Patient_ID"] = ref_id or ""

            row[f"{prefix}_Oral_Check"] = ""
            row[f"{prefix}_Diagnosis"] = ""
            row[f"{prefix}_Treatment_Plan"] = ""
            row[f"{prefix}_Handle"] = ""
            row[f"{prefix}_Doctor_Advices"] = ""

            continue


        # ----------------------------------------------------
        # Historical retrieved patient record
        # ----------------------------------------------------

        ref = full_lookup[ref_id]

        row[f"{prefix}_Patient_ID"] = ref_id

        row[f"{prefix}_Oral_Check"] = \
            clean_value(ref.get("Oral Check"))

        row[f"{prefix}_Diagnosis"] = \
            clean_value(ref.get("Diagnosis"))

        row[f"{prefix}_Treatment_Plan"] = \
            clean_value(ref.get("Treatment plan"))

        row[f"{prefix}_Handle"] = \
            clean_value(ref.get("Handle"))

        row[f"{prefix}_Doctor_Advices"] = \
            clean_value(ref.get("Doctor advices"))


    rows.append(row)


# ============================================================
# 7. CREATE DATAFRAME
# ============================================================

final_df = pd.DataFrame(rows)


# Sort numerically by patient ID
final_df["_sort_id"] = pd.to_numeric(
    final_df["Case_ID"],
    errors="coerce"
)

final_df = (
    final_df
    .sort_values("_sort_id")
    .drop(columns="_sort_id")
    .reset_index(drop=True)
)


# ============================================================
# 8. VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET VALIDATION")
print("=" * 70)

print("Rows:", len(final_df))
print("Unique patients:", final_df["Case_ID"].nunique())

assert len(final_df) == 101, \
    f"Expected 101 rows, got {len(final_df)}"

assert final_df["Case_ID"].nunique() == 101, \
    "Duplicate Case IDs detected."


# Check retrieval count
retrieval_id_columns = [
    f"RETRIEVAL_{i}_Patient_ID"
    for i in range(1, 6)
]

retrieval_counts = (
    final_df[retrieval_id_columns]
    .replace("", np.nan)
    .notna()
    .sum(axis=1)
)

print(
    "Patients with exactly 5 retrieved exemplars:",
    int((retrieval_counts == 5).sum()),
    "/ 101"
)


# Check accidental self-retrieval
self_retrieval = []

for _, r in final_df.iterrows():

    current_id = clean_id(r["Case_ID"])

    refs = [
        clean_id(r[c])
        for c in retrieval_id_columns
        if clean_id(r[c]) is not None
    ]

    if current_id in refs:
        self_retrieval.append(current_id)


print(
    "Self-retrieval cases:",
    len(self_retrieval)
)

if self_retrieval:
    print("WARNING:", self_retrieval)


# Technical generation results
success_count = int(
    final_df["Technical_Success"].sum()
)

failure_count = len(final_df) - success_count

print(
    f"Technical successes: {success_count}/101"
)

print(
    f"Technical failures: {failure_count}/101"
)


# ============================================================
# 9. SAVE CSV
# ============================================================

os.makedirs(
    os.path.dirname(OUTPUT_CSV),
    exist_ok=True
)

final_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 70)
print("SAVED FINAL CSV")
print("=" * 70)

print(OUTPUT_CSV)

print("\nShape:", final_df.shape)


# ============================================================
# 10. SHOW COLUMN STRUCTURE
# ============================================================

print("\nCOLUMN GROUPS:")

print("\n--- CURRENT PATIENT ---")
print([
    "Case_ID",
    "Patient_Age",
    "Patient_Sex",
    "Patient_Main_Appeal",
])

print("\n--- GROUND TRUTH ---")
print([
    c for c in final_df.columns
    if c.startswith("GROUND_TRUTH_")
])

print("\n--- LATEST LLM PREDICTION ---")
print([
    c for c in final_df.columns
    if c.startswith("LLM_PREDICTION_")
])

print("\n--- RETRIEVAL EXEMPLARS ---")

for i in range(1, 6):
    print(
        f"Retrieval {i}:",
        [
            c for c in final_df.columns
            if c.startswith(f"RETRIEVAL_{i}_")
        ]
    )


# ============================================================
# 11. QUICK PREVIEW
# ============================================================

preview_cols = [

    "Case_ID",
    "Patient_Main_Appeal",

    "GROUND_TRUTH_Diagnosis",
    "LLM_PREDICTION_Diagnosis",

    "RETRIEVAL_1_Patient_ID",
    "RETRIEVAL_1_Diagnosis",

    "RETRIEVAL_2_Patient_ID",
    "RETRIEVAL_2_Diagnosis",

    "Technical_Success",
]

display(final_df[preview_cols].head(10))

Run-2 cases: 101
Ground-truth patients: 101
Loaded full patient file using pd.read_excel()
Full patient records: 660
Columns: ['Filename', 'Main appeal', 'Subsequent', 'Present medical history', 'Past medical history', 'Oral Check', 'Diagnosis', 'Treatment plan', 'Handle', 'Doctor advices', 'Age', 'Age_group', 'Sex', 'Diagnosis_cleaned', 'Diagnosis_categories', 'num_labels', 'primary_diagnosis', 'semantic_text', 'semantic_text_cbct']

FINAL DATASET VALIDATION
Rows: 101
Unique patients: 101
Patients with exactly 5 retrieved exemplars: 101 / 101
Self-retrieval cases: 0
Technical successes: 77/101
Technical failures: 24/101

SAVED FINAL CSV
/content/drive/MyDrive/Exp2D_QLoRA_Run2/final_GT_LLM_RetrievalExemplars_101.csv

Shape: (101, 49)

COLUMN GROUPS:

--- CURRENT PATIENT ---
['Case_ID', 'Patient_Age', 'Patient_Sex', 'Patient_Main_Appeal']

--- GROUND TRUTH ---
['GROUND_TRUTH_Oral_Check', 'GROUND_TRUTH_Diagnosis', 'GROUND_TRUTH_Treatment_Plan', 'GROUND_TRUTH_Handle', 'GROUND_TRUTH_Doctor

,Case_ID,Patient_Main_Appeal,GROUND_TRUTH_Diagnosis,LLM_PREDICTION_Diagnosis,RETRIEVAL_1_Patient_ID,RETRIEVAL_1_Diagnosis,RETRIEVAL_2_Patient_ID,RETRIEVAL_2_Diagnosis,Technical_Success
0,6,Consult for jaw restoration.,tooth 31 (lower left central incisor) periodon...,"*2*, *4*, *4*, *3*, *5*: Malocclusion type II ...",71,tooth 48 (lower right third molar) pulpitis to...,523,tooth 12 (upper right lateral incisor)-tooth 1...,True
1,15,The upper right front tooth has been missing f...,tooth 22 (upper left lateral incisor) three pe...,The postextraction site showed alveolar ridge ...,182,chronic gingivitis tooth 17 (upper right secon...,217,residual tooth root (k 08.300 x 002),True
2,19,Gum irritation and bleeding all over the mouth...,chronic gingivitis . tooth 47 (lower right sec...,Gingivitis .,271,tooth 13 (upper right canine) tooth 11 (upper ...,652,tooth defect 5 teeth longitudinal fracture,True
3,23,Lower right back tooth pain for several years,chronic gingivitis tooth 46 (lower right first...,*chronic apical periodontitis*,552,tooth 18 (upper right third molar) impacted te...,382,tooth 18 (upper right third molar) impacted teeth,True
4,32,The lower right back tooth feels uncomfortable...,tooth 46 (lower right first molar) chronic pul...,Malformed teeth Malformation type II Chronic P...,264,tooth 24 (upper left first premolar) tooth 26 ...,523,tooth 12 (upper right lateral incisor)-tooth 1...,True
5,35,He complained of pain in his posterior teeth a...,"tooth 16 (upper right first molar), tooth 14 (...",[TECHNICAL GENERATION FAILURE],109,"tooth 11 (upper right central incisor), tooth ...",107,tooth 41 (lower right central incisor) residua...,False
6,38,The self-reported request was to have the wisd...,"tooth 18 (upper right third molar), tooth 48 (...",Poor response during implant placement due to ...,297,tooth 17 (upper right second molar) tooth 27 (...,493,malformed teeth crooked teeth. tooth 46 (lower...,True
7,40,Request tooth extraction,partial loss of dentition . partial loss of de...,*27 Chronic sinusitis secondary to chronic ton...,242,malformed teeth,626,"treatment plan for tooth loss due to accident,...",True
8,41,,tooth 46 (lower right first molar) tooth defec...,oacute stomatitis,400,gingivitis.,109,"tooth 11 (upper right central incisor), tooth ...",True
9,43,Discomfort of upper right back tooth,tooth 15 (upper right second premolar) chronic...,Malformed permanent teeth (K07.30),9,tooth 28 (upper left third molar) impacted teeth,626,"treatment plan for tooth loss due to accident,...",True
